# RNN 파일별 2 · rnn.ipynb 실습

**한 줄 요약**

- 강의 노트북 97셀을 원본 순서대로 실행 · 정리 · L0 기준선 · L1 RNN · L1b 패딩 · L2 거리 · L3 기울기 · L3b LSTM 문 · L4 HAR 비교
- 과제 2 ~ 4 빈칸 정답까지

- **원본** · `7. 딥러닝/04_RNN/실습/rnn.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/08_RNN` RNN-1 (순서 데이터 · RNN · LSTM · GRU) 전체 · 강의는 RNN 파일별 1

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. 합 · 평균 · 마지막 값 · 펼치기 · 상태 기준선을 돌려 각각 몇 %인지와 이유를 말한다
2. `RNNCell` + for 문을 `nn.RNN`과 맞춰 보고 one_hot · batch_first · output · h_n 모양을 다룬다
3. 길이 5~60 패딩 배치에서 -1 · lens - 1 · 평균 · mask 평균의 정확도 차이를 재현한다
4. 거리 실험 · 칸별 기울기 · float64 · 가중치 행렬로 기울기 소실을 재고 LSTM forget bias의 효과를 보인다
5. HAR에서 RNN · LSTM · GRU를 같은 조건 · seed · 파라미터 맞춤으로 비교하고 과제 2 ~ 4를 푼다

## 어디에 쓰나

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">실제 상황</th>
<th style="text-align: left;">이 권에서 배우는 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">시퀀스 분류 모델을 처음 짤 때</td>
<td style="text-align: left;">Net · train · one_hot · batch_first (1 · 2장)</td>
</tr>
<tr>
<td style="text-align: left;">길이가 다른 데이터를 배치로 학습할 때</td>
<td style="text-align: left;">pad · lens - 1 · mask (3 · 4장)</td>
</tr>
<tr>
<td style="text-align: left;">긴 시퀀스에서 학습이 안 될 때</td>
<td style="text-align: left;">거리 실험 · grad_profile · forget bias (5 ~ 7장)</td>
</tr>
<tr>
<td style="text-align: left;">RNN · LSTM · GRU 중 고를 때</td>
<td style="text-align: left;">같은 조건 · seed · 파라미터 맞춤 (8장)</td>
</tr>
<tr>
<td style="text-align: left;">실험 결과를 보고할 때</td>
<td style="text-align: left;">seed 여러 번 · 평균 · 폭 (8-2 · 과제 3)</td>
</tr>
</tbody>
</table>

## 학습 지도

<img src="images/rn_map.png" alt="rnn 학습 지도" width="763">

**예상 학습 시간** · 개념 1시간 30분 + 실습 2시간 (GPU 전체 실행 약 10분 · CPU는 훨씬 오래)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">장</th>
<th style="text-align: left;">제목</th>
<th style="text-align: left;">이 장의 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">0</td>
<td style="text-align: left;">준비</td>
<td style="text-align: left;">실행 환경 · 원본 구성</td>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">합으로 풀어보고 깨뜨리기 (L0)</td>
<td style="text-align: left;">넘길 것을 사람이 정하면 어디까지 되고 어디서 깨지는가?</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">하나씩 읽는 모델 (L1)</td>
<td style="text-align: left;">요약을 학습으로 정하는 RNN은 어떻게 만들고 어떤 모양을 주고받는가?</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">HAR 데이터와 패딩 준비</td>
<td style="text-align: left;">실제 데이터 HAR과 길이가 다른 줄은 어떻게 텐서로 만드는가?</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">빈 칸을 읽으면 (L1b)</td>
<td style="text-align: left;">패딩한 배치에서 요약을 어떻게 꺼내야 정확도가 무너지지 않는가?</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">거리를 늘려 무너뜨리기 (L2)</td>
<td style="text-align: left;">먼 거리에서 실패하는 것은 기억 탓인가, 학습 탓인가?</td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">기울기가 어디까지 오는가 (L3)</td>
<td style="text-align: left;">마지막 손실의 신호는 몇 칸까지 오고, 왜 줄어드는가?</td>
</tr>
<tr>
<td style="text-align: left;">7</td>
<td style="text-align: left;">문을 달면 살아날까 (L3b)</td>
<td style="text-align: left;">LSTM의 문은 언제 기울기를 살리는가?</td>
</tr>
<tr>
<td style="text-align: left;">8</td>
<td style="text-align: left;">실제 데이터에서 비교 (L4)</td>
<td style="text-align: left;">HAR에서 RNN · LSTM · GRU를 공정하게 비교하려면?</td>
</tr>
<tr>
<td style="text-align: left;">9</td>
<td style="text-align: left;">종합 실습</td>
<td style="text-align: left;">hidden 크기 · seed 수 · 앞 절반 섞기를 바꾸면 결론이 달라지는가?</td>
</tr>
<tr>
<td style="text-align: left;">—</td>
<td style="text-align: left;">마무리 · 부록</td>
<td style="text-align: left;">오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td>
</tr>
</tbody>
</table>

---
## 0. 준비

원본은 CPU 기준 · 이 노트북은 GPU가 있으면 GPU로 실행 (seed 고정 · cuDNN 결정적 모드). 학습 정확도의 끝자리는 강의 화면과 다를 수 있음. **셀은 위에서부터 차례로 실행**.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">항목</th>
<th style="text-align: left;">설정</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">라이브러리</td>
<td style="text-align: left;"><code>torch</code> · <code>random</code> · <code>statistics</code> · <code>urllib</code> · <code>zipfile</code> · <code>decimal</code></td>
</tr>
<tr>
<td style="text-align: left;">seed</td>
<td style="text-align: left;"><code>SEED = 20260912</code> (원본과 같음) · cuDNN 결정적 모드</td>
</tr>
<tr>
<td style="text-align: left;">장치</td>
<td style="text-align: left;">GPU가 있으면 학습만 GPU · 기울기 측정은 원본처럼 CPU</td>
</tr>
<tr>
<td style="text-align: left;">데이터</td>
<td style="text-align: left;">3과 7 순서 과제 (노트북 안에서 생성) · UCI HAR (3장에서 내려받음 · 약 60MB)</td>
</tr>
<tr>
<td style="text-align: left;">원본 셀 수</td>
<td style="text-align: left;">97개 (코드 47 · 마크다운 50 · 과제 빈칸 3곳)</td>
</tr>
</tbody>
</table>

In [1]:
# 라이브러리 · seed · 장치 (원본 셀 2 앞부분)
import math                  # log10 (7-2 문 값 곱)
import os                    # 폴더 · 파일 경로
import random                # 파이썬 난수 (데이터 만들기)
import statistics            # 평균 (8장 seed 비교)
import sys                   # 실행 환경 확인 (colab 여부)
import time                  # 학습 시간 재기
import urllib.request        # 데이터 내려받기 (3-1)
import zipfile               # 압축 풀기 (3-1)
from decimal import Decimal  # 아주 작은 수를 소수로 적기 (5-3)

import torch                     # 텐서 · 자동 미분
import torch.nn as nn            # 층: RNN · LSTM · GRU · Linear
import torch.nn.functional as F  # one_hot

SEED = 20260912                                                     # 원본과 같은 seed
random.seed(SEED)                                                   # 파이썬 random 고정
torch.manual_seed(SEED)                                             # CPU · GPU 난수 고정
torch.backends.cudnn.deterministic = True                           # GPU에서도 매번 같은 계산 결과
torch.backends.cudnn.benchmark = False                              # 매번 같은 계산 방법 선택
ENV = "colab" if "google.colab" in sys.modules else "local"         # 원본의 환경 표시
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU 없으면 CPU
print(f"환경 {ENV} · torch {torch.__version__} · device {DEV.type}")  # 버전 · 장치 기록

환경 local · torch 2.11.0+cu128 · device cuda


### 원본 구성

- **0. 준비** · seed · 데이터 함수 (순서 과제 · 맨 앞 과제 · one-hot · 학습 함수)
- **L0 · 합으로 풀어보고 깨뜨리기** · 합 · 질문 바꾸기 · 평균 · 마지막 · 첫 값 · 모델 키우기 · 펼치기 · 봤는가 · 먼저 봤는가
- **L1 · 하나씩 읽는 모델** · Net · 손으로 펼친 판 · nn.RNN · 학습 · one_hot · batch_first · 정말 같은 값 · UCI HAR · 패딩
- **L1b · 빈 칸을 읽으면** · 길이 5~60 · -1 vs lens - 1 · 길이 4~12
- **L2 · 거리를 늘려 무너뜨리기** · 20 · 60 epoch · lr · 앞 → 뒤 (학습 전 · 20 · 60 epoch) · 손으로 심은 가중치
- **L3 · 기울기가 어디까지 오는가** · float64 · 학습한 모델 · output.grad · 가중치 행렬 · LSTM 반환
- **L3b · 문을 달면 살아날까** · LSTM 기본 · 문 값 · 문을 열어두면
- **L4 · 실제 데이터에서 비교** · RNN · LSTM · GRU · seed · 파라미터 맞춤 · 시간 순서 섞기 · LSTM · GRU 식
- **과제** · 1 다시 돌려 비교 · 2 hidden 32 · 128 · 3 seed 두 개 추가 · 4 앞 절반만 섞기 · 5 막힌 단계 번호

**원본 힌트**

- 과제 2 · `for hid in (___, ___)` → 32와 128
- 과제 3 · `SEEDS5 = (SEED, 7, 1234, ___, ___)` → 원하는 seed 두 개
- 과제 4 · `out[b, :half] = ___` → 위 `shuffle_time`을 참고해 앞 half칸만 섞기
- 과제 셀은 'UCI HAR 열어보기' 셀을 먼저 실행해야 돌아감 (원본 '돔' → 고침) · 한 셀에 몇 분씩 걸림 (CPU)

---
## 1. 합으로 풀어보고 깨뜨리기 (L0)

> **이 장의 흐름**
>
> - 원본 L0 · 줄 하나를 숫자 하나로 줄여 MLP에
> - 합 → 질문 바꾸기 → 다른 요약 → 모델 키우기 → 펼치기 → 상태

<img src="images/rn_baselines.png" alt="넘길 것별 정확도 · 이 노트북 실행 결과" width="782">

**파이프라인 위치** · ① 문제 정의 · 기준선

### 1-1. `A` 순서 과제 데이터와 학습 함수 · `make_order_task()` · `to_tensor()` · `train()` <sub>원본 셀 2</sub>

> 🎯 **3과 7 순서 과제 데이터는 어떻게 만들고, 모든 실험이 같이 쓰는 학습 함수는 무엇을 하는가?**

💡 **쉽게 말하면**

- 숫자 열 칸에 **3과 7을 하나씩** 넣고 3이 먼저면 1 · 7이 먼저면 0.
- `train()`은 **Adam · 배치 128 · 30 epoch** 학습 후 평가 정확도 하나를 돌려줌.

**실행 흐름**

1. 채울 숫자 → 3 · 7 자리 → 정답 먼저 정하고 삽입
2. 맨 앞 과제 · one-hot 변환 · 학습 함수 정의
3. 학습 6000 · 평가 2000줄

**먼저 예측** · 첫 학습 줄의 정답은 '네'일까 '아니요'일까? — 답은 코드 실행 결과 아래에

In [2]:
# 3과 7이 하나씩 든 숫자 줄: 3이 먼저면 1 · 7이 먼저면 0
OTHERS = [d for d in range(10) if d not in (3, 7)]  # 3 · 7을 뺀 채울 숫자


def make_order_task(n, length=10, gap=None, seed=None):           # 순서 과제 줄 n개
    rng = random.Random(seed) if seed is not None else random     # seed가 있으면 전용 난수
    xs, ys = [], []                                               # 입력 줄 · 정답 목록
    for _ in range(n):                                            # 줄 n개
        filler = [rng.choice(OTHERS) for _ in range(length - 2)]  # 채울 숫자 (길이 - 2)
        if gap is None:                                           # 자리를 아무 데나
            i, j = sorted(rng.sample(range(length), 2))           # 3 · 7 자리 두 개
        else:                                                     # 간격을 정해 둘 때 (7-2)
            i = rng.randrange(0, length - gap)                    # 앞 자리
            j = i + gap                                           # 뒤 자리 = 앞 + 간격
        y = rng.randint(0, 1)                                     # 정답을 먼저 정함
        a, b = (3, 7) if y == 1 else (7, 3)                       # 정답 1이면 3이 앞
        s = filler[:]                                             # 채울 숫자 복사
        s.insert(i, a)                                            # 앞 자리에 첫 숫자
        s.insert(j, b)                                            # 뒤 자리에 둘째 숫자
        xs.append(s[:length])                                     # 길이를 맞춰 저장
        ys.append(y)                                              # 정답 저장
    return xs, ys                                                 # 줄 목록 · 정답 목록


def make_front_task(n, length, seed=None):                     # 3 · 7을 맨 앞 두 칸에 (5장)
    rng = random.Random(seed) if seed is not None else random  # 전용 난수
    xs, ys = [], []                                            # 입력 줄 · 정답 목록
    for _ in range(n):                                         # 줄 n개
        y = rng.randint(0, 1)                                  # 정답을 먼저 정함
        a, b = (3, 7) if y == 1 else (7, 3)                    # 정답 1이면 3이 앞
        xs.append([a, b] + [rng.choice(OTHERS) for _ in range(length - 2)])  # 뒤는 상관없는 숫자
        ys.append(y)                                           # 정답 저장
    return xs, ys                                              # 줄 목록 · 정답 목록


def to_tensor(xs):                                              # 숫자 줄 → (줄, 길이, 10) one-hot
    return F.one_hot(torch.tensor(xs), num_classes=10).float()  # 원본의 이중 for 문과 같은 결과


def train(model, X, Y, Xe, Ye, epochs=30, lr=3e-3):               # 학습 → 평가 정확도
    model = model.to(DEV)                                         # 모델을 GPU로 (원본은 CPU)
    X, Y, Xe, Ye = X.to(DEV), Y.to(DEV), Xe.to(DEV), Ye.to(DEV)   # 데이터도 같은 장치로
    opt = torch.optim.Adam(model.parameters(), lr=lr)             # Adam
    lf = nn.CrossEntropyLoss()                                    # 분류 손실
    for _ in range(epochs):                                       # epoch 반복
        p = torch.randperm(len(X))  # 순서 섞기 (CPU 난수 = 원본과 같은 순서)
        for i in range(0, len(X), 128):                           # 128줄씩
            j = p[i:i + 128]                                      # 이번 배치 번호
            opt.zero_grad()                                       # 지난 기울기 지우기
            lf(model(X[j]), Y[j]).backward()                      # 예측 → 손실 → 기울기
            opt.step()                                            # 가중치 한 걸음 수정
    with torch.no_grad():                                         # 평가는 기울기 기록 안 함
        return (model(Xe).argmax(1) == Ye).float().mean().item()  # 맞힌 비율


xtr, ytr = make_order_task(6000, 10, seed=1)                # 학습 6000줄
xte, yte = make_order_task(2000, 10, seed=2)                # 평가 2000줄 (다른 seed)
Ytr, Yte = torch.tensor(ytr), torch.tensor(yte)             # 정답 → 텐서
print(f"학습 {len(xtr)}줄 · 평가 {len(xte)}줄")             # 개수
print("예시 :", xtr[0], "→", "네" if ytr[0] else "아니요")  # 첫 줄과 정답

학습 6000줄 · 평가 2000줄
예시 : [2, 3, 1, 7, 5, 1, 9, 9, 9, 8] → 네


**결과** · 예측 정답은 네 — 3이 7보다 앞

- `학습 6000줄 · 평가 2000줄` — seed 1 · 2로 다른 줄
- `예시 : [2, 3, 1, 7, 5, 1, 9, 9, 9, 8] → 네` — 3이 두 번째 · 7이 네 번째

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>make_order_task(n, L)</code></td>
<td style="text-align: left;">3 · 7 자리가 아무 데나</td>
<td style="text-align: left;">L0 · L1 · L1b 기본 과제</td>
</tr>
<tr>
<td style="text-align: left;"><code>make_order_task(..., gap=g)</code></td>
<td style="text-align: left;">3 · 7 간격을 g로 고정</td>
<td style="text-align: left;">7-2 문 값 측정</td>
</tr>
<tr>
<td style="text-align: left;"><code>make_front_task(n, L)</code></td>
<td style="text-align: left;">3 · 7이 맨 앞 두 칸</td>
<td style="text-align: left;">L2 · L3 거리 실험</td>
</tr>
<tr>
<td style="text-align: left;"><code>to_tensor(xs)</code></td>
<td style="text-align: left;">(줄, 길이) → (줄, 길이, 10)</td>
<td style="text-align: left;">RNN 입력</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 원본은 CPU · 이 노트북은 GPU가 있으면 GPU (정확도 끝자리가 강의와 다를 수 있음)
> - 원본 `to_tensor`는 이중 for 문 · 여기서는 같은 결과의 `F.one_hot`

> ✅ **한 줄 정리** · 3과 7의 순서로 정답이 정해지는 데이터를 seed로 고정해 만들고, 모든 실험은 같은 `train()`으로 비교한다.

**딥러닝 코드에서는**

- 실험 비교는 데이터 · 학습 함수 · seed를 고정하고 한 가지만 바꿈

### 1-2. `A` 합 하나로 풀고 질문 바꾸기 · `summarize()` · `MLP` <sub>원본 셀 3 ~ 6 · L0</sub>

> 🎯 **줄을 합 하나로 줄여 MLP에 넣으면 몇 %이고, 질문을 '합이 45 이상인가'로 바꾸면?**

💡 **쉽게 말하면**

- 넘길 것을 **사람이 정해 봄** · 줄 하나 → 숫자 하나 → MLP.
- 같은 데이터 · 같은 합 모델 · **질문만** 바꿈.

**실행 흐름**

1. 요약 함수 4가지 · MLP 정의
2. 합으로 질문 A 학습
3. 합의 중앙값 · 질문 B 학습

**먼저 예측** · 합 하나로 질문 A를 풀면 찍기(50%)보다 높을까? — 답은 코드 실행 결과 아래에

In [3]:
# 줄 하나 → 숫자 하나 (사람이 정한 요약 4가지)
def summarize(xs, how):                                       # 줄 → (N, 1)
    if how == "sum":                                          # 합
        return torch.tensor([[float(sum(s))] for s in xs])    # 줄마다 합
    if how == "mean":                                         # 평균
        return torch.tensor([[sum(s) / len(s)] for s in xs])  # 줄마다 평균
    if how == "last":                                         # 마지막 값
        return torch.tensor([[float(s[-1])] for s in xs])     # 줄마다 끝 칸
    return torch.tensor([[float(s[0])] for s in xs])          # first: 줄마다 첫 칸


class MLP(nn.Module):                                         # 요약값을 받는 분류기
    def __init__(self, i, width=32, depth=2):                 # 입력 크기 · 너비 · 은닉층 수
        super().__init__()                                    # nn.Module 준비 (필수)
        layers = [nn.Linear(i, width), nn.ReLU()]             # 첫 층
        for _ in range(depth - 1):                            # 은닉층 추가
            layers += [nn.Linear(width, width), nn.ReLU()]    # 같은 너비 층
        self.n = nn.Sequential(*layers, nn.Linear(width, 2))  # 출력 2 = 네 · 아니요

    def forward(self, x):  # 입력이 지나가는 길
        return self.n(x)   # 층을 차례로 통과


torch.manual_seed(SEED)                                                      # 모델 시작점 고정
acc = train(MLP(1), summarize(xtr, "sum"), Ytr, summarize(xte, "sum"), Yte)  # 합 하나로 학습
print(f"sum    {acc:.4f}")                                                   # 질문 A 정확도


# 질문만 바꾸기: 합이 45 이상인가 (원본 셀 6)
sums = [sum(s) for s in xtr]                                      # 줄마다 합
print("학습 데이터 합의 중앙값 :", sorted(sums)[len(sums) // 2])  # 기준 45의 근거
ytr45 = torch.tensor([1 if sum(s) >= 45 else 0 for s in xtr])     # 새 질문의 정답 (학습)
yte45 = torch.tensor([1 if sum(s) >= 45 else 0 for s in xte])     # 새 질문의 정답 (평가)
print(f"평가 데이터에서 합이 45 이상인 줄의 비율 : {yte45.float().mean().item():.3f}")  # 반반인지
torch.manual_seed(SEED)                                           # 같은 시작점
acc = train(MLP(1), summarize(xtr, "sum"), ytr45, summarize(xte, "sum"), yte45)  # 같은 합 모델
print(f"합 모델 · '합이 45 이상인가' → {acc:.4f}")                # 질문 B 정확도

sum    0.4880
학습 데이터 합의 중앙값 : 45
평가 데이터에서 합이 45 이상인 줄의 비율 : 0.522


합 모델 · '합이 45 이상인가' → 0.9575


**결과** · 예측 정답은 낮음 — 0.4880

- `sum    0.4880` — 찍기보다 낮음 · 합에는 순서가 남아 있지 않음 (강의 48.8%)
- `학습 데이터 합의 중앙값 : 45` — 질문 B의 기준을 중앙값으로 → 반반
- `평가 데이터에서 합이 45 이상인 줄의 비율 : 0.522` — 한쪽만 찍어서는 높게 못 나옴
- `합 모델 · '합이 45 이상인가' → 0.9575` — 같은 합 모델이 질문 B는 거의 맞힘 (강의 96%)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">질문 A + 합</td>
<td style="text-align: left;">입력에 답이 없음</td>
<td style="text-align: left;">찍기 수준</td>
</tr>
<tr>
<td style="text-align: left;">질문 B + 합</td>
<td style="text-align: left;">입력이 답을 정함</td>
<td style="text-align: left;">작은 모델로 충분</td>
</tr>
<tr>
<td style="text-align: left;">MLP(1)</td>
<td style="text-align: left;">숫자 하나 입력 · 은닉 32 × 2</td>
<td style="text-align: left;">요약값 기준선</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 모델을 키우거나 학습을 늘려도 없는 정보는 생기지 않음 (1-4)

> ✅ **한 줄 정리** · 합 하나로는 질문 A가 0.4880으로 찍기 수준이지만 질문 B는 0.9575이므로, 순서가 중요한지는 질문이 정한다.

**딥러닝 코드에서는**

- 새 문제마다 '순서를 버린 기준선'을 먼저 돌려 보면 순서의 몫이 보임

### 1-3. `A` 평균 · 마지막 값 · 첫 값 · `s[-1]` · `next()` <sub>원본 셀 7 ~ 10</sub>

> 🎯 **나머지 요약 셋은 몇 %이고, 마지막 값이 60% 근처인 이유를 데이터로 어떻게 세는가?**

💡 **쉽게 말하면**

- 끝 칸이 3 · 7이면 답이 확정 · 아니면 찍기.
- 그 비율을 직접 세서 기대 정확도를 계산.

**실행 흐름**

1. 평균 · 마지막 · 첫 값으로 학습
2. 끝 칸이 3 · 7인 줄 세기
3. 끝 칸이 표식이 아닌 예 보기

**먼저 예측** · 마지막 값 하나의 정확도는 0.6보다 높을까? — 답은 코드 실행 결과 아래에

In [4]:
# 나머지 세 가지 요약 (원본 셀 8)
for how in ("mean", "last", "first"):  # 평균 · 마지막 값 · 첫 값
    torch.manual_seed(SEED)            # 요약마다 같은 시작점
    acc = train(MLP(1), summarize(xtr, how), Ytr, summarize(xte, how), Yte)  # 요약 하나로 학습
    print(f"{how:<6} {acc:.4f}")       # 요약별 정확도


# 마지막 칸 세어보기 (원본 셀 10)
n_hit = sum(1 for s in xte if s[-1] in (3, 7))  # 끝 칸이 3 · 7인 줄 수
print(f"마지막 칸이 3 이나 7 인 줄 : {n_hit} / {len(xte)} ({n_hit / len(xte):.3f})")  # 비율
print(f"기대 정확도 = {n_hit / len(xte):.3f} × 1 + {1 - n_hit / len(xte):.3f} × 0.5"  # 확정 + 찍기
      f" = {n_hit / len(xte) + (1 - n_hit / len(xte)) / 2:.3f}")
print("마지막 칸이 3 도 7 도 아닌 줄 — 정답별로 하나씩")                              # 단서 없는 예
for want in (1, 0):  # 정답 1 · 0 하나씩
    k = next(i for i, s in enumerate(xte) if s[-1] not in (3, 7) and yte[i] == want)  # 첫 예
    print(" ", " ".join(map(str, xte[k])), "→", "네" if yte[k] else "아니요")  # 끝 칸만으로는 모름

mean   0.4950


last   0.6065


first  0.6085
마지막 칸이 3 이나 7 인 줄 : 422 / 2000 (0.211)
기대 정확도 = 0.211 × 1 + 0.789 × 0.5 = 0.606
마지막 칸이 3 도 7 도 아닌 줄 — 정답별로 하나씩
  8 8 6 9 5 3 0 7 0 6 → 네
  2 9 8 6 6 6 7 3 9 2 → 아니요


**결과** · 예측 정답은 거의 같음 — 0.6065 (기대값 0.606)

- `mean   0.4950` — 평균도 순서를 버림 (강의 49.9%)
- `last   0.6065` — 끝 칸이 표식인 줄만 맞힘 (강의 60.4%)
- `first  0.6085` — 첫 칸도 같은 이유 (강의 60.9%)
- `마지막 칸이 3 이나 7 인 줄 : 422 / 2000 (0.211)` — 2000줄 중 422줄
- `기대 정확도 = 0.211 × 1 + 0.789 × 0.5 = 0.606` — 학습한 결과 0.6065와 같음 → 모델은 상한까지 배움
- `8 8 6 9 5 3 0 7 0 6 → 네` — 끝 칸 6 · 같은 끝 칸에서 답이 '아니요'인 줄도 있음 → 단서 없음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">평균</td>
<td style="text-align: left;">순서 없음</td>
<td style="text-align: left;">순서가 상관없을 때</td>
</tr>
<tr>
<td style="text-align: left;">마지막 값</td>
<td style="text-align: left;">끝 한 칸</td>
<td style="text-align: left;">최근 값이 답일 때</td>
</tr>
<tr>
<td style="text-align: left;">첫 값</td>
<td style="text-align: left;">처음 한 칸</td>
<td style="text-align: left;">시작이 답일 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 강의는 '0.2 × 1 + 0.8 × 0.5 = 0.6'으로 어림 · 실제 비율은 0.211

> ✅ **한 줄 정리** · 마지막 값 하나의 정확도는 끝 칸이 표식인 비율로 정해지며, 실제 0.211 × 1 + 0.789 × 0.5 = 0.606과 학습 결과가 같다.

**딥러닝 코드에서는**

- 기준선 정확도를 '이론 상한'과 비교하면 모델 탓인지 입력 탓인지 바로 보임

### 1-4. `B` 모델을 키우면 · `MLP(1, width, depth)` <sub>원본 셀 11 · 12 (미리 돌린 결과)</sub>

> 🎯 **마지막 값 하나만 넘긴 채 층 · 너비 · 학습 횟수를 늘리면 60%를 넘을까?**

💡 **쉽게 말하면**

- 입력이 그대로면 **모델 크기는 상한을 못 바꿈**.

**실행 흐름**

1. 층 2 ~ 8 · 너비 32 ~ 256 · epoch 30 ~ 300
2. 같은 seed로 5번 학습

**먼저 예측** · 가장 큰 모델(8층 · 256 · 300 epoch)은 몇 %일까? — 답은 코드 실행 결과 아래에

In [5]:
# 마지막 값 하나만 넘긴 채 모델만 키우기 (원본 셀 12 · 미리 돌린 결과)
X_last, Xe_last = summarize(xtr, "last"), summarize(xte, "last")             # 입력은 끝 칸 하나
print(" 층  너비  epoch   정확도")                                           # 표 머리
# 5가지
for depth, width, ep in ((2, 32, 30), (4, 64, 30), (8, 128, 30), (4, 64, 150), (8, 256, 300)):
    torch.manual_seed(SEED)                                                  # 같은 시작점
    acc = train(MLP(1, width, depth), X_last, Ytr, Xe_last, Yte, epochs=ep)  # 크기 · 횟수만 바꿈
    print(f" {depth:>2}  {width:>4}  {ep:>5}   {acc:.4f}")                   # 한 줄 결과

 층  너비  epoch   정확도


  2    32     30   0.6065


  4    64     30   0.5865


  8   128     30   0.6070


  4    64    150   0.6235


  8   256    300   0.5935


**결과** · 예측 정답은 60% 근처 그대로 — 0.5935

- `2    32     30   0.6065` — 기본 크기
- `8   128     30   0.6070` — 층 · 너비를 4배로 늘려도 그대로
- `4    64    150   0.6235` — 가장 높아도 상한 근처 (평가 데이터에 맞춘 우연)
- `8   256    300   0.5935` — 가장 큰 모델이 오히려 낮음 (강의 48.8%)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">모델 키우기</td>
<td style="text-align: left;">같은 입력에서 표현력만 늘림</td>
<td style="text-align: left;">입력에 답이 있을 때</td>
</tr>
<tr>
<td style="text-align: left;">입력 바꾸기 (1-6 · 2장)</td>
<td style="text-align: left;">넘기는 정보를 바꿈</td>
<td style="text-align: left;">입력에 답이 없을 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 300 epoch는 시간이 오래 걸림 (원본 '미리 돌린 결과') · GPU에서 약 1분

> ✅ **한 줄 정리** · 마지막 값 하나만 넣으면 층 · 너비 · 학습을 늘려도 60% 근처를 넘지 못하므로 문제는 모델이 아니라 넘긴 것이다.

**딥러닝 코드에서는**

- 성능이 막히면 모델을 키우기 전에 입력에 정답 정보가 있는지부터 확인

### 1-5. `B` 통째로 펼치기와 데이터 양 · `flat()` · `reshape()` <sub>원본 셀 13 ~ 16</sub>

> 🎯 **줄 전체를 L × 10칸으로 펼쳐 넣으면 길이 · 데이터 양에 따라 어떻게 되는가?**

💡 **쉽게 말하면**

- 정보는 다 넘기지만 **입력 칸이 길이만큼** 커져 파라미터가 늘어남.
- 같은 길이에서 데이터를 줄이면 배울 기회가 모자람.

**실행 흐름**

1. 펼친 입력 함수 · 길이별 학습
2. 길이 50 · 데이터 6000 · 2000 · 500

**먼저 예측** · 길이 100을 펼치면 몇 %일까? — 답은 코드 실행 결과 아래에

In [6]:
# 길이 L 줄을 L × 10칸 한 줄로 펼쳐 MLP에 (원본 셀 14)
def flat(xs, L):                                   # (N, L) → (N, L*10)
    return to_tensor(xs).reshape(len(xs), L * 10)  # 칸 t의 숫자 d → 자리 t*10+d


def run_flat(L, n_train=6000):                              # 길이 · 학습량 → 정확도 · 파라미터
    xa, ya = make_order_task(n_train, L, seed=1)            # 학습 줄
    xb, yb = make_order_task(2000, L, seed=2)               # 평가 줄
    torch.manual_seed(SEED)                                 # 같은 시작점
    model = MLP(L * 10)                                     # 입력 크기가 길이를 따라 커짐
    acc = train(model, flat(xa, L), torch.tensor(ya), flat(xb, L), torch.tensor(yb))  # 학습
    return acc, sum(p.numel() for p in model.parameters())  # 정확도 · 파라미터 수


for L in (10, 20, 50, 100):                               # 길이 4가지
    acc, n = run_flat(L)                                  # 펼쳐서 학습
    print(f"길이 {L:>3} → {acc:.4f} · 파라미터 {n:,}개")  # 결과


# 길이 50 고정 · 학습 데이터만 줄이기 (원본 셀 16)
for n_train in (6000, 2000, 500):                                # 학습량 3가지
    acc, _ = run_flat(50, n_train)                               # 같은 모델 크기
    print(f"길이 50 · 학습 데이터 {n_train:>5,}건 → {acc:.4f}")  # 결과

길이  10 → 1.0000 · 파라미터 4,354개


길이  20 → 0.9955 · 파라미터 7,554개


길이  50 → 0.9585 · 파라미터 17,154개


길이 100 → 0.8840 · 파라미터 33,154개


길이 50 · 학습 데이터 6,000건 → 0.9585


길이 50 · 학습 데이터 2,000건 → 0.8935


길이 50 · 학습 데이터   500건 → 0.6695


**결과** · 예측 정답은 0.8840 — 길이 10의 1.0000에서 떨어짐

- `길이  10 → 1.0000 · 파라미터 4,354개` — 짧으면 완벽 (강의 100%)
- `길이  50 → 0.9585 · 파라미터 17,154개` — 강의 95.9%
- `길이 100 → 0.8840 · 파라미터 33,154개` — 길어질수록 떨어짐 (강의 88.4%)
- `길이 50 · 학습 데이터 2,000건 → 0.8935` — 강의 89.4%
- `길이 50 · 학습 데이터   500건 → 0.6695` — 데이터가 모자람 (강의 67.0%)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">펼치기</td>
<td style="text-align: left;">정보 손실 없음 · 크기가 길이에 묶임</td>
<td style="text-align: left;">짧고 고정 길이 · 데이터가 많을 때</td>
</tr>
<tr>
<td style="text-align: left;">요약 하나</td>
<td style="text-align: left;">크기 고정 · 정보 손실</td>
<td style="text-align: left;">질문에 맞는 요약을 알 때</td>
</tr>
<tr>
<td style="text-align: left;">RNN (2장)</td>
<td style="text-align: left;">크기 고정 요약을 학습</td>
<td style="text-align: left;">길이가 길거나 다를 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 펼친 모델은 학습 때 길이로 고정 → 더 긴 줄은 넣을 수 없음

> ✅ **한 줄 정리** · 통째로 펼치면 길이가 길수록 · 데이터가 적을수록 정확도가 떨어지며, 순서 문제를 길이 문제로 바꿔 놓을 뿐이다.

**딥러닝 코드에서는**

- 이미지를 펼쳐 MLP에 넣는 것과 같은 한계 → CNN · RNN은 같은 가중치를 재사용

### 1-6. `B` 봤는가 vs 먼저 봤는가 · `seen_two()` · `first_seen()` <sub>원본 셀 17 ~ 20</sub>

> 🎯 **'3을 봤는가 · 7을 봤는가'와 '어느 쪽을 먼저 봤는가'는 왜 결과가 정반대인가?**

💡 **쉽게 말하면**

- 3 · 7은 **늘 둘 다 있음** → '봤는가'는 모든 줄이 같은 입력.
- '먼저 봤는가' 상태는 **답 그 자체**.

**실행 흐름**

1. 봤는가 2칸으로 학습 · 입력 종류 확인
2. 끝 상태 3칸 one-hot으로 학습

**먼저 예측** · '봤는가' 2칸 입력은 몇 종류일까? — 답은 코드 실행 결과 아래에

In [7]:
# 끝까지 읽고 '3을 봤는가 · 7을 봤는가' 두 칸만 넘기기 (원본 셀 18)
def seen_two(xs):                                                      # 봤는가 2칸
    return torch.tensor([[float(3 in s), float(7 in s)] for s in xs])  # 3 있음 · 7 있음


torch.manual_seed(SEED)                                               # 같은 시작점
acc = train(MLP(2, depth=1), seen_two(xtr), Ytr, seen_two(xte), Yte, epochs=40)  # 2칸으로 학습
print(f"3 을 봤는가 · 7 을 봤는가 → {acc:.4f}")                       # 정확도
print("서로 다른 입력 종류 :", seen_two(xtr).unique(dim=0).tolist())  # 모든 줄이 같은 입력


# 상태 셋 (아직 안 봄 · 3이 먼저 · 7이 먼저) 중 끝 상태 넘기기 (원본 셀 20)
def first_seen(xs):                                                  # 끝 상태 3칸 one-hot
    return torch.tensor([[0.0, 1.0, 0.0] if s.index(3) < s.index(7)  # 3이 먼저
                         else [0.0, 0.0, 1.0] for s in xs])          # 7이 먼저


torch.manual_seed(SEED)                                                              # 같은 시작점
acc = train(MLP(3, depth=1), first_seen(xtr), Ytr, first_seen(xte), Yte, epochs=40)  # 상태로 학습
print(f"어느 쪽을 먼저 봤는가 → {acc:.4f}")                                          # 정확도

3 을 봤는가 · 7 을 봤는가 → 0.5120
서로 다른 입력 종류 : [[1.0, 1.0]]


어느 쪽을 먼저 봤는가 → 1.0000


**결과** · 예측 정답은 한 종류 — [1, 1]

- `3 을 봤는가 · 7 을 봤는가 → 0.5120` — 찍기 (강의 51.2%)
- `서로 다른 입력 종류 : [[1.0, 1.0]]` — 모든 줄이 같은 입력이라 구분 불가
- `어느 쪽을 먼저 봤는가 → 1.0000` — 상태 셋이면 완벽 (강의도 만점)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">봤는가 (2칸)</td>
<td style="text-align: left;">포함 여부</td>
<td style="text-align: left;">있는지가 답일 때</td>
</tr>
<tr>
<td style="text-align: left;">먼저 봤는가 (3칸)</td>
<td style="text-align: left;">순서 상태</td>
<td style="text-align: left;">순서가 답일 때</td>
</tr>
<tr>
<td style="text-align: left;">학습한 상태 (2장)</td>
<td style="text-align: left;">모델이 상태를 정함</td>
<td style="text-align: left;">규칙을 모를 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 상태 셋은 사람이 답을 알고 만든 것 → 다른 문제에는 다시 설계해야 함

> ✅ **한 줄 정리** · 필요한 것은 '봤는가'가 아니라 '먼저 봤는가'라는 상태이고, 이 상태를 사람 대신 학습으로 정하는 것이 다음 장의 RNN이다.

**딥러닝 코드에서는**

- 좋은 특징은 '답을 가르는 정보'를 담은 것 · 많이 넣는다고 좋은 게 아님

---
## 2. 하나씩 읽는 모델 (L1)

> **이 장의 흐름**
>
> - 원본 L1 · 모델을 만들고 파라미터를 셈
> - 손으로 펼친 판 → nn.RNN과 맞춤 → 학습 → 모양 맞추기

<img src="images/rl_hidden_flow.png" alt="hidden state 시간축 흐름 · 같은 상자를 칸마다 다시 씀" width="627">

**파이프라인 위치** · ④ 모델 — RNN 분류기

### 2-1. `A` 하나씩 읽는 모델 Net · `nn.RNN` · `o[:, -1, :]` <sub>원본 셀 21 · 22 · L1</sub>

> 🎯 **RNN 모델은 길이가 달라도 왜 같은 모델 · 같은 파라미터 수로 돌아가는가?**

💡 **쉽게 말하면**

- 한 칸씩 읽고 **요약(32칸)을 다음 칸으로** 넘김.
- 요약 크기가 고정이라 상자 하나를 칸마다 다시 씀.

**실행 흐름**

1. Net 정의 (RNN · LSTM · GRU 선택)
2. 길이 10 · 50 · 200 넣기
3. 파라미터 손계산

**먼저 예측** · 길이 200을 넣으면 파라미터 수는? — 답은 코드 실행 결과 아래에

In [8]:
# 한 칸씩 읽고 요약을 넘기는 모델 (원본 셀 22)
class Net(nn.Module):                              # RNN 분류 모델
    def __init__(self, kind="RNN", hid=32):        # 셀 종류 · 요약 크기
        super().__init__()                         # nn.Module 준비 (필수)
        layer = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind]  # 이름으로 층 고르기
        self.r = layer(10, hid, batch_first=True)  # 입력 10 · (배치, 길이, 입력)
        self.o = nn.Linear(hid, 2)                 # 요약 → 점수 2개

    def forward(self, x):           # 입력이 지나가는 길
        o, _ = self.r(x)            # o: (N, L, hid)
        return self.o(o[:, -1, :])  # 마지막 칸 요약 → (N, 2)


net = Net("RNN")                                                        # 학습 전 모델 하나
for L in (10, 50, 200):                                                 # 길이만 바꿈
    xs, _ = make_order_task(4, L, seed=3)                               # 길이 L 4줄
    out = net(to_tensor(xs))                                            # 같은 모델에 넣음
    n = sum(p.numel() for p in net.parameters())                        # 파라미터 수
    print(f"길이 {L:>3} → 출력 {tuple(out.shape)} · 파라미터 {n:,}개")  # 길이와 무관
print("손계산 :", 32 * 10 + 32 * 32 + 32 + 32, "+", 32 * 2 + 2)         # RNN + Linear

길이  10 → 출력 (4, 2) · 파라미터 1,474개
길이  50 → 출력 (4, 2) · 파라미터 1,474개
길이 200 → 출력 (4, 2) · 파라미터 1,474개
손계산 : 1408 + 66


**결과** · 예측 정답은 그대로 1,474개

- `길이  10 → 출력 (4, 2) · 파라미터 1,474개` — 줄 4개 · 답 2개
- `길이 200 → 출력 (4, 2) · 파라미터 1,474개` — 길이가 20배여도 그대로
- `손계산 : 1408 + 66` — RNN 32 × (10 + 32) + 32 + 32 · Linear 32 × 2 + 2

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>Net('RNN')</code></td>
<td style="text-align: left;">요약 32 · 마지막 칸 → 2점수</td>
<td style="text-align: left;">L1 · L2</td>
</tr>
<tr>
<td style="text-align: left;"><code>Net('LSTM')</code> · <code>Net('GRU')</code></td>
<td style="text-align: left;">같은 틀 · 셀만 교체</td>
<td style="text-align: left;">비교 실험</td>
</tr>
<tr>
<td style="text-align: left;">펼치기 MLP (1-5)</td>
<td style="text-align: left;">길이마다 크기가 다름</td>
<td style="text-align: left;">비교 기준</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `self.r` · `self.o`는 원본 이름 그대로 (원본은 `self` 대신 `s`라고 적음)

> ✅ **한 줄 정리** · RNN은 고정 크기 요약을 칸마다 같은 상자로 갱신하므로 길이와 상관없이 파라미터가 1,474개로 같다.

**딥러닝 코드에서는**

- `nn.RNN(input_size, hidden_size, batch_first=True)` + `nn.Linear(hidden, 클래스)`가 시퀀스 분류의 기본 틀

### 2-2. `A` 손으로 펼친 판과 nn.RNN · `nn.RNNCell` · `torch.stack()` <sub>원본 셀 23 ~ 26</sub>

> 🎯 **RNNCell + for 문으로 계산한 요약은 nn.RNN의 output과 정말 같은가?**

💡 **쉽게 말하면**

- 같은 가중치를 복사해 두면 **for 문 판**과 **감싼 판**의 결과가 같아야 함.

**실행 흐름**

1. nn.RNN 가중치를 cell에 복사
2. 4줄 · 10칸 for 문
3. 요약 10개를 쌓아 output과 비교

**먼저 예측** · 두 결과의 최대 차이는? — 답은 코드 실행 결과 아래에

In [9]:
# 손으로 펼친 판: RNNCell 하나 + for 문 (원본 셀 24)
torch.manual_seed(SEED)                     # 같은 시작점
rnn = nn.RNN(10, 32, batch_first=True)      # PyTorch가 감싸 둔 판
cell = nn.RNNCell(10, 32)                   # 상자 하나: 입력 10 · 요약 32
with torch.no_grad():                       # 맞춰 보려고 같은 가중치를 복사
    cell.weight_ih.copy_(rnn.weight_ih_l0)  # 입력 가중치
    cell.weight_hh.copy_(rnn.weight_hh_l0)  # 이전 요약 가중치
    cell.bias_ih.copy_(rnn.bias_ih_l0)      # bias 두 개
    cell.bias_hh.copy_(rnn.bias_hh_l0)      # (더해서 씀)

X = to_tensor(xtr[:4])                                         # 학습 데이터 네 줄 (4, 10, 10)
h = torch.zeros(4, 32)                                         # 처음 요약 = 0
hs = []                                                        # 칸마다 요약을 모음
for t in range(10):                                            # 10칸 차례로
    h = cell(X[:, t, :], h)                                    # 이번 칸 숫자 + 직전 요약 → 새 요약
    hs.append(h)                                               # 이번 칸 요약 저장
print("h.shape :", tuple(h.shape))                             # 길이 축이 사라짐
print("h[0, :4] :", [round(v, 4) for v in h[0, :4].tolist()])  # 첫 줄 요약 앞 4칸


# nn.RNN 과 맞춰 보기 (원본 셀 26)
manual = torch.stack(hs, 1)                          # 요약 10개 → (4, 10, 32)
output, h_n = rnn(X)                                 # 같은 입력 · 같은 가중치
print("(output - manual).abs().max() :", (output - manual).abs().max().item())  # 0이면 같은 계산
# 세 모양
print("x :", tuple(X.shape), "· output :", tuple(output.shape), "· h_n :", tuple(h_n.shape))
print("torch.allclose(output[:, -1, :], h_n[0]) :",  # 마지막 칸 = 마지막 요약
      torch.allclose(output[:, -1, :], h_n[0]))

h.shape : (4, 32)
h[0, :4] : [0.101, -0.3322, 0.0603, 0.02]
(output - manual).abs().max() : 0.0
x : (4, 10, 10) · output : (4, 10, 32) · h_n : (1, 4, 32)
torch.allclose(output[:, -1, :], h_n[0]) : True


**결과** · 예측 정답은 `0.0`

- `h.shape : (4, 32)` — 줄 4 · 요약 32 · 길이 축은 없음
- `(output - manual).abs().max() : 0.0` — 완전히 같은 계산
- `x : (4, 10, 10) · output : (4, 10, 32) · h_n : (1, 4, 32)` — output은 칸마다 · h_n은 마지막
- `torch.allclose(output[:, -1, :], h_n[0]) : True` — 마지막 칸 요약 = 마지막 요약

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>RNNCell</code> + for</td>
<td style="text-align: left;">칸마다 개입 가능</td>
<td style="text-align: left;">기울기 측정 (6장)</td>
</tr>
<tr>
<td style="text-align: left;"><code>nn.RNN</code></td>
<td style="text-align: left;">빠름 (cuDNN)</td>
<td style="text-align: left;">학습</td>
</tr>
<tr>
<td style="text-align: left;"><code>torch.stack(hs, 1)</code></td>
<td style="text-align: left;">요약 목록 → (N, L, H)</td>
<td style="text-align: left;">output과 맞춰 볼 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 원본은 `print(h)`로 32 × 4 값을 모두 출력 · 여기서는 앞 4칸만

> ✅ **한 줄 정리** · 같은 가중치의 RNNCell + for 문과 nn.RNN은 차이 0.0으로 같은 계산이며, nn.RNN은 칸별 요약 output과 마지막 요약 h_n을 준다.

**딥러닝 코드에서는**

- 커스텀 셀을 만들 때 nn.RNN과 같은 값을 내는지 이렇게 확인

### 2-3. `A` 학습시켜 보기 · `train(Net('RNN'), ...)` <sub>원본 셀 27 · 28</sub>

> 🎯 **넘길 것을 학습으로 정하면 1장의 기준선들과 비교해 몇 %인가?**

💡 **쉽게 말하면**

- 1장과 **같은 데이터 · 같은 학습 함수** · 모델만 RNN.

**실행 흐름**

1. one-hot 입력 (6000, 10, 10)
2. 30 epoch 학습 · 평가

**먼저 예측** · RNN은 몇 %일까? — 답은 코드 실행 결과 아래에

In [10]:
# 학습시켜 보기 (원본 셀 28): 마지막 요약 → 네 · 아니요
torch.manual_seed(SEED)                                            # 같은 시작점
acc = train(Net("RNN"), to_tensor(xtr), Ytr, to_tensor(xte), Yte)  # 1장과 같은 데이터 · 학습 함수
print(f"RNN {acc:.4f}")                                            # 요약을 학습으로 정한 결과

RNN 1.0000


**결과** · 예측 정답은 1.0000

- `RNN 1.0000` — 1장 기준선(합 0.4880 · 마지막 값 0.6065)보다 훨씬 높음 (강의도 만점)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">사람이 정한 요약</td>
<td style="text-align: left;">질문에 맞을 때만</td>
<td style="text-align: left;">기준선</td>
</tr>
<tr>
<td style="text-align: left;">학습한 요약 (RNN)</td>
<td style="text-align: left;">무엇을 담을지 학습</td>
<td style="text-align: left;">규칙을 모를 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 길이 10에서는 펼치기도 1.0000 → RNN의 장점은 길이가 길거나 다를 때 드러남

> ✅ **한 줄 정리** · 넘길 것을 학습으로 정한 RNN은 같은 조건에서 1.0000을 내며, 사람이 정한 요약들의 한계를 넘는다.

**딥러닝 코드에서는**

- `model = Net('RNN')` → 같은 `train()` 함수 · 같은 데이터로 모델만 교체해 비교

### 2-4. `A` 모양 맞추기 · one_hot · batch_first · `F.one_hot()` · `batch_first` <sub>원본 셀 29 · 30</sub>

> 🎯 **숫자 줄을 그대로 넣으면 왜 에러가 나고, batch_first를 끄면 무엇이 바뀌는가?**

💡 **쉽게 말하면**

- 입력 세 겹의 **마지막 겹 = 표현 크기 10** · 숫자는 one-hot으로 열 칸에.
- `batch_first`는 앞 두 겹 순서 · 끄면 **에러 없이** 다르게 읽음.

**실행 흐름**

1. 축만 붙여 넣기 → 에러
2. one_hot으로 통과
3. batch_first 켜고 끄기

**먼저 예측** · 끈 채로 (8, 10, 10)을 넣으면 h_n은? — 답은 코드 실행 결과 아래에

In [11]:
# 모양 맞추기: 축만 붙이면 에러 (원본 셀 30)
digits = torch.tensor([xtr[0]])                          # (1, 10) · 숫자 그대로
rnn = nn.RNN(10, 32, batch_first=True)                   # 칸마다 숫자 10개를 기대
x_bad = digits.unsqueeze(-1).float()                     # (1, 10) → (1, 10, 1)
print("unsqueeze 로 축만 붙이면 :", tuple(x_bad.shape))  # 마지막 축이 1
try:
    rnn(x_bad)                                           # 칸마다 1개짜리 입력
except RuntimeError as e:                                # 에러를 잡아 이어서 실행
    print("  에러 :", str(e).splitlines()[0])            # 에러 첫 줄


# one_hot: 숫자 하나를 열 칸에 적기
x = F.one_hot(digits, num_classes=10).float()           # (1, 10) → (1, 10, 10)
output, h_n = rnn(x)                                    # 이번엔 통과
print("one_hot 으로 열 칸에 적으면 :", tuple(x.shape),  # 입력 모양
      "→ output", tuple(output.shape), "· h_n", tuple(h_n.shape))


# batch_first: 같은 (8, 10, 10)을 켜고 끄고
x8 = torch.randn(8, 10, 10)                           # 8줄 · 10칸 · 10개 (앞 두 축 길이가 다름)
for bf in (True, False):                              # 두 설정
    output, h_n = nn.RNN(10, 32, batch_first=bf)(x8)  # 에러 없이 둘 다 돎
    # 모양 비교
    print(f"batch_first={str(bf):<5}  x {tuple(x8.shape)} · output {tuple(output.shape)}"
          f" · h_n {tuple(h_n.shape)}")

unsqueeze 로 축만 붙이면 : (1, 10, 1)
  에러 : input.size(-1) must be equal to input_size. Expected 10, got 1
one_hot 으로 열 칸에 적으면 : (1, 10, 10) → output (1, 10, 32) · h_n (1, 1, 32)
batch_first=True   x (8, 10, 10) · output (8, 10, 32) · h_n (1, 8, 32)
batch_first=False  x (8, 10, 10) · output (8, 10, 32) · h_n (1, 10, 32)


**결과** · 예측 정답은 `(1, 10, 32)` — 10을 줄 수로 읽음

- `unsqueeze 로 축만 붙이면 : (1, 10, 1)` — 세 겹은 맞지만 마지막이 1
- `에러 : input.size(-1) must be equal to input_size. Expected 10, got 1` — 칸마다 10개를 기대
- `one_hot 으로 열 칸에 적으면 : (1, 10, 10) → output (1, 10, 32) · h_n (1, 1, 32)` — 통과
- `batch_first=False  x (8, 10, 10) · output (8, 10, 32) · h_n (1, 10, 32)` — 8을 길이로 · 10을 줄 수로

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>unsqueeze(-1)</code></td>
<td style="text-align: left;">크기 1 축</td>
<td style="text-align: left;">값 하나가 특성 하나일 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>F.one_hot</code></td>
<td style="text-align: left;">번호 → 열 칸</td>
<td style="text-align: left;">번호가 범주일 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>batch_first=True</code></td>
<td style="text-align: left;">(N, L, I)</td>
<td style="text-align: left;">기본으로 쓸 것</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - output 모양은 두 설정 모두 (8, 10, 32)라 h_n으로 확인해야 함

> ✅ **한 줄 정리** · 번호 입력은 one_hot으로 표현 크기를 맞추고, `batch_first`를 끄면 앞 두 축을 바꿔 에러 없이 틀리게 읽는다.

**딥러닝 코드에서는**

- 입력을 넣기 직전에 `print(x.shape)` · 모델 안 첫 줄에 `assert x.dim() == 3`

### 2-5. `B` 정말 같은 값일까 · `torch.equal()` · `num_layers=2` <sub>원본 셀 31 · 32 (+ 강의 '그대로 쓸 수 있는 조건')</sub>

> 🎯 **`output[:, -1]`과 `h_n[0]`은 비트까지 같은가, 그리고 2층 · 양방향이면?**

💡 **쉽게 말하면**

- 1층 · 한 방향이면 **비트까지 같음**.
- 2층이면 h_n이 층마다 · 양방향이면 output이 두 배.

**실행 흐름**

1. equal · allclose
2. 2층에서 h_n[-1] · h_n[0] 비교
3. 양방향 모양

**먼저 예측** · 2층에서 `output[:, -1]`과 같은 것은 h_n[0]일까 h_n[-1]일까? — 답은 코드 실행 결과 아래에

In [12]:
# 정말 같은 값일까 (원본 셀 32): equal = 비트까지 · allclose = 작은 오차 허용
torch.manual_seed(SEED)                 # 같은 시작점
rnn = nn.RNN(10, 32, batch_first=True)  # 1층 · 한 방향
output, h_n = rnn(to_tensor(xtr[:8]))   # 8줄 넣기
# 비트 비교
print("torch.equal(output[:, -1, :], h_n[0])    :", torch.equal(output[:, -1, :], h_n[0]))
# 오차 허용
print("torch.allclose(output[:, -1, :], h_n[0]) :", torch.allclose(output[:, -1, :], h_n[0]))


# 같다는 조건이 깨지면: 2층 · 양방향 (강의 슬라이드 '그대로 쓸 수 있는 조건')
two = nn.RNN(10, 32, num_layers=2, batch_first=True)       # 2층
output, h_n = two(to_tensor(xtr[:8]))                      # h_n이 층마다 하나
last = torch.allclose(output[:, -1], h_n[-1])              # 마지막 층과 비교
first = torch.allclose(output[:, -1], h_n[0])              # 첫 층과 비교
print("2층   h_n :", tuple(h_n.shape),                     # 층마다 하나
      "· output[:, -1] == h_n[-1] :", last, "· == h_n[0] :", first)
bi = nn.RNN(10, 32, bidirectional=True, batch_first=True)  # 양방향
output, h_n = bi(to_tensor(xtr[:8]))                       # output 마지막 축 = 32 × 2
print("양방향 output :", tuple(output.shape), "· h_n :", tuple(h_n.shape))  # 방향 수만큼

torch.equal(output[:, -1, :], h_n[0])    : True
torch.allclose(output[:, -1, :], h_n[0]) : True
2층   h_n : (2, 8, 32) · output[:, -1] == h_n[-1] : True · == h_n[0] : False
양방향 output : (8, 10, 64) · h_n : (2, 8, 32)


**결과** · 예측 정답은 `h_n[-1]` — 마지막 층

- `torch.equal(output[:, -1, :], h_n[0])    : True` — 비트까지 같음
- `2층   h_n : (2, 8, 32) · output[:, -1] == h_n[-1] : True · == h_n[0] : False` — output은 마지막 층의 값
- `양방향 output : (8, 10, 64) · h_n : (2, 8, 32)` — 두 방향을 이어 붙임 · 방향마다 h_n

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>h_n[0]</code></td>
<td style="text-align: left;">첫 층(또는 앞 방향)</td>
<td style="text-align: left;">1층 · 한 방향</td>
</tr>
<tr>
<td style="text-align: left;"><code>h_n[-1]</code></td>
<td style="text-align: left;">마지막 층(또는 뒤 방향)</td>
<td style="text-align: left;">여러 층 분류</td>
</tr>
<tr>
<td style="text-align: left;"><code>output[:, -1]</code></td>
<td style="text-align: left;">마지막 층의 마지막 칸</td>
<td style="text-align: left;">패딩 없을 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 양방향에서 `output[:, -1]`의 뒤쪽 절반은 뒤 방향이 1칸만 읽은 값

> ✅ **한 줄 정리** · 1층 · 한 방향에서만 `output[:, -1]`과 `h_n[0]`이 같고, 여러 층이면 마지막 층 `h_n[-1]`과 같다.

**딥러닝 코드에서는**

- 여러 층 · 양방향 분류는 `h_n[-1]` 또는 `torch.cat([h_n[-2], h_n[-1]], 1)`

---
## 3. HAR 데이터와 패딩 준비

> **이 장의 흐름**
>
> - 원본 L1 뒤쪽 · HAR을 열어 모양을 보고 뒤에서 쓸 정의를 실행
> - 길이가 다른 세 줄을 채워 봄

<img src="images/rn_har.png" alt="HAR 한 줄 = (128칸, 9채널) · 걷기와 눕기" width="895">

**파이프라인 위치** · ③ 전처리 — 텐서 만들기 · 정규화 · 패딩

### 3-1. `A` UCI HAR 열어보기 · 정규화 · `urllib.request` · `permute()` <sub>원본 셀 33 · 34 앞부분</sub>

> 🎯 **HAR 데이터는 어떻게 (개수, 128칸, 9채널) 텐서가 되고, 정규화는 무엇으로 하는가?**

💡 **쉽게 말하면**

- 압축 안에 압축이 또 있음 · **Inertial Signals**의 9채널 파일을 읽음 (`X_train.txt`는 561차원 요약이라 안 씀).
- 정규화는 **학습 데이터의 채널별** 평균 · 표준편차로만 · 평가에도 같은 값.

**실행 흐름**

1. 처음 한 번 내려받기 · 두 번 풀기
2. 9채널 읽기 → (N, 128, 9)
3. 라벨 1 ~ 6 → 0 ~ 5
4. 학습 통계로 정규화

**먼저 예측** · 학습 데이터의 모양은? — 답은 코드 실행 결과 아래에

In [13]:
# UCI HAR 내려받기 (처음 한 번만 · 약 60MB · 노트북 옆 UCI_HAR/ 폴더)
URL = ("https://archive.ics.uci.edu/static/public/240/"                  # 원본과 같은 주소
       "human+activity+recognition+using+smartphones.zip")
ROOT = os.path.join("UCI_HAR", "UCI HAR Dataset")  # 압축 푼 폴더 (원본은 노트북 폴더 바로 아래)
if not os.path.exists(ROOT):                                             # 처음 한 번만
    os.makedirs("UCI_HAR", exist_ok=True)                                # 받을 폴더
    urllib.request.urlretrieve(URL, os.path.join("UCI_HAR", "har.zip"))  # 내려받기
    # 바깥
    zipfile.ZipFile(os.path.join("UCI_HAR", "har.zip")).extract("UCI HAR Dataset.zip", "UCI_HAR")
    # 안쪽 압축
    zipfile.ZipFile(os.path.join("UCI_HAR", "UCI HAR Dataset.zip")).extractall("UCI_HAR")

CH = ["body_acc_x", "body_acc_y", "body_acc_z",  # 9채널 = 신호 3종 × 축 3개
      "body_gyro_x", "body_gyro_y", "body_gyro_z",
      "total_acc_x", "total_acc_y", "total_acc_z"]


def load_har(split):                                       # train · test → (N, 128, 9) · 라벨
    base = os.path.join(ROOT, split)                       # train 또는 test 폴더
    chans = []                                             # 채널마다 표
    for c in CH:                                           # 9채널
        with open(os.path.join(base, "Inertial Signals", f"{c}_{split}.txt")) as f:  # 채널 파일
            chans.append([[float(v) for v in line.split()] for line in f])  # 줄마다 128개
    X = torch.tensor(chans).permute(1, 2, 0).contiguous()  # (9, N, 128) → (N, 128, 9)
    with open(os.path.join(base, f"y_{split}.txt")) as f:  # 정답 파일
        y = torch.tensor([int(v) - 1 for v in f.read().split()])  # 라벨 1–6 → 0–5
    return X, y                                            # 입력 · 정답


Xh, Yh = load_har("train")                     # 학습 7352줄
Xh_te, Yh_te = load_har("test")                # 평가 2947줄
print("X.shape   :", tuple(Xh.shape))          # (개수, 길이, 채널)
print("Xte.shape :", tuple(Xh_te.shape))       # 평가도 128칸
mu = Xh.mean((0, 1), keepdim=True)             # 학습 데이터 채널별 평균 (1, 1, 9)
sd = Xh.std((0, 1), keepdim=True)              # 채널별 표준편차
Xn, Xn_te = (Xh - mu) / sd, (Xh_te - mu) / sd  # 평가도 학습 통계로 정규화
print(f"정규화 후 train 평균 {Xn.mean():.3f} · 표준편차 {Xn.std():.3f}")  # 0 · 1 확인

X.shape   : (7352, 128, 9)
Xte.shape : (2947, 128, 9)
정규화 후 train 평균 0.000 · 표준편차 1.000


**결과** · 예측 정답은 `(7352, 128, 9)`

- `X.shape   : (7352, 128, 9)` — 7352개 · 128칸(2.56초) · 9채널
- `Xte.shape : (2947, 128, 9)` — 평가 2947개
- `정규화 후 train 평균 0.000 · 표준편차 1.000` — 학습 통계로 맞춤

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>Inertial Signals</code> 9채널</td>
<td style="text-align: left;">시퀀스 (128칸)</td>
<td style="text-align: left;">RNN 입력</td>
</tr>
<tr>
<td style="text-align: left;"><code>X_train.txt</code> 561차원</td>
<td style="text-align: left;">미리 계산한 요약 특징</td>
<td style="text-align: left;">MLP · 트리 모델</td>
</tr>
<tr>
<td style="text-align: left;">학습 통계 정규화</td>
<td style="text-align: left;">평가 정보 누설 없음</td>
<td style="text-align: left;">항상 (RNN-2)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 데이터 폴더는 노트북 옆 `UCI_HAR/` (원본은 노트북 폴더 바로 아래 · 약 60MB · 올리지 않음)
> - 출처 · UCI HAR (Human Activity Recognition Using Smartphones) · CC BY 4.0

> ✅ **한 줄 정리** · HAR은 9채널 파일을 읽어 (7352, 128, 9)로 쌓고, 학습 데이터의 채널별 평균 · 표준편차로 학습 · 평가를 정규화한다.

**딥러닝 코드에서는**

- `Dataset.__getitem__`에서 (128, 9) 한 줄을 돌려주면 DataLoader가 (배치, 128, 9)로 묶음

### 3-2. `A` HAR 모델 · 학습 함수 정의 · `HarNet` · `run_har()` · `clip_grad_norm_()` <sub>원본 셀 34 뒷부분</sub>

> 🎯 **8장 비교와 과제에서 쓰는 HAR 모델과 학습 조건은 무엇인가?**

💡 **쉽게 말하면**

- 입력 9 · 출력 6 · 마지막 칸 요약 → 활동 6가지.
- Adam 1e-3 · 배치 64 · 12 epoch · **기울기 자르기 5.0** · 같은 seed.

**실행 흐름**

1. HarNet 정의
2. run_har · report 정의
3. LSTM 64 파라미터 확인

**먼저 예측** · HarNet('LSTM', 64)의 파라미터 수는? — 답은 코드 실행 결과 아래에

In [14]:
# 뒤 8장 · 과제에서 쓰는 정의 (원본 셀 34 뒷부분)
class HarNet(nn.Module):                                               # HAR 분류 모델
    def __init__(self, kind, hid):                                     # 셀 종류 · hidden
        super().__init__()                                             # nn.Module 준비 (필수)
        layer = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind]  # 이름으로 층 고르기
        self.r = layer(9, hid, batch_first=True)                       # 입력 9채널
        self.o = nn.Linear(hid, 6)                                     # 활동 6가지

    def forward(self, x):           # 입력이 지나가는 길
        o, _ = self.r(x)            # 모든 칸 요약
        return self.o(o[:, -1, :])  # 마지막 칸 → 점수 6개


def run_har(kind, hid, seed=SEED, Xa=None, Xb=None, epochs=12):  # 학습 → 정확도 · 파라미터 · 초
    Xa = (Xn if Xa is None else Xa).to(DEV)                      # 기본은 정규화한 train
    Xb = (Xn_te if Xb is None else Xb).to(DEV)                   # 기본은 정규화한 test
    Ya, Yb = Yh.to(DEV), Yh_te.to(DEV)                           # 정답도 같은 장치로
    random.seed(seed)                                            # seed마다 다른 시작점
    torch.manual_seed(seed)                                      # 모델 초기값 · 섞는 순서
    m = HarNet(kind, hid).to(DEV)                                # 새 모델
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)              # Adam 1e-3
    lf = nn.CrossEntropyLoss()                                   # 분류 손실
    t = time.time()                                              # 시간 재기 시작
    for _ in range(epochs):                                      # epoch 반복
        p = torch.randperm(len(Xa))                              # 순서 섞기 (CPU 난수)
        for i in range(0, len(Xa), 64):                          # 64줄씩
            j = p[i:i + 64]                                      # 이번 배치
            opt.zero_grad()                                      # 지난 기울기 지우기
            lf(m(Xa[j]), Ya[j]).backward()                       # 손실 → 기울기
            torch.nn.utils.clip_grad_norm_(m.parameters(), 5.0)  # 기울기 폭발 방지
            opt.step()                                           # 가중치 한 걸음 수정
    with torch.no_grad():                                        # 평가는 기울기 기록 안 함
        acc = (m(Xb).argmax(1) == Yb).float().mean().item()      # test 정확도
    return round(acc, 4), sum(q.numel() for q in m.parameters()), time.time() - t  # 결과 셋


def report(kind, hid):                # 한 줄 보고
    acc, n, sec = run_har(kind, hid)  # 학습 · 평가
    # 초는 환경마다 다름
    print(f"{kind:<5} hidden {hid:<4} 정확도 {acc:.4f} · 파라미터 {n:>6,} · {sec:.1f}s")


n_lstm = sum(q.numel() for q in HarNet("LSTM", 64).parameters())  # 정의 확인
print("HarNet('LSTM', 64) 파라미터 :", n_lstm)                    # 8장에서 다시 셈

HarNet('LSTM', 64) 파라미터 : 19590


**결과** · 예측 정답은 19,590개

- `HarNet('LSTM', 64) 파라미터 : 19590` — 4 × (64 × (9 + 64) + 2 × 64) + 64 × 6 + 6

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>run_har(kind, hid)</code></td>
<td style="text-align: left;">정확도 · 파라미터 · 초</td>
<td style="text-align: left;">같은 조건 비교</td>
</tr>
<tr>
<td style="text-align: left;"><code>seed=</code></td>
<td style="text-align: left;">초기값 · 섞는 순서</td>
<td style="text-align: left;">흔들림 측정 (8-2)</td>
</tr>
<tr>
<td style="text-align: left;"><code>Xa=</code> · <code>Xb=</code></td>
<td style="text-align: left;">입력만 바꿈</td>
<td style="text-align: left;">섞기 실험 (8-4 · 과제 4)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 실행 시간(초)은 환경마다 다름 · 원본은 CPU 기준

> ✅ **한 줄 정리** · HAR 비교는 셀 종류 · hidden · seed · 입력만 바꾸고 나머지(Adam 1e-3 · 배치 64 · 12 epoch · clip 5.0)는 고정한다.

**딥러닝 코드에서는**

- 실험 함수에 바꿀 것만 인자로 두면 통제된 비교가 쉬워짐

### 3-3. `A` 길이가 다른 세 줄 채우기 · `torch.zeros()` · `lens` <sub>원본 셀 35 ~ 40</sub>

> 🎯 **길이가 3 · 6 · 4인 줄은 어떻게 한 텐서에 담고, 무엇으로 채워야 하는가?**

💡 **쉽게 말하면**

- 그대로는 **직사각형이 아니라** 에러.
- 가장 긴 6칸 틀 · 빈 칸은 **열 칸 모두 0** (숫자 0과 구분).

**실행 흐름**

1. 그대로 텐서 → 에러
2. 6칸 틀에 채우기 · lens
3. 빈 벡터 vs 숫자 0 비교

**먼저 예측** · 숫자 0으로 채운 칸은 진짜 0 칸과 구분될까? — 답은 코드 실행 결과 아래에

In [15]:
# 길이가 다른 세 줄을 그대로 텐서로 (원본 셀 36)
seqs = [[3, 1, 7],                      # 길이 3
        [9, 7, 0, 5, 3, 2],             # 길이 6
        [1, 3, 4, 7]]                   # 길이 4
try:
    torch.tensor(seqs)                  # 직사각형이 아니라 실패
except ValueError as e:                 # 에러를 잡아 이어서 실행
    print("그대로 텐서로 만들면 :", e)  # 에러 내용


# 세 줄 채워 보기 (원본 셀 38): 가장 긴 6칸 틀 · 빈 칸은 열 칸 모두 0
ML = max(len(s) for s in seqs)                                 # 가장 긴 길이 6
X = torch.zeros(len(seqs), ML, 10)                             # (3, 6, 10) 빈 틀
for b, s in enumerate(seqs):                                   # 줄마다
    for t, d in enumerate(s):                                  # 진짜 칸만
        X[b, t, d] = 1.0                                       # 숫자 d 자리에 1
lens = torch.tensor([len(s) for s in seqs])                    # 진짜 길이표
print("X.shape :", tuple(X.shape), "· lens :", lens.tolist())  # 모양 · 길이


# 무엇으로 채웠는지 (원본 셀 40): 빈 벡터 vs 숫자 0
X0 = X.clone()                                                            # 복사본
X0[0, 3:, 0] = 1.0  # 첫 줄 빈 칸을 '숫자 0'(0번 자리 1)으로 채우면
print("빈 벡터로 채움   · 첫 줄 칸별 합 :", X[0].sum(1).int().tolist())   # 빈 칸 = 0
print("숫자 0 으로 채움 · 첫 줄 칸별 합 :", X0[0].sum(1).int().tolist())  # 빈 칸도 1
# 구분 불가
print("숫자 0 으로 채운 빈 칸 == 둘째 줄의 진짜 0 칸 :", torch.equal(X0[0, 3], X[1, 2]))

그대로 텐서로 만들면 : expected sequence of length 3 at dim 1 (got 6)
X.shape : (3, 6, 10) · lens : [3, 6, 4]
빈 벡터로 채움   · 첫 줄 칸별 합 : [1, 1, 1, 0, 0, 0]
숫자 0 으로 채움 · 첫 줄 칸별 합 : [1, 1, 1, 1, 1, 1]
숫자 0 으로 채운 빈 칸 == 둘째 줄의 진짜 0 칸 : True


**결과** · 예측 정답은 안 됨 — True

- `그대로 텐서로 만들면 : expected sequence of length 3 at dim 1 (got 6)` — 직사각형이 아님
- `X.shape : (3, 6, 10) · lens : [3, 6, 4]` — 길이표를 따로 둠
- `빈 벡터로 채움   · 첫 줄 칸별 합 : [1, 1, 1, 0, 0, 0]` — 빈 칸이 표시됨
- `숫자 0 으로 채운 빈 칸 == 둘째 줄의 진짜 0 칸 : True` — 구분 불가

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">빈 벡터</td>
<td style="text-align: left;">채운 칸 = 모두 0</td>
<td style="text-align: left;">one-hot 입력</td>
</tr>
<tr>
<td style="text-align: left;">숫자 0</td>
<td style="text-align: left;">진짜 0과 같음</td>
<td style="text-align: left;">쓰지 않음</td>
</tr>
<tr>
<td style="text-align: left;">lens</td>
<td style="text-align: left;">진짜 길이</td>
<td style="text-align: left;">읽을 때 (4장)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 빈 벡터로 채워도 RNN은 빈 칸을 계산함 → 꺼내는 법이 4장

> ✅ **한 줄 정리** · 길이가 다른 줄은 가장 긴 틀에 빈 벡터로 채우고 진짜 길이 lens를 따로 들고 다닌다.

**딥러닝 코드에서는**

- `pad_sequence(batch_first=True)` + `lengths` (RNN-3)

---
## 4. 빈 칸을 읽으면 (L1b)

> **이 장의 흐름**
>
> - 원본 L1b · 같은 데이터 · 같은 모델 · 꺼내는 법만 4가지
> - 길이 편차를 줄이면 차이가 사라지는 것까지

<img src="images/rl_padding.png" alt="가장 긴 6칸 틀에서 -1과 lens - 1이 가리키는 자리" width="686">

**파이프라인 위치** · ③ 전처리 → ④ 모델 — 패딩 배치 읽기

### 4-1. `B` 꺼내는 법 4가지 모델 · `ReadNet` · `make_ragged()` · `pad()` <sub>원본 셀 41 · 42 앞부분 · L1b</sub>

> 🎯 **같은 RNN에서 요약을 꺼내는 법만 다른 모델 네 가지는 어떻게 만드는가?**

💡 **쉽게 말하면**

- RNN · 분류기는 같고 `forward`에서 **요약을 꺼내는 줄만** 다름.
- 꺼내려면 진짜 길이 `lens`를 입력과 함께 받아야 함.

**실행 흐름**

1. 길이 제각각 데이터 함수 · 채우기 함수
2. ReadNet 정의 (-1 · lens - 1 · 평균 · mask 평균)
3. 3-3의 세 줄로 모양 확인

**먼저 예측** · 세 줄을 넣으면 출력 모양은? — 답은 코드 실행 결과 아래에

In [16]:
# 길이 5~60 줄을 빈 벡터로 채워 한 텐서에 · 요약 꺼내는 법 4가지 (원본 셀 42)
def make_ragged(n, lo, hi, seed):                                # 길이가 제각각인 줄
    rng = random.Random(seed)                                    # 전용 난수
    xs, ys = [], []                                              # 입력 줄 · 정답 목록
    for _ in range(n):                                           # 줄 n개
        L = rng.randint(lo, hi)                                  # 줄마다 길이
        x, y = make_order_task(1, L, seed=rng.randrange(10**6))  # 줄 하나
        xs.append(x[0])                                          # 줄 저장
        ys.append(y[0])                                          # 정답 저장
    return xs, ys                                                # 줄 목록 · 정답 목록


def pad(xs, T):                            # 길이 T 틀에 채우기 (뒤는 빈 벡터)
    X = torch.zeros(len(xs), T, 10)        # 빈 틀
    for b, s in enumerate(xs):             # 줄마다
        X[b, :len(s)] = to_tensor([s])[0]  # 앞쪽에 진짜 칸
    return X                               # 채운 텐서


class ReadNet(nn.Module):                          # 요약 꺼내는 방법만 다른 모델
    def __init__(self, how):                       # 꺼내는 방법
        super().__init__()                         # nn.Module 준비 (필수)
        self.r = nn.RNN(10, 32, batch_first=True)  # 같은 RNN
        self.o = nn.Linear(32, 2)                  # 같은 분류기
        self.how = how                             # 방법 기억

    def forward(self, x, lens):                                       # 입력 + 길이표
        out, _ = self.r(x)                                            # (N, T, 32)
        if self.how == "마지막 칸(-1)":                               # 틀의 끝
            h = out[:, -1, :]                                         # 빈 칸을 지난 값일 수 있음
        elif self.how == "실제 마지막 칸":                            # 줄마다 진짜 끝
            h = out[torch.arange(len(x), device=x.device), lens - 1]  # 줄마다 자기 끝
        elif self.how == "전부 평균":                                 # 빈 칸까지 평균
            h = out.mean(1)                                           # 길이 축 평균
        else:                                                         # mask 평균: 진짜 칸만 1
            m = (torch.arange(x.shape[1], device=x.device)[None, :] < lens[:, None])  # (N, T)
            m = m.float().unsqueeze(-1)                               # (N, T, 1)
            h = (out * m).sum(1) / m.sum(1)                           # 진짜 칸만 평균
        return self.o(h)                                              # 요약 → 점수


for how in ("마지막 칸(-1)", "실제 마지막 칸", "전부 평균", "mask 평균"):  # 4가지
    print(f"{how:<10} → 출력 {tuple(ReadNet(how)(X, lens).shape)}")        # 3-3의 세 줄로 모양 확인

마지막 칸(-1)  → 출력 (3, 2)
실제 마지막 칸   → 출력 (3, 2)
전부 평균      → 출력 (3, 2)
mask 평균    → 출력 (3, 2)


**결과** · 예측 정답은 `(3, 2)` — 줄마다 점수 2개

- `마지막 칸(-1) → 출력 (3, 2)` — 틀의 끝을 읽음
- `실제 마지막 칸 → 출력 (3, 2)` — 줄마다 lens - 1
- `mask 평균 → 출력 (3, 2)` — 진짜 칸만 평균 · 모양은 넷 다 같음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>out[:, -1, :]</code></td>
<td style="text-align: left;">틀의 끝</td>
<td style="text-align: left;">빈 칸이 없을 때만</td>
</tr>
<tr>
<td style="text-align: left;"><code>out[arange(N), lens - 1]</code></td>
<td style="text-align: left;">줄마다 진짜 끝</td>
<td style="text-align: left;">패딩 배치 분류</td>
</tr>
<tr>
<td style="text-align: left;"><code>out.mean(1)</code></td>
<td style="text-align: left;">빈 칸까지 평균</td>
<td style="text-align: left;">쓰지 않음</td>
</tr>
<tr>
<td style="text-align: left;">mask 평균</td>
<td style="text-align: left;">진짜 칸만 평균</td>
<td style="text-align: left;">평균 요약이 필요할 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `lens`가 GPU 텐서면 `torch.arange(..., device=x.device)`로 같은 장치에서 만들어야 함

> ✅ **한 줄 정리** · 꺼내는 법만 다른 네 모델은 출력 모양이 모두 같아서, 차이는 학습해 봐야 드러난다.

**딥러닝 코드에서는**

- `h = out[torch.arange(len(x)), lens - 1]` — 패딩 배치에서 마지막 요약 꺼내기의 기본형

### 4-2. `A` 빈 칸을 읽으면 · 길이 5~60 · `compare_reads()` · `clip_grad_norm_()` <sub>원본 셀 42 뒷부분</sub>

> 🎯 **길이 5~60 줄을 채워 넣고 요약을 네 가지로 꺼내면 각각 몇 %인가?**

💡 **쉽게 말하면**

- 같은 데이터 · 같은 RNN · **꺼내는 법만** 바꿈.
- 틀의 끝(-1)은 대부분 줄에서 빈 칸을 수십 칸 지난 값.

<img src="images/rn_reads.png" alt="꺼내는 법 4가지 · 길이 5~60 vs 4~12" width="911">

**실행 흐름**

1. 학습 3000 · 평가 1000줄 · 가장 긴 길이로 채움
2. 방법마다 같은 seed로 25 epoch 학습
3. 평가 정확도

**먼저 예측** · 마지막 칸(-1)으로 읽으면 몇 %일까? — 답은 코드 실행 결과 아래에

In [17]:
def compare_reads(lo, hi):  # 같은 데이터 · 같은 모델 · 꺼내는 법만 4가지
    xa, ya = make_ragged(3000, lo, hi, seed=11)                          # 학습 3000줄
    xb, yb = make_ragged(1000, lo, hi, seed=12)                          # 평가 1000줄
    T = max(len(s) for s in xa + xb)                                     # 가장 긴 길이 = 틀
    Xa, Xb = pad(xa, T).to(DEV), pad(xb, T).to(DEV)                      # 채워서 장치로
    La = torch.tensor([len(s) for s in xa], device=DEV)                  # 학습 길이표
    Lb = torch.tensor([len(s) for s in xb], device=DEV)                  # 평가 길이표
    Ya, Yb = torch.tensor(ya, device=DEV), torch.tensor(yb, device=DEV)  # 정답
    for how in ("마지막 칸(-1)", "실제 마지막 칸", "전부 평균", "mask 평균"):  # 4가지
        random.seed(SEED)                                                # 방법마다 같은 시작점
        torch.manual_seed(SEED)                                          # 모델 초기값 고정
        model = ReadNet(how).to(DEV)                                     # 새 모델
        opt = torch.optim.Adam(model.parameters(), lr=3e-3)              # Adam
        lf = nn.CrossEntropyLoss()                                       # 분류 손실
        for _ in range(25):                                              # 25 epoch
            p = torch.randperm(len(Xa))                                  # 순서 섞기
            for i in range(0, len(Xa), 64):                              # 64줄씩
                j = p[i:i + 64].to(DEV)                                  # 이번 배치
                opt.zero_grad()                                          # 지난 기울기 지우기
                lf(model(Xa[j], La[j]), Ya[j]).backward()                # 손실 → 기울기
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)  # 폭발 방지
                opt.step()                                               # 가중치 수정
        with torch.no_grad():                                            # 평가
            acc = (model(Xb, Lb).argmax(1) == Yb).float().mean().item()  # 정확도
        print(f"{how:<10} {acc:.3f}")                                    # 방법별 결과


print("길이 5~60")    # 편차가 큰 경우
compare_reads(5, 60)  # 4가지 비교

길이 5~60


마지막 칸(-1)  0.500


실제 마지막 칸   1.000


전부 평균      0.483


mask 평균    1.000


**결과** · 예측 정답은 찍기 — 0.500

- `마지막 칸(-1)  0.500` — 빈 칸을 지난 요약 → 찍기 (강의 51.4%)
- `실제 마지막 칸   1.000` — 줄마다 진짜 끝 → 만점
- `전부 평균      0.483` — 빈 칸까지 평균 → 찍기 (강의 0.483)
- `mask 평균    1.000` — 진짜 칸만 평균 → 만점

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">-1 · 전부 평균</td>
<td style="text-align: left;">빈 칸 요약이 섞임</td>
<td style="text-align: left;">쓰지 않음</td>
</tr>
<tr>
<td style="text-align: left;">lens - 1 · mask 평균</td>
<td style="text-align: left;">진짜 칸만</td>
<td style="text-align: left;">패딩 배치</td>
</tr>
<tr>
<td style="text-align: left;"><code>pack_padded_sequence</code></td>
<td style="text-align: left;">빈 칸을 아예 안 지남</td>
<td style="text-align: left;">실무 표준 (RNN-3)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 빈 칸 요약에는 답이 없고 몇 칸 더 지났는지도 줄마다 달라 모델이 일관되게 틀릴 수도 없음

> ✅ **한 줄 정리** · 빈 칸까지 읽는 -1 · 전부 평균은 찍기 수준이고, 진짜 끝(lens - 1)이나 mask 평균으로 읽어야 1.000이 된다.

**딥러닝 코드에서는**

- 패딩 배치 분류 모델은 학습 전에 '읽는 자리'부터 확인

### 4-3. `A` -1 과 lens - 1 · 길이 4~12 · `output[idx, lens - 1]` <sub>원본 셀 43 ~ 46</sub>

> 🎯 **세 줄에서 -1과 lens - 1은 어디가 다르고, 길이 편차를 줄이면 왜 차이가 사라지는가?**

💡 **쉽게 말하면**

- -1은 **틀의 끝**이라 가장 긴 줄만 맞음.
- 길이 4~12면 빈 칸이 많아야 8칸 → 버그가 숨음.

**실행 흐름**

1. 세 줄 X에서 두 자리 비교
2. mask 첫 줄
3. 길이 4~12로 같은 비교

**먼저 예측** · 길이 4~12에서 -1로 읽으면? — 답은 코드 실행 결과 아래에

In [18]:
# -1 과 lens - 1 (원본 셀 44): 세 줄 X (3, 6, 10)에서 같은 자리인가
torch.manual_seed(SEED)                                             # 같은 시작점
output, _ = nn.RNN(10, 32, batch_first=True)(X)                     # 3-2에서 채운 세 줄
idx = torch.arange(len(X))                                          # 줄 번호 0 · 1 · 2
print("줄마다 output[:, -1] 과 output[idx, lens - 1] 이 같은가 :",  # 길이 6 줄만 True
      [torch.equal(a, b) for a, b in zip(output[:, -1, :], output[idx, lens - 1])])
m = (torch.arange(X.shape[1])[None, :] < lens[:, None]).int()       # 진짜 칸 = 1
print("mask 첫 줄 :", m[0].tolist())                                # 1 1 1 0 0 0


# 길이 4~12 로 좁히면 (원본 셀 46)
print("길이 4~12")    # 편차가 작은 경우
compare_reads(4, 12)  # 4가지 비교

줄마다 output[:, -1] 과 output[idx, lens - 1] 이 같은가 : [False, True, False]
mask 첫 줄 : [1, 1, 1, 0, 0, 0]
길이 4~12


마지막 칸(-1)  1.000


실제 마지막 칸   1.000


전부 평균      0.999


mask 평균    1.000


**결과** · 예측 정답은 1.000 — 차이가 안 남

- `줄마다 output[:, -1] 과 output[idx, lens - 1] 이 같은가 : [False, True, False]` — 길이 6 줄만 같음
- `mask 첫 줄 : [1, 1, 1, 0, 0, 0]` — 진짜 칸만 1
- `마지막 칸(-1)  1.000` — 4~12에서는 빈 칸이 적어 통과
- `전부 평균      0.999` — 빈 칸이 적으면 평균도 거의 통과 (강의 1.000)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">길이 5~60</td>
<td style="text-align: left;">빈 칸 최대 55칸</td>
<td style="text-align: left;">버그가 드러남</td>
</tr>
<tr>
<td style="text-align: left;">길이 4~12</td>
<td style="text-align: left;">빈 칸 최대 8칸</td>
<td style="text-align: left;">버그가 숨음</td>
</tr>
<tr>
<td style="text-align: left;"><code>pack_padded_sequence</code></td>
<td style="text-align: left;">빈 칸을 안 지남</td>
<td style="text-align: left;">실무 표준 (RNN-3)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 짧은 길이로만 테스트하면 통과 → 길이 편차가 큰 데이터로 꼭 확인

> ✅ **한 줄 정리** · -1은 가장 긴 줄에서만 진짜 끝이고, 길이 편차가 작으면 버그가 숨으므로 편차가 큰 데이터로 확인한다.

**딥러닝 코드에서는**

- 배치를 길이별로 묶어도(bucketing) 진짜 끝은 lens - 1로 읽음

---
## 5. 거리를 늘려 무너뜨리기 (L2)

> **이 장의 흐름**
>
> - 원본 L2 · 거리를 늘려 절벽을 찾고 epoch · lr로 회복
> - 앞 → 뒤 차이를 학습 전 · 후 모델로 재고 손으로 심은 가중치로 가능성 확인

<img src="images/rl_whisper.png" alt="맨 앞 두 칸만 바꾼 쌍 · 칸마다 남은 차이 (강의 숫자)" width="881">

**파이프라인 위치** · ⑤ 학습 — 먼 의존 진단

### 5-1. `A` 거리를 늘려 무너뜨리기 · `make_front_task()` <sub>원본 셀 47 · 48 · L2 (미리 돌린 결과)</sub>

> 🎯 **3 · 7을 맨 앞에 두고 끝까지의 거리를 늘리면 같은 20 epoch로 어디서 무너지는가?**

💡 **쉽게 말하면**

- 답은 첫 두 칸에서 정해짐 → **끝까지 들고 가야** 맞힘.
- 거리 = 길이 - 2.

<img src="images/rn_distance.png" alt="거리별 정확도 · 20 epoch vs 60 epoch" width="862">

**실행 흐름**

1. 거리 8 ~ 398 데이터
2. 같은 RNN · 20 epoch 학습
3. 거리별 정확도

**먼저 예측** · 거리 98은 맞힐까? — 답은 코드 실행 결과 아래에

In [19]:
# 3 · 7 을 맨 앞에 두고 끝까지의 거리를 늘리기 · 같은 20 epoch (원본 셀 48)
for dist in (8, 48, 78, 98, 198, 398):                        # 거리 = 길이 - 2
    L = dist + 2                                              # 줄 길이
    xa, ya = make_front_task(4000, L, seed=1)                 # 학습 4000줄
    xb, yb = make_front_task(1000, L, seed=2)                 # 평가 1000줄
    torch.manual_seed(SEED)                                   # 같은 시작점
    acc = train(Net("RNN"), to_tensor(xa), torch.tensor(ya),  # 같은 모델 · 같은 예산
                to_tensor(xb), torch.tensor(yb), epochs=20)
    print(f"거리 {dist:>3} → {acc:.3f}")                      # 거리별 정확도

거리   8 → 1.000


거리  48 → 1.000


거리  78 → 1.000


거리  98 → 0.517


거리 198 → 0.501


거리 398 → 0.484


**결과** · 예측 정답은 못 맞힘 — 0.517

- `거리  78 → 1.000` — 78까지는 하나도 안 틀림
- `거리  98 → 0.517` — 98부터 찍기 (강의 51.7%)
- `거리 398 → 0.484` — 찍기 근처 (강의 48.3%)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">거리 ≤ 78</td>
<td style="text-align: left;">20 epoch로 학습됨</td>
<td style="text-align: left;">짧은 의존</td>
</tr>
<tr>
<td style="text-align: left;">거리 ≥ 98</td>
<td style="text-align: left;">20 epoch로 안 됨</td>
<td style="text-align: left;">먼 의존 → 5-2 · 6장</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 78과 98 사이의 절벽 · 중간이 거의 없음 → 정확도만으로는 원인을 모름

> ✅ **한 줄 정리** · 같은 20 epoch에서 RNN은 거리 78까지 1.000이지만 98부터 찍기 수준으로 무너진다.

**딥러닝 코드에서는**

- 긴 시퀀스 모델은 '거리별 정확도'로 어디까지 닿는지 먼저 잼

### 5-2. `A` 학습을 세 배 · 학습률을 키우면 · `epochs=60` · `lr=1e-2` <sub>원본 셀 49 · 50 (미리 돌린 결과)</sub>

> 🎯 **60 epoch나 학습률 1e-2로 바꾸면 무너진 거리를 다시 맞히는가?**

💡 **쉽게 말하면**

- 거리가 멀어서 **못 하는** 것인지, **덜 배운** 것인지 가르는 실험.

**실행 흐름**

1. 거리 98 · 198 · 398을 60 epoch
2. 거리 98을 20 epoch · lr 1e-2

**먼저 예측** · 60 epoch면 거리 98은? — 답은 코드 실행 결과 아래에

In [20]:
# 학습을 세 배 더 시키면 · 60 epoch (원본 셀 50)
for dist in (98, 198, 398):                                   # 20 epoch에서 무너진 거리
    xa, ya = make_front_task(4000, dist + 2, seed=1)          # 학습
    xb, yb = make_front_task(1000, dist + 2, seed=2)          # 평가
    torch.manual_seed(SEED)                                   # 같은 시작점
    acc = train(Net("RNN"), to_tensor(xa), torch.tensor(ya),  # epoch만 60
                to_tensor(xb), torch.tensor(yb), epochs=60)
    print(f"60 epoch · 거리 {dist:>3} → {acc:.3f}")           # 결과


# 횟수 대신 학습률을 키우면 (20 epoch · lr 1e-2)
xa, ya = make_front_task(4000, 100, seed=1)               # 거리 98
xb, yb = make_front_task(1000, 100, seed=2)               # 평가
torch.manual_seed(SEED)                                   # 같은 시작점
acc = train(Net("RNN"), to_tensor(xa), torch.tensor(ya),  # lr만 바꿈
            to_tensor(xb), torch.tensor(yb), epochs=20, lr=1e-2)
print(f"20 epoch · 학습률 1e-2 · 거리  98 → {acc:.3f}")   # 결과

60 epoch · 거리  98 → 1.000


60 epoch · 거리 198 → 0.499


60 epoch · 거리 398 → 0.473


20 epoch · 학습률 1e-2 · 거리  98 → 1.000


**결과** · 예측 정답은 1.000 — 덜 배운 것

- `60 epoch · 거리  98 → 1.000` — 세 배 학습으로 회복 (강의도 만점)
- `60 epoch · 거리 198 → 0.499` — 이 PC에서는 198부터 여전히 실패 (강의는 만점)
- `20 epoch · 학습률 1e-2 · 거리  98 → 1.000` — 횟수 대신 보폭을 키워도 회복

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">epoch 늘리기</td>
<td style="text-align: left;">같은 보폭으로 더 오래</td>
<td style="text-align: left;">시간 여유가 있을 때</td>
</tr>
<tr>
<td style="text-align: left;">학습률 키우기</td>
<td style="text-align: left;">보폭을 크게</td>
<td style="text-align: left;">빨리 · 단 불안정할 수 있음</td>
</tr>
<tr>
<td style="text-align: left;">구조 바꾸기 (7장)</td>
<td style="text-align: left;">신호가 오는 길을 냄</td>
<td style="text-align: left;">먼 거리가 핵심일 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 강의(CPU)는 60 epoch에서 198 · 398도 100% · 이 PC(GPU)는 98만 회복 → 환경에 따라 절벽 위치가 다름

> ✅ **한 줄 정리** · 거리 98의 실패는 60 epoch나 lr 1e-2로 회복되므로 기억이 못 닿는 게 아니라 배우기가 느린 것이다.

**딥러닝 코드에서는**

- 학습이 멈춘 듯하면 epoch · lr을 바꿔 '덜 배운 것'인지 먼저 확인

### 5-3. `B` 앞 → 뒤 · 학습 전 모델 · `Decimal` · `finfo.eps` <sub>원본 셀 51 · 52</sub>

> 🎯 **맨 앞 두 칸만 바꾼 두 줄의 요약 차이는 학습 전 모델에서 몇 칸까지 남는가?**

💡 **쉽게 말하면**

- 귓속말처럼 한 칸 갈 때마다 차이가 **절반 남짓**만 남음.
- float32 바닥보다 작으면 잴 수 없음.

**실행 흐름**

1. 맨 앞만 맞바꾼 512쌍
2. 칸마다 상대 차이 (소수로 표시)
3. float32 바닥

**먼저 예측** · 20칸 뒤 차이는? — 답은 코드 실행 결과 아래에

In [21]:
# 앞 → 뒤: 맨 앞 두 칸만 맞바꾼 두 줄의 요약 차이가 칸마다 얼마나 남나 (원본 셀 52)
def plain(v):                                # 0.0000077 처럼 소수로 적기
    return format(Decimal(f"{v:.3g}"), "f")  # 유효숫자 3자리


def whisper(model, L=100, n=512, cols=(1, 2, 5, 10, 20, 40, 70, 99)):  # 칸별 상대 차이
    xs, _ = make_front_task(n, L, seed=31)                             # 512줄
    xs_b = [[s[1], s[0]] + s[2:] for s in xs]                          # 맨 앞 두 칸만 맞바꿈
    r = model.r.cpu()                                                  # 비교는 CPU에서
    with torch.no_grad():                                              # 측정만
        oa, _ = r(to_tensor(xs))                                       # 원래 줄 요약
        ob, _ = r(to_tensor(xs_b))                                     # 바꾼 줄 요약
    # 차이 ÷ 크기
    diff = {t: ((oa[:, t] - ob[:, t]).norm(dim=1) / oa[:, t].norm(dim=1)).mean().item()
            for t in cols}
    model.to(DEV)                                                      # 장치 원래대로
    return diff                                                        # {칸: 평균 상대 차이}


torch.manual_seed(SEED)                  # 같은 시작점
untrained = Net("RNN")                   # 학습 전 모델
for t, v in whisper(untrained).items():  # 칸마다
    print(f"{t:>2}칸  {plain(v)}")       # 상대 차이
# 측정의 바닥
print("float32 가 구분하는 가장 작은 상대 차이 :", plain(torch.finfo(torch.float32).eps))

 1칸  0.736
 2칸  0.378
 5칸  0.061
10칸  0.00255
20칸  0.00000767
40칸  0.0000000782
70칸  0.000000079
99칸  0.0000000794
float32 가 구분하는 가장 작은 상대 차이 : 0.000000119


**결과** · 예측 정답은 0.00000767 — 거의 사라짐

- ` 1칸  0.736` — 강의와 같은 값 (CPU 측정)
- `20칸  0.00000767` — 스무 칸에서 거의 사라짐
- `99칸  0.0000000794` — 바닥 근처
- `float32 가 구분하는 가장 작은 상대 차이 : 0.000000119` — 이보다 작으면 측정 불가

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">상대 차이 = 차이 ÷ 크기</td>
<td style="text-align: left;">크기와 상관없이 비교</td>
<td style="text-align: left;">칸마다 비교</td>
</tr>
<tr>
<td style="text-align: left;"><code>plain()</code></td>
<td style="text-align: left;">지수 대신 소수로</td>
<td style="text-align: left;">작은 수의 크기 감 잡기</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 측정은 CPU로 (GPU 모델은 잠시 CPU로 옮겼다 돌려놓음)

> ✅ **한 줄 정리** · 학습 전 RNN은 맨 앞 차이가 칸마다 줄어 20 ~ 40칸에서 float32 바닥에 닿는다.

**딥러닝 코드에서는**

- 입력 앞부분을 바꿔 출력이 바뀌는지 보는 것은 '모델이 앞을 기억하나' 점검법

### 5-4. `B` 앞 → 뒤 · 학습한 모델 · `train_front()` · `trained` <sub>원본 셀 53 ~ 56 (미리 돌린 결과)</sub>

> 🎯 **거리 98에서 20 epoch · 60 epoch로 학습한 모델은 앞 차이를 끝까지 나르는가?**

💡 **쉽게 말하면**

- 실패한 모델(20 epoch)과 성공한 모델(60 epoch)을 **같은 자**로 잼.

**실행 흐름**

1. 거리 98 · 20 epoch 학습 → 차이 측정
2. 60 epoch 학습 → 차이 측정
3. 두 모델은 6-3에서 재사용

**먼저 예측** · 60 epoch 모델의 99칸 차이는? — 답은 코드 실행 결과 아래에

In [22]:
# 같은 측정을 학습시킨 모델로 · 거리 98 (원본 셀 54 · 56)
def train_front(dist, epochs):                           # 거리 · epoch → 학습한 모델 · 정확도
    xa, ya = make_front_task(4000, dist + 2, seed=1)     # 학습
    xb, yb = make_front_task(1000, dist + 2, seed=2)     # 평가
    torch.manual_seed(SEED)                              # 같은 시작점
    model = Net("RNN")                                   # 새 모델
    acc = train(model, to_tensor(xa), torch.tensor(ya),  # 같은 학습 함수
                to_tensor(xb), torch.tensor(yb), epochs=epochs)
    return model, acc                                    # 모델 · 정확도


trained = {}                                              # epoch별 모델 (6-3에서 다시 씀)
for ep in (20, 60):                                       # 20 · 60 epoch
    trained[ep], acc = train_front(98, ep)                # 거리 98 학습
    print(f"{ep} epoch 모델 · 거리 98 정확도 {acc:.3f}")  # 정확도
    # 칸별 차이
    print("   " + " · ".join(f"{t}칸 {plain(v)}" for t, v in whisper(trained[ep]).items()))

20 epoch 모델 · 거리 98 정확도 0.517
   1칸 1.72 · 2칸 1.42 · 5칸 0.971 · 10칸 0.228 · 20칸 0.0139 · 40칸 0.000151 · 70칸 0.000000572 · 99칸 0.00000013


60 epoch 모델 · 거리 98 정확도 1.000
   1칸 1.91 · 2칸 1.99 · 5칸 2 · 10칸 2 · 20칸 2 · 40칸 2 · 70칸 2 · 99칸 2


**결과** · 예측 정답은 2 — 끝까지 그대로

- `20 epoch 모델 · 거리 98 정확도 0.517` — 실패한 모델
- `1칸 1.72 · 2칸 1.42 · 5칸 0.971 · 10칸 0.228 · 20칸 0.0139 · 40칸 0.000151 · 70칸 0.000000572 · 99칸 0.00000013` — 학습 전보다는 오래 남지만 99칸에서 바닥 (강의 0.000000143)
- `60 epoch 모델 · 거리 98 정확도 1.000` — 성공한 모델
- `1칸 1.91 · 2칸 1.99 · 5칸 2 · 10칸 2 · 20칸 2 · 40칸 2 · 70칸 2 · 99칸 2` — 99칸까지 차이가 그대로 (2 = 두 요약이 정반대 방향)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">학습 전</td>
<td style="text-align: left;">40칸 근처에서 사라짐</td>
<td style="text-align: left;">출발점</td>
</tr>
<tr>
<td style="text-align: left;">20 epoch</td>
<td style="text-align: left;">더 오래 남지만 끝에선 사라짐</td>
<td style="text-align: left;">덜 배움</td>
</tr>
<tr>
<td style="text-align: left;">60 epoch</td>
<td style="text-align: left;">끝까지 그대로</td>
<td style="text-align: left;">기억이 닿음</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 강의 숫자(1.73 · 1.68 등)와 끝자리가 다름 · GPU 학습

> ✅ **한 줄 정리** · 기억은 닿을 수 있고, 닿게 만드는 것이 학습이며, 60 epoch 모델은 99칸 뒤까지 맨 앞 차이를 그대로 나른다.

**딥러닝 코드에서는**

- 학습 후 모델이 먼 입력을 쓰는지는 '앞만 바꾼 쌍' 테스트로 확인 가능

### 5-5. `B` 학습 없이 손으로 심은 가중치 · `torch.eye()` <sub>원본 셀 57 · 58</sub>

> 🎯 **요약을 그대로 넘기는 가중치를 직접 심으면 거리 398도 닿는가?**

💡 **쉽게 말하면**

- weight_hh = 단위 행렬 · 3이면 +0.5 · 7이면 -0.5를 첫 칸에.

**실행 흐름**

1. 가중치 직접 적기
2. 거리 8 · 98 · 398
3. 부호로 판정

**먼저 예측** · 거리 398에서 정확도는? — 답은 코드 실행 결과 아래에

In [23]:
# 학습 없이 손으로 심은 가중치 (원본 셀 58): 요약을 그대로 넘기는 RNNCell
def handmade(L, scale=0.5):                                       # 길이 L → 정확도 · 끝에 남은 크기
    torch.manual_seed(SEED)                                       # 같은 시작점
    cell = nn.RNNCell(10, 8)                                      # 요약 8칸
    with torch.no_grad():                                         # 가중치를 직접 적음
        cell.weight_ih.zero_()                                    # 입력 가중치 0
        cell.bias_ih.zero_()                                      # bias 0
        cell.bias_hh.zero_()                                      # bias 0
        cell.weight_hh.copy_(torch.eye(8))                        # 이전 요약을 그대로 넘김
        cell.weight_ih[0, 3] = scale                              # 3이면 첫 칸에 +0.5
        cell.weight_ih[0, 7] = -scale                             # 7이면 첫 칸에 -0.5
    xs, ys = make_front_task(1000, L, seed=2)                     # 평가 1000줄
    X, Y = to_tensor(xs), torch.tensor(ys)                        # 텐서
    h = torch.zeros(1000, 8)                                      # 처음 요약 0
    with torch.no_grad():                                         # 측정만
        for t in range(L):                                        # L칸
            h = cell(X[:, t, :], h)                               # 같은 상자 반복
    acc = max(((h[:, 0] > 0).long() == Y).float().mean().item(),  # 부호로 판정 (양쪽 중 큰 쪽)
              ((h[:, 0] < 0).long() == Y).float().mean().item())
    return acc, h[:, 0].abs().mean().item()                       # 정확도 · 첫 칸 크기


for L in (10, 100, 400):                                               # 거리 8 · 98 · 398
    acc, size = handmade(L)                                            # 손 가중치로
    print(f"거리 {L - 2:>3} → {acc:.3f} · 끝에 남은 크기 {size:.4f}")  # 결과

거리   8 → 1.000 · 끝에 남은 크기 0.0377
거리  98 → 1.000 · 끝에 남은 크기 0.0362


거리 398 → 1.000 · 끝에 남은 크기 0.0322


**결과** · 예측 정답은 1.000

- `거리   8 → 1.000 · 끝에 남은 크기 0.0377` — 강의와 같은 값
- `거리 398 → 1.000 · 끝에 남은 크기 0.0322` — 398칸을 지나도 남음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">손으로 심음</td>
<td style="text-align: left;">가능성 증명</td>
<td style="text-align: left;">구조 이해</td>
</tr>
<tr>
<td style="text-align: left;">학습</td>
<td style="text-align: left;">같은 해를 찾아야 함</td>
<td style="text-align: left;">실제</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - tanh 때문에 크기가 조금씩 줄지만(0.0377 → 0.0322) 부호는 그대로

> ✅ **한 줄 정리** · 요약을 그대로 넘기는 가중치를 심으면 398칸도 닿으므로 RNN 구조 자체는 먼 기억을 나를 수 있다.

**딥러닝 코드에서는**

- '그대로 넘기는 길'을 구조에 넣은 것이 LSTM 메모 · 잔차 연결

---
## 6. 기울기가 어디까지 오는가 (L3)

> **이 장의 흐름**
>
> - 원본 L3 · 손으로 펼친 RNNCell로 칸별 기울기
> - float64 · 학습한 모델 · output.grad 함정 · 가중치 행렬

<img src="images/rl_grad.png" alt="거슬러 간 칸별 기울기 · RNN · LSTM 기본 · 열어둠 · 활짝" width="941">

**파이프라인 위치** · ⑤ 학습 — 기울기 소실 측정

### 6-1. `A` 기울기가 어디까지 오는가 · `retain_grad()` · `show()` <sub>원본 셀 59 · 60 · L3</sub>

> 🎯 **마지막 손실의 기울기는 몇 칸 거슬러 가면 사라지는가?**

💡 **쉽게 말하면**

- 정확도로는 기억 탓인지 학습 탓인지 못 가림 → **기울기를 직접 찍음**.
- nn.RNN은 안이 안 보여 손으로 펼친 `RNNCell` + `retain_grad`.

<img src="images/rn_grad.png" alt="칸별 기울기 · 학습 전 · 20 epoch · 60 epoch" width="918">

**실행 흐름**

1. grad_profile 정의 (학습한 모델 · forget bias · float64 옵션)
2. 학습 전 RNN 측정
3. 배율 계산

**먼저 예측** · 20칸 거슬러 가면 몇 배 작아질까? — 답은 코드 실행 결과 아래에

In [24]:
# 뒤 → 앞: 마지막 손실의 기울기가 k칸 거슬러 간 요약에 얼마나 오나 (원본 셀 60 · CPU)
STEPS = (0, 10, 20, 40, 70, 99)  # 거슬러 간 칸 수


def grad_profile(kind, T=100, hid=32, batch=64,           # 칸별 기울기 크기
                 forget_bias=None, dtype=torch.float32, steps=STEPS, model=None):
    torch.manual_seed(SEED)                               # 같은 시작점
    cell = {"RNN": nn.RNNCell, "LSTM": nn.LSTMCell}[kind](10, hid).to(dtype)  # 손으로 펼칠 상자
    head = nn.Linear(hid, 2).to(dtype)                    # 끝 분류기
    if model is not None:                                 # 학습한 Net의 가중치를 옮김 (6-3)
        with torch.no_grad():                             # 복사만
            for a, b in [("weight_ih", "weight_ih_l0"), ("weight_hh", "weight_hh_l0"),  # 이름 짝
                         ("bias_ih", "bias_ih_l0"), ("bias_hh", "bias_hh_l0")]:
                getattr(cell, a).copy_(getattr(model.r, b).cpu())  # RNN 가중치
            head.weight.copy_(model.o.weight.cpu())       # 분류기 가중치
            head.bias.copy_(model.o.bias.cpu())           # 분류기 bias
    if forget_bias is not None:  # bias 순서 [input | forget | cell | output] (7장)
        with torch.no_grad():                             # 값만 넣음
            cell.bias_ih[hid:2 * hid].fill_(forget_bias)  # forget 구간
            cell.bias_hh[hid:2 * hid].fill_(0.0)          # 두 bias가 더해지므로 한쪽은 0
    xs, ys = make_front_task(batch, T, seed=7)            # 64줄 · 길이 T
    X, Y = to_tensor(xs).to(dtype), torch.tensor(ys)      # 자료형 맞춤
    h = torch.zeros(batch, hid, dtype=dtype)              # 처음 요약
    c = torch.zeros(batch, hid, dtype=dtype)              # LSTM 메모
    hs = []                                               # 칸마다 요약
    for t in range(T):                                    # T칸 차례로
        if kind == "RNN":                                 # RNN은 h만
            h = cell(X[:, t, :], h)                       # 새 요약
        else:                                             # LSTM은 h · c 둘
            h, c = cell(X[:, t, :], (h, c))               # 새 요약 · 새 메모
        h.retain_grad()                                   # 중간 요약의 기울기를 남겨 달라고 표시
        hs.append(h)                                      # 저장
    nn.CrossEntropyLoss()(head(hs[-1]), Y).backward()     # 끝에서 손실 → 역전파
    return [hs[T - 1 - b].grad.norm().item() for b in steps]  # b칸 거슬러 간 기울기 크기


def show(name, values, fmt=".2e"):                            # 한글 2칸 폭으로 줄 맞춤
    width = sum(2 if ord(ch) > 0x1100 else 1 for ch in name)  # 화면 폭
    print(name + " " * (14 - width) + "".join(f"{v:>10}" if isinstance(v, int)  # 정수는 그대로
                                              else f"{v:>10{fmt}}" for v in values))


show("거슬러 간 칸", STEPS)                                          # 머리 줄
rnn_prof = grad_profile("RNN")                                       # 학습 전 RNN
show("RNN", rnn_prof)                                                # 칸별 크기
ratios = [f"{rnn_prof[0] / rnn_prof[k]:,.0f}배" for k in (1, 2, 3)]  # 0칸 ÷ k칸
print("배율 10 · 20 · 40스텝 :", " · ".join(ratios))                 # 줄어든 배율

거슬러 간 칸           0        10        20        40        70        99
RNN             5.16e-02  1.06e-04  2.94e-07  1.33e-12  1.50e-20  0.00e+00
배율 10 · 20 · 40스텝 : 486배 · 175,503배 · 38,772,643,119배


**결과** · 예측 정답은 약 17만 배

- `RNN             5.16e-02  1.06e-04  2.94e-07  1.33e-12  1.50e-20  0.00e+00` — 강의와 같은 값
- `배율 10 · 20 · 40스텝 : 486배 · 175,503배 · 38,772,643,119배` — 강의 화면과 같은 배율 (17만 배 · 수백억 배)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>h.retain_grad()</code></td>
<td style="text-align: left;">중간 요약 기울기 보존</td>
<td style="text-align: left;">칸별 측정</td>
</tr>
<tr>
<td style="text-align: left;"><code>show()</code></td>
<td style="text-align: left;">한글 폭 맞춤 표 출력</td>
<td style="text-align: left;">결과 비교</td>
</tr>
<tr>
<td style="text-align: left;"><code>model=</code> 인자</td>
<td style="text-align: left;">학습한 가중치를 옮겨 측정</td>
<td style="text-align: left;">6-3</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `grad_profile`은 CPU에서 측정 · 학습한 GPU 모델 가중치는 `.cpu()`로 옮겨 복사

> ✅ **한 줄 정리** · 학습 전 RNN의 기울기는 10칸에 486배 · 20칸에 약 17만 배 작아져 99칸에서는 0으로 찍힌다.

**딥러닝 코드에서는**

- 긴 시퀀스 학습 진단 첫 단계 · 칸별 기울기 크기를 로그 눈금으로 그려 봄

### 6-2. `B` 0이 진짜 0일까 · `torch.float64` <sub>원본 셀 61 · 62</sub>

> 🎯 **99칸의 0.00e+00은 진짜 0인가?**

💡 **쉽게 말하면**

- 같은 계산을 **float64**로 나란히 놓음.

**실행 흐름**

1. float32 · float64 측정
2. float32가 처음 0이 되는 칸

**먼저 예측** · float64로 재면 99칸은? — 답은 코드 실행 결과 아래에

In [25]:
# 0 이 진짜 0 일까 (원본 셀 62): 같은 계산을 float64로
show("거슬러 간 칸", STEPS)                                    # 머리 줄
show("RNN float32", grad_profile("RNN"))                       # 99칸이 0
show("RNN float64", grad_profile("RNN", dtype=torch.float64))  # 0이 아니라 아주 작은 수
every = grad_profile("RNN", steps=range(100))                  # 0 ~ 99칸 전부
print("float32 에서 처음 0 으로 찍히는 자리 :",                # 적을 수 있는 한계
      next(b for b in range(100) if every[b] == 0.0), "스텝 거슬러 간 곳")

거슬러 간 칸           0        10        20        40        70        99
RNN float32     5.16e-02  1.06e-04  2.94e-07  1.33e-12  1.50e-20  0.00e+00
RNN float64     5.16e-02  1.06e-04  2.94e-07  1.33e-12  1.50e-20  3.38e-28


float32 에서 처음 0 으로 찍히는 자리 : 77 스텝 거슬러 간 곳


**결과** · 예측 정답은 3.38e-28 — 0이 아님

- `RNN float64     5.16e-02  1.06e-04  2.94e-07  1.33e-12  1.50e-20  3.38e-28` — 0이 아니라 float32가 못 적는 값
- `float32 에서 처음 0 으로 찍히는 자리 : 77 스텝 거슬러 간 곳` — 77칸부터 float32로 0

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">float32</td>
<td style="text-align: left;">학습 기본 · 작은 값은 0</td>
<td style="text-align: left;">학습</td>
</tr>
<tr>
<td style="text-align: left;">float64</td>
<td style="text-align: left;">더 작은 값까지</td>
<td style="text-align: left;">측정 · 검증</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 학습에는 3.38e-28이나 0이나 같음

> ✅ **한 줄 정리** · 99칸의 0은 float32가 못 적는 3.38e-28이며, 학습에는 0과 다를 바 없다.

**딥러닝 코드에서는**

- NaN · 0이 의심되면 float64로 같은 계산을 해 보는 것이 빠른 확인법

### 6-3. `B` 학습시킨 모델의 기울기 · `model=trained[ep]` <sub>원본 셀 63 · 64 (미리 돌린 결과)</sub>

> 🎯 **5-4의 20 · 60 epoch 모델에서는 먼 칸까지 기울기가 오는가?**

💡 **쉽게 말하면**

- 학습한 가중치를 손으로 편 판에 옮겨 **같은 자**로 잼.

**실행 흐름**

1. 학습 전 · 20 · 60 epoch 측정
2. tanh 미분 평균으로 포화 확인

**먼저 예측** · 60 epoch 모델의 99칸 기울기는 학습 전보다 클까? — 답은 코드 실행 결과 아래에

In [26]:
# 학습시킨 모델의 기울기 (원본 셀 64): 5-4의 20 · 60 epoch 모델
show("거슬러 간 칸", STEPS)                                      # 머리 줄
show("학습 전", grad_profile("RNN"))                             # 출발점
for ep in (20, 60):                                              # 학습한 두 모델
    show(f"{ep} epoch", grad_profile("RNN", model=trained[ep]))  # 가중치를 옮겨 측정


# 기울기가 0이 된 이유 확인: 요약 값이 ±1에 붙으면(포화) tanh 미분 1 - h² 이 0 근처
xs7, _ = make_front_task(64, 100, seed=7)                                # 같은 데이터
torch.manual_seed(SEED)                                                  # 학습 전 모델 시작점
# 세 모델
for name, m in [("학습 전", Net("RNN")), ("20 epoch", trained[20]), ("60 epoch", trained[60])]:
    with torch.no_grad():                                                # 측정만
        o, _ = m.r.cpu()(to_tensor(xs7))                                 # 칸별 요약 (64, 100, 32)
    m.to(DEV)                                                            # 장치 원래대로
    print(f"{name:<8} tanh 미분 평균 {(1 - o ** 2).mean().item():.3f}")  # 작을수록 포화

거슬러 간 칸           0        10        20        40        70        99
학습 전         5.16e-02  1.06e-04  2.94e-07  1.33e-12  1.50e-20  0.00e+00


20 epoch        3.86e-02  1.17e-02  1.19e-03  3.22e-05  1.26e-07  7.10e-10


60 epoch        1.55e-05  4.38e-12  3.87e-18  0.00e+00  0.00e+00  0.00e+00
학습 전     tanh 미분 평균 0.963
20 epoch tanh 미분 평균 0.945
60 epoch tanh 미분 평균 0.145


**결과** · 예측 정답은 이 PC에서는 아님 — 40칸부터 0

- `20 epoch        3.86e-02  1.17e-02  1.19e-03  3.22e-05  1.26e-07  7.10e-10` — 학습 중에는 먼 칸까지 신호가 옴
- `60 epoch        1.55e-05  4.38e-12  3.87e-18  0.00e+00  0.00e+00  0.00e+00` — 다 배운 뒤에는 다시 사라짐 (강의는 99칸 1.45e-04)
- `60 epoch tanh 미분 평균 0.145` — 요약이 ±1에 붙음(포화) → 미분이 작아 신호가 끊김
- `학습 전     tanh 미분 평균 0.963` — 학습 전은 포화 없음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">학습 전</td>
<td style="text-align: left;">포화 없음 · 반복 곱으로 소실</td>
<td style="text-align: left;">출발점</td>
</tr>
<tr>
<td style="text-align: left;">20 epoch</td>
<td style="text-align: left;">먼 칸까지 신호 · 학습 진행 중</td>
<td style="text-align: left;">학습 중간</td>
</tr>
<tr>
<td style="text-align: left;">60 epoch (이 PC)</td>
<td style="text-align: left;">포화 · 손실도 작아 신호 작음</td>
<td style="text-align: left;">다 배운 뒤</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 강의(CPU)와 결과가 다름 · 강의는 60 epoch에서도 99칸 1.45e-04
> - 다 배운 모델은 손실 자체가 작아 0스텝 기울기도 작음 (1.55e-05)

> ✅ **한 줄 정리** · 학습 중인 모델은 먼 칸까지 신호가 오지만, 다 배워 요약이 포화된 모델은 tanh 미분이 0.145로 작아 기울기가 다시 끊긴다.

**딥러닝 코드에서는**

- 기울기 크기는 학습 단계마다 다름 → 한 시점 측정으로 결론 내지 않기

### 6-4. `B` output.grad 로 재면 · `output.retain_grad()` <sub>원본 셀 65 · 66</sub>

> 🎯 **`output.grad`로 칸별 기울기를 재면 왜 틀리는가?**

💡 **쉽게 말하면**

- output은 칸마다 값을 **보여 줄 뿐**, 다음 칸으로 가는 길 위의 h는 안에 감춰져 있음.

**실행 흐름**

1. nn.RNN · output.retain_grad
2. 끝 칸만 손실에 씀
3. 칸별 크기 · 0인 칸 세기

**먼저 예측** · 기울기가 정확히 0인 칸은 몇 칸일까? — 답은 코드 실행 결과 아래에

In [27]:
# output.grad 로 재면 (원본 셀 66): 쓰면 안 되는 지름길
torch.manual_seed(SEED)                                              # 같은 시작점
rnn = nn.RNN(10, 32, batch_first=True)                               # 감싼 판
head = nn.Linear(32, 2)                                              # 분류기
xs, ys = make_front_task(64, 100, seed=7)                            # 6장과 같은 데이터
output, h_n = rnn(to_tensor(xs))                                     # 칸마다 요약
output.retain_grad()                                                 # output의 기울기 남기기
nn.CrossEntropyLoss()(head(output[:, -1, :]), torch.tensor(ys)).backward()  # 끝 칸만 손실에 씀
g = output.grad                                                      # (64, 100, 32)
show("거슬러 간 칸", STEPS)                                          # 머리 줄
show("output.grad", [g[:, 99 - b, :].norm().item() for b in STEPS])  # 끝 칸만 값이 있음
print("기울기가 정확히 0 인 칸 :",                                   # 손실에 직접 쓰지 않은 칸
      sum(1 for t in range(100) if g[:, t, :].abs().max().item() == 0.0), "/ 100")

거슬러 간 칸           0        10        20        40        70        99
output.grad     5.16e-02  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00
기울기가 정확히 0 인 칸 : 99 / 100


**결과** · 예측 정답은 99칸

- `output.grad      5.16e-02  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00` — 끝 칸만 값이 있음
- `기울기가 정확히 0 인 칸 : 99 / 100` — 손실에 직접 안 쓴 칸은 0 → 소실 측정이 아님

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>output.grad</code></td>
<td style="text-align: left;">손실에 직접 쓴 칸만</td>
<td style="text-align: left;">쓰지 않음</td>
</tr>
<tr>
<td style="text-align: left;">cell + <code>h.retain_grad()</code></td>
<td style="text-align: left;">다음 칸으로 가는 길 위</td>
<td style="text-align: left;">칸별 측정 (6-1)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 이 0은 소실이 아니라 측정 위치가 틀린 것

> ✅ **한 줄 정리** · `output.grad`는 손실에 직접 쓴 칸에만 값이 있어 99칸이 0이므로, 칸별 기울기는 손으로 편 cell에서 재야 한다.

**딥러닝 코드에서는**

- 중간 기울기는 `torch.autograd.grad(loss, hs)`로도 꺼낼 수 있음

### 6-5. `B` 가중치 행렬을 재면 · `svdvals()` · `eigvals()` <sub>원본 셀 67 · 68</sub>

> 🎯 **기울기가 한 칸에 몇 배씩 줄어드는지를 가중치 행렬로 설명할 수 있는가?**

💡 **쉽게 말하면**

- 실제로 곱해지는 것은 숫자 하나가 아니라 **행렬 W_hh**와 **tanh 미분**.

**실행 흐름**

1. float64로 100칸 측정
2. 최대 특이값 · 고유값 크기 · tanh 미분
3. 실측 비율과 비교

**먼저 예측** · 고유값 크기 × tanh 미분은 실측 비율과 비슷할까? — 답은 코드 실행 결과 아래에

In [28]:
# 가중치 행렬을 재면 (원본 셀 68): 학습 전 · float64 · 6-1과 같은 모델 · 데이터
torch.manual_seed(SEED)                                   # 같은 시작점
cell = nn.RNNCell(10, 32).to(torch.float64)               # 상자
head = nn.Linear(32, 2).to(torch.float64)                 # 분류기
xs, ys = make_front_task(64, 100, seed=7)                 # 같은 데이터
X, Y = to_tensor(xs).to(torch.float64), torch.tensor(ys)  # float64
h = torch.zeros(64, 32, dtype=torch.float64)              # 처음 요약
hs = []                                                   # 칸마다 요약
for t in range(100):                                      # 100칸
    h = cell(X[:, t, :], h)                               # 새 요약
    h.retain_grad()                                       # 기울기 남기기
    hs.append(h)                                          # 저장
nn.CrossEntropyLoss()(head(hs[-1]), Y).backward()         # 역전파
norms = [hs[t].grad.norm().item() for t in range(100)]    # 칸별 기울기 크기

with torch.no_grad():                                                      # 측정만
    W = cell.weight_hh                                                     # 이전 요약에 곱하는 행렬
    radius = torch.linalg.eigvals(W).abs().max().item()                    # 고유값 크기의 최댓값
    tanh_d = (1 - torch.stack(hs) ** 2).mean().item()                      # tanh 미분 평균
    smax = torch.linalg.svdvals(W).max().item()                            # 최대 특이값
    print(f"한 번 곱할 때 최대로 늘이는 배율 (최대 특이값) : {smax:.2f}")  # 한 번 곱의 최대
    print(f"거듭 곱할 때 남는 배율 (고유값 크기의 최댓값)  : {radius:.2f}")  # 반복 곱의 배율
    print(f"tanh 미분 평균                               : {tanh_d:.2f}")  # 활성화가 깎는 몫
    print(f"두 값을 곱하면                                : {radius * tanh_d:.2f}")  # 예상 배율
per = (norms[99 - 70] / norms[99]) ** (1 / 70)                             # 70칸 비율의 70제곱근
print(f"실제 기울기에서 잰 한 칸당 비율 (0 → 70스텝)   : {per:.2f}")       # 실측

한 번 곱할 때 최대로 늘이는 배율 (최대 특이값) : 1.10
거듭 곱할 때 남는 배율 (고유값 크기의 최댓값)  : 0.57
tanh 미분 평균                               : 0.96
두 값을 곱하면                                : 0.55
실제 기울기에서 잰 한 칸당 비율 (0 → 70스텝)   : 0.54


**결과** · 예측 정답은 비슷함 — 0.55 vs 0.54

- `한 번 곱할 때 최대로 늘이는 배율 (최대 특이값) : 1.10` — 한 번은 늘 수도 있음
- `거듭 곱할 때 남는 배율 (고유값 크기의 최댓값)  : 0.57` — 여러 번 곱하면 0.57배씩
- `두 값을 곱하면                                : 0.55` — 0.57 × 0.96
- `실제 기울기에서 잰 한 칸당 비율 (0 → 70스텝)   : 0.54` — 실측과 거의 같음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">최대 특이값</td>
<td style="text-align: left;">한 번 곱의 최대 배율</td>
<td style="text-align: left;">폭발 가능성</td>
</tr>
<tr>
<td style="text-align: left;">고유값 크기 최댓값</td>
<td style="text-align: left;">거듭 곱의 배율</td>
<td style="text-align: left;">소실 · 폭발 판단</td>
</tr>
<tr>
<td style="text-align: left;">tanh 미분 평균</td>
<td style="text-align: left;">활성화가 깎는 몫</td>
<td style="text-align: left;">포화 확인</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 0.9를 50번 그림은 감 잡기용 · 답은 재서 냄

> ✅ **한 줄 정리** · 한 칸당 기울기 비율은 W_hh 고유값 크기(0.57) × tanh 미분(0.96) ≈ 0.55로 실측 0.54와 맞는다.

**딥러닝 코드에서는**

- 직교 초기화(`nn.init.orthogonal_`)는 고유값 크기를 1에 맞춰 소실 · 폭발을 늦춤

---
## 7. 문을 달면 살아날까 (L3b)

> **이 장의 흐름**
>
> - 원본 L3b · 같은 측정을 LSTMCell로
> - 문 값 기록 → 문을 열어 둔 시작

<img src="images/rl_lstm_cell.png" alt="LSTM 한 칸 · 메모 c는 곱하기와 더하기만 거쳐 곧게 지나감" width="703">

**파이프라인 위치** · ④ 모델 — 셀 구조 · 초기화

### 7-1. `A` LSTM 이 돌려주는 것 · `output, (h_n, c_n)` <sub>원본 셀 69 · 70</sub>

> 🎯 **LSTM은 무엇을 몇 개 돌려주고, h_n과 c_n은 같은가?**

💡 **쉽게 말하면**

- 돌려받는 것이 **세 개** · 칸별 요약 · 마지막 요약 · 마지막 메모.

**실행 흐름**

1. LSTM(10, 32)에 8줄
2. 네 모양
3. h_n · c_n 비교 · 파라미터

**먼저 예측** · h_n과 c_n은 같은 값일까? — 답은 코드 실행 결과 아래에

In [29]:
# LSTM 이 돌려주는 것 (원본 셀 70)
torch.manual_seed(SEED)                                      # 같은 시작점
lstm = nn.LSTM(10, 32, batch_first=True)                     # 입력 10 · hidden 32
x = to_tensor(xtr[:8])                                       # (8, 10, 10)
output, (h_n, c_n) = lstm(x)                                 # 괄호 안 두 개 = 튜플 풀기
print("x      :", tuple(x.shape))                            # 입력
print("output :", tuple(output.shape))                       # 칸마다 h
print("h_n    :", tuple(h_n.shape))                          # 마지막 h
print("c_n    :", tuple(c_n.shape))                          # 마지막 메모 c
print("h_n 과 c_n 이 같은 값인가 :", torch.equal(h_n, c_n))  # h = o × tanh(c) 라 다름
print("파라미터 RNN · LSTM :", sum(p.numel() for p in nn.RNN(10, 32).parameters()),  # 4벌 계산
      "·", sum(p.numel() for p in lstm.parameters()))

x      : (8, 10, 10)
output : (8, 10, 32)
h_n    : (1, 8, 32)
c_n    : (1, 8, 32)
h_n 과 c_n 이 같은 값인가 : False
파라미터 RNN · LSTM : 1408 · 5632


**결과** · 예측 정답은 다름 — False

- `c_n    : (1, 8, 32)` — 메모가 하나 더
- `h_n 과 c_n 이 같은 값인가 : False` — h = o × tanh(c)
- `파라미터 RNN · LSTM : 1408 · 5632` — 4벌 계산

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>output, h_n</code></td>
<td style="text-align: left;">RNN · GRU</td>
<td style="text-align: left;">두 개</td>
</tr>
<tr>
<td style="text-align: left;"><code>output, (h_n, c_n)</code></td>
<td style="text-align: left;">LSTM</td>
<td style="text-align: left;">세 개 (튜플)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `output, h_n = lstm(x)`로 받으면 h_n 자리에 튜플

> ✅ **한 줄 정리** · LSTM은 `output, (h_n, c_n)`을 돌려주고 h_n과 c_n은 다른 값이며 파라미터는 RNN의 4배다.

**딥러닝 코드에서는**

- 모델에서 셀을 바꿀 때 받는 줄도 함께 바꿀 것

### 7-2. `A` 문을 달아도 · 문이 얼마나 열렸나 · `forget_stats()` <sub>원본 셀 71 ~ 74 · L3b</sub>

> 🎯 **보통의 LSTM(forget bias 0)은 99칸 기울기가 살아 있고, 문은 얼마나 열려 있었나?**

💡 **쉽게 말하면**

- 같은 측정을 LSTMCell로 · **forget 문 bias 0** = 보통의 LSTM.
- 문 값을 백 칸 기록해 평균 · 곱을 봄.

**실행 흐름**

1. RNN · LSTM 기본 기울기
2. forget 문 값 백 칸 기록 (bias 0 · 2)

**먼저 예측** · LSTM 기본은 99칸에서 살아 있을까? — 답은 코드 실행 결과 아래에

In [30]:
# 같은 측정을 LSTMCell로 · forget 문 bias 0 = 'LSTM 기본' (원본 셀 72)
show("거슬러 간 칸", STEPS)                                      # 머리 줄
show("RNN", grad_profile("RNN"), ".1e")                          # 비교 기준
show("LSTM 기본", grad_profile("LSTM", forget_bias=0.0), ".1e")  # 문을 달았지만


# 문이 얼마나 열려 있었나 (원본 셀 74): 3과 7 사이가 99칸인 데이터
def forget_stats(forget_bias, T=100, hid=32, batch=64):                # 칸별 forget 문 평균값
    torch.manual_seed(SEED)                                            # 같은 시작점
    cell = nn.LSTMCell(10, hid)                                        # LSTM 상자
    with torch.no_grad():                                              # 값만 넣음
        cell.bias_ih[hid:2 * hid].fill_(forget_bias)                   # forget 구간
        cell.bias_hh[hid:2 * hid].fill_(0.0)                           # 한쪽은 0
    xs, _ = make_order_task(batch, T, gap=T - 1, seed=7)               # 3 · 7이 양 끝
    X = to_tensor(xs)                                                  # 텐서
    h, c = torch.zeros(batch, hid), torch.zeros(batch, hid)            # 처음 요약 · 메모
    fs = []                                                            # 칸마다 문 값 평균
    with torch.no_grad():                                              # 측정만
        for t in range(T):                                             # 100칸
            # 네 문 계산
            g = X[:, t, :] @ cell.weight_ih.T + cell.bias_ih + h @ cell.weight_hh.T + cell.bias_hh
            fs.append(torch.sigmoid(g[:, hid:2 * hid]).mean().item())  # forget 문 값 평균
            h, c = cell(X[:, t, :], (h, c))                            # 한 칸 진행
    return sum(fs) / len(fs), sum(math.log10(v) for v in fs)           # 평균 · 곱의 log10


for b in (0.0, 2.0):                                            # 더하는 값 0 · 2
    mean_f, log_prod = forget_stats(b)                          # 측정
    print(f"문에 더하는 값 {b:.0f} · 백 칸 평균 {mean_f:.4f}",  # 문 값 평균
          f"· 백 칸의 값을 다 곱하면 10^{log_prod:.1f}")

거슬러 간 칸           0        10        20        40        70        99


RNN              5.2e-02   1.1e-04   2.9e-07   1.3e-12   1.5e-20   0.0e+00


LSTM 기본        5.3e-02   4.6e-05   4.0e-07   2.8e-11   2.1e-17   0.0e+00


문에 더하는 값 0 · 백 칸 평균 0.4997 · 백 칸의 값을 다 곱하면 10^-30.1


문에 더하는 값 2 · 백 칸 평균 0.8779 · 백 칸의 값을 다 곱하면 10^-5.7


**결과** · 예측 정답은 죽음 — 0.0e+00

- `LSTM 기본        5.3e-02   4.6e-05   4.0e-07   2.8e-11   2.1e-17   0.0e+00` — RNN과 같이 99칸 0
- `문에 더하는 값 0 · 백 칸 평균 0.4997 · 백 칸의 값을 다 곱하면 10^-30.1` — 반쯤 열린 문 백 번
- `문에 더하는 값 2 · 백 칸 평균 0.8779 · 백 칸의 값을 다 곱하면 10^-5.7` — 열어 두면 곱이 훨씬 큼

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">bias 0</td>
<td style="text-align: left;">문 0.5 · 곱 10^-30</td>
<td style="text-align: left;">기본 초기화</td>
</tr>
<tr>
<td style="text-align: left;">bias 2</td>
<td style="text-align: left;">문 0.88 · 곱 10^-5.7</td>
<td style="text-align: left;">먼 의존</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 문이 있어서 살아나는 게 아님 → 문이 열려 있을 때 살아남

> ✅ **한 줄 정리** · forget bias 0인 LSTM은 문 값이 0.4997이라 백 칸 곱이 10^-30으로 RNN처럼 죽는다.

**딥러닝 코드에서는**

- `forget_stats`처럼 문 값을 직접 기록하면 LSTM이 무엇을 기억하는지 볼 수 있음

### 7-3. `A` 문을 열어두면 · `fill_(2.0)` <sub>원본 셀 75 · 76</sub>

> 🎯 **forget bias를 2 · 4로 두면 99칸 기울기는 얼마나 남는가?**

💡 **쉽게 말하면**

- 시작 문 값만 0.88 · 0.98로 옮김.

<img src="images/rn_forget.png" alt="LSTM forget bias 0 · 2 · 4의 칸별 기울기" width="941">

**실행 흐름**

1. bias 0 · 2 · 4 측정
2. 0스텝 ÷ 99스텝
3. sigmoid 시작값

**먼저 예측** · 열어둔 LSTM의 0스텝 ÷ 99스텝은? — 답은 코드 실행 결과 아래에

In [31]:
# 문을 열어두면 (원본 셀 76): forget 문 bias 2 · 4
show("거슬러 간 칸", STEPS)                                            # 머리 줄
show("RNN", grad_profile("RNN"), ".1e")                                # 기준
show("LSTM 기본", grad_profile("LSTM", forget_bias=0.0), ".1e")        # 0 → 문 값 0.5
show("LSTM 열어둠", grad_profile("LSTM", forget_bias=2.0), ".1e")      # 2 → 0.88
show("LSTM 활짝", grad_profile("LSTM", forget_bias=4.0), ".1e")        # 4 → 0.98
opened = grad_profile("LSTM", forget_bias=2.0)                         # 열어둔 경우
print(f"열어둔 경우 0스텝 ÷ 99스텝 : {opened[0] / opened[5]:,.0f}배")  # 해결이 아니라 늦춤
start = torch.sigmoid(torch.tensor([0., 2., 4.]))                      # bias → 문 시작값
print("sigmoid(0 · 2 · 4) :", [round(v, 2) for v in start.tolist()])   # 0.5 · 0.88 · 0.98

거슬러 간 칸           0        10        20        40        70        99


RNN              5.2e-02   1.1e-04   2.9e-07   1.3e-12   1.5e-20   0.0e+00
LSTM 기본        5.3e-02   4.6e-05   4.0e-07   2.8e-11   2.1e-17   0.0e+00


LSTM 열어둠      5.3e-02   1.0e-03   5.2e-04   1.9e-04   3.9e-05   1.6e-05
LSTM 활짝        5.3e-02   7.2e-04   5.9e-04   6.7e-04   9.8e-04   4.5e-03
열어둔 경우 0스텝 ÷ 99스텝 : 3,303배
sigmoid(0 · 2 · 4) : [0.5, 0.88, 0.98]


**결과** · 예측 정답은 3,303배 — 사라지진 않지만 줄어듦

- `LSTM 열어둠      5.3e-02   1.0e-03   5.2e-04   1.9e-04   3.9e-05   1.6e-05` — 99칸에도 남음
- `LSTM 활짝        5.3e-02   7.2e-04   5.9e-04   6.7e-04   9.8e-04   4.5e-03` — 멀수록 오히려 커지기도
- `열어둔 경우 0스텝 ÷ 99스텝 : 3,303배` — 해결이 아니라 크게 늦춤
- `sigmoid(0 · 2 · 4) : [0.5, 0.88, 0.98]` — 문 시작값

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">forget bias 0</td>
<td style="text-align: left;">죽음</td>
<td style="text-align: left;">짧은 의존</td>
</tr>
<tr>
<td style="text-align: left;">forget bias 2</td>
<td style="text-align: left;">남음 · 3,303배 감소</td>
<td style="text-align: left;">긴 의존</td>
</tr>
<tr>
<td style="text-align: left;">forget bias 4</td>
<td style="text-align: left;">더 남음</td>
<td style="text-align: left;">아주 긴 의존</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `bias_hh` 쪽은 0으로 · 두 bias가 더해짐

> ✅ **한 줄 정리** · forget 문을 열어 둔 LSTM만 99칸 뒤까지 기울기가 남으며, 그래도 3,303배 줄어들므로 해결이 아니라 늦춤이다.

**딥러닝 코드에서는**

- LSTM 초기화 요령 · forget 구간 bias를 1 ~ 2로

---
## 8. 실제 데이터에서 비교 (L4)

> **이 장의 흐름**
>
> - 원본 L4 · 같은 조건 → seed 3번 → 파라미터 맞춤 → 시간 순서 섞기
> - LSTM · GRU 식을 손계산으로 확인

<img src="images/rl_gru.png" alt="LSTM은 손잡이 3 · 줄 2, GRU는 손잡이 2 · 줄 1" width="718">

**파이프라인 위치** · ⑥ 평가 — 통제된 비교

### 8-1. `A` 같은 조건에서 RNN · LSTM · GRU · `report()` <sub>원본 셀 77 ~ 80 · L4 (미리 돌린 결과)</sub>

> 🎯 **HAR에서 hidden 64로 맞추면 세 모델은 몇 %이고 파라미터는 얼마인가?**

💡 **쉽게 말하면**

- **구조만** 바꾼 비교 · 데이터 · 학습 설정 · 분류기는 같음.

<img src="images/rn_compare.png" alt="HAR 같은 조건 비교 · 파라미터 맞춘 비교" width="721">

**실행 흐름**

1. RNN · LSTM · GRU 각각 학습
2. 정확도 · 파라미터 · 시간

**먼저 예측** · LSTM과 GRU 중 어느 쪽이 높을까? — 답은 코드 실행 결과 아래에

In [32]:
# 같은 조건 비교 (원본 셀 78 · 80): hidden 64 · Adam 1e-3 · batch 64 · 12 epoch · clip 5.0
report("RNN", 64)   # 문 없음
report("LSTM", 64)  # 문 3개 + 메모
report("GRU", 64)   # 문 2개

RNN   hidden 64   정확도 0.8073 · 파라미터  5,190 · 17.8s


LSTM  hidden 64   정확도 0.8683 · 파라미터 19,590 · 19.4s


GRU   hidden 64   정확도 0.9026 · 파라미터 14,790 · 15.6s


**결과** · 예측 정답은 이 PC에서는 GRU — 0.9026

- `RNN   hidden 64   정확도 0.8073 · 파라미터  5,190` — 강의 82.5%
- `LSTM  hidden 64   정확도 0.8683 · 파라미터 19,590` — 강의 90.6%
- `GRU   hidden 64   정확도 0.9026 · 파라미터 14,790` — 강의 90.1%

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">RNN 64</td>
<td style="text-align: left;">가장 작고 빠름 · 정확도 낮음</td>
<td style="text-align: left;">먼저 돌려 볼 것</td>
</tr>
<tr>
<td style="text-align: left;">LSTM 64</td>
<td style="text-align: left;">4벌 · 가장 큼</td>
<td style="text-align: left;">자유도</td>
</tr>
<tr>
<td style="text-align: left;">GRU 64</td>
<td style="text-align: left;">3벌</td>
<td style="text-align: left;">가볍게</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 실행 시간(초)은 환경마다 다름 · 여기 숫자는 GPU

> ✅ **한 줄 정리** · 같은 hidden 64에서 문 달린 LSTM · GRU가 RNN보다 6 ~ 10%p 높지만 파라미터도 3 ~ 4배다.

**딥러닝 코드에서는**

- 비교표에는 정확도 · 파라미터 · 시간을 같이

### 8-2. `B` 시드를 세 번 바꾸면 · `seed=s` · `statistics.mean()` <sub>원본 셀 81 · 82 (미리 돌린 결과)</sub>

> 🎯 **seed만 바꿔도 정확도는 얼마나 흔들리고, 모델 차이는 그보다 큰가?**

💡 **쉽게 말하면**

- 한 번 돌린 숫자는 **우연**이 섞임 → seed 3개로 폭을 봄.

<img src="images/rn_seeds.png" alt="seed별 정확도 · 평균과 폭" width="686">

**실행 흐름**

1. 모델마다 seed 3개
2. 평균 · 폭

**먼저 예측** · RNN의 흔들림 폭은? — 답은 코드 실행 결과 아래에

In [33]:
# 시드를 세 번 바꾸면 (원본 셀 82)
accs3 = {}                           # 모델별 정확도 3개 (8-4 · 과제에서 재사용)
for kind in ("RNN", "LSTM", "GRU"):  # 세 모델
    accs3[kind] = [run_har(kind, 64, seed=s)[0] for s in (SEED, 7, 1234)]  # seed만 바꿔 3번
    a = accs3[kind]                  # 짧은 이름
    print(f"{kind:<5} {a} · 평균 {statistics.mean(a):.4f} · 폭 {max(a) - min(a):.4f}")  # 흔들림

RNN   [0.8073, 0.8246, 0.8039] · 평균 0.8119 · 폭 0.0207


LSTM  [0.8683, 0.889, 0.8748] · 평균 0.8774 · 폭 0.0207


GRU   [0.9026, 0.9006, 0.9002] · 평균 0.9011 · 폭 0.0024


**결과** · 예측 정답은 0.0207

- `RNN   [0.8073, 0.8246, 0.8039] · 평균 0.8119 · 폭 0.0207` — 강의 폭 2.75%p
- `LSTM  [0.8683, 0.889, 0.8748] · 평균 0.8774 · 폭 0.0207` — 강의 0.95%p
- `GRU   [0.9026, 0.9006, 0.9002] · 평균 0.9011 · 폭 0.0024` — 강의 1.46%p

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">seed 1번</td>
<td style="text-align: left;">우연 포함</td>
<td style="text-align: left;">빠른 확인</td>
</tr>
<tr>
<td style="text-align: left;">seed 3 ~ 5번</td>
<td style="text-align: left;">평균 · 폭</td>
<td style="text-align: left;">결론 낼 때</td>
</tr>
<tr>
<td style="text-align: left;">차이 &gt; 폭</td>
<td style="text-align: left;">의미 있는 차이</td>
<td style="text-align: left;">판단 기준</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 세 번도 적음 → 과제 3에서 5번

> ✅ **한 줄 정리** · seed만 바꿔도 2%p 가까이 흔들리므로, 모델 차이는 흔들림 폭과 비교해 판단한다.

**딥러닝 코드에서는**

- 논문 · 보고서의 '평균 ± 표준편차' 표기가 이 이유

### 8-3. `A` 파라미터 수를 맞추면 · `HarNet(kind, hid)` <sub>원본 셀 83 ~ 86 (미리 돌린 결과)</sub>

> 🎯 **RNN을 3.5배 넘게 키우고 GRU를 LSTM 크기로 맞추면 간격이 바뀌는가?**

💡 **쉽게 말하면**

- hidden이 같다고 크기가 같은 게 아님 → **파라미터 수**를 맞춤.

**실행 흐름**

1. 학습 없이 파라미터 세기
2. RNN 128 · GRU 74 학습

**먼저 예측** · RNN 128은 LSTM 64를 따라잡을까? — 답은 코드 실행 결과 아래에

In [34]:
# 파라미터 수를 비슷하게 맞추면 (원본 셀 84 · 학습 없이 셈)
for kind, hid in (("RNN", 64), ("RNN", 128), ("GRU", 74), ("LSTM", 64)):  # 크기 조합
    n = sum(p.numel() for p in HarNet(kind, hid).parameters())            # 파라미터 수
    print(f"{kind:<5} hidden {hid:<4} 파라미터 {n:>6,}")                  # LSTM 64 근처로
print("손계산 RNN 64 :", 64 * (9 + 64) + 2 * 64 + 64 * 6 + 6)             # 1벌 + Linear


# 맞춘 크기로 다시 돌리면 (원본 셀 86)
report("RNN", 128)  # 3.5배 넘게 키운 RNN
report("GRU", 74)   # LSTM 근처 GRU

RNN   hidden 64   파라미터  5,190
RNN   hidden 128  파라미터 18,566
GRU   hidden 74   파라미터 19,320
LSTM  hidden 64   파라미터 19,590
손계산 RNN 64 : 5190


RNN   hidden 128  정확도 0.8554 · 파라미터 18,566 · 19.1s


GRU   hidden 74   정확도 0.9097 · 파라미터 19,320 · 23.7s


**결과** · 예측 정답은 못 따라잡음 — 0.8554

- `RNN   hidden 128  파라미터 18,566` — 3.5배 넘게 키움
- `GRU   hidden 74   파라미터 19,320` — LSTM 근처
- `손계산 RNN 64 : 5190` — 64 × (9 + 64) + 2 × 64 + 64 × 6 + 6
- `RNN   hidden 128  정확도 0.8554 · 파라미터 18,566` — 강의 83.2% · 여전히 LSTM · GRU 아래
- `GRU   hidden 74   정확도 0.9097 · 파라미터 19,320` — 강의 90.8%

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">구조만 바꾼 비교</td>
<td style="text-align: left;">hidden 같게</td>
<td style="text-align: left;">첫 확인</td>
</tr>
<tr>
<td style="text-align: left;">조건 맞춘 비교</td>
<td style="text-align: left;">파라미터 같게</td>
<td style="text-align: left;">크기 덕 가르기</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 파라미터가 같아도 학습 시간은 다름

> ✅ **한 줄 정리** · 파라미터를 맞춰도 RNN은 문 달린 쪽보다 낮으므로 차이는 크기 덕이 아니라 구조 덕이다.

**딥러닝 코드에서는**

- 모델 비교는 파라미터 수 · FLOPs를 맞춘 표를 함께

### 8-4. `B` 한 줄 안의 시간 순서를 섞으면 · `shuffle_time()` · `torch.Generator` <sub>원본 셀 87 · 88 (미리 돌린 결과)</sub>

> 🎯 **128칸의 순서만 섞으면 LSTM 정확도는 얼마나 떨어지는가?**

💡 **쉽게 말하면**

- 값의 모음은 그대로 · **순서만** 섞음 (학습 · 평가 모두).

**실행 흐름**

1. 줄마다 칸 순서 섞기
2. LSTM 64 · seed 3개
3. 그대로와 비교

**먼저 예측** · 섞으면 몇 %p 떨어질까? — 답은 코드 실행 결과 아래에

In [35]:
# 한 줄 안의 시간 순서를 섞으면 (원본 셀 88): 128칸 순서만 섞음 · LSTM hidden 64
def shuffle_time(X, seed):                                      # 줄마다 칸 순서 섞기
    g = torch.Generator().manual_seed(seed)                     # 이 작업 전용 난수
    out = X.clone()                                             # 원본 보존
    for b in range(X.shape[0]):                                 # 줄마다
        out[b] = X[b][torch.randperm(X.shape[1], generator=g)]  # 값 모음은 그대로 · 순서만
    return out                                                  # 섞은 텐서


normal = accs3["LSTM"]  # 그대로 = 8-2 결과 재사용 (같은 조건)
Xs, Xs_te = shuffle_time(Xn, 1), shuffle_time(Xn_te, 2)  # 학습 · 평가 모두 섞음
shuffled = [run_har("LSTM", 64, seed=s, Xa=Xs, Xb=Xs_te)[0] for s in (SEED, 7, 1234)]    # 3 seed
print(f"그대로  {normal} · 평균 {statistics.mean(normal):.4f}")                          # 순서 유지
print(f"섞음    {shuffled} · 평균 {statistics.mean(shuffled):.4f}")                      # 순서 파괴
print(f"떨어진 폭 {(statistics.mean(normal) - statistics.mean(shuffled)) * 100:.2f}%p")  # 순서의 몫

그대로  [0.8683, 0.889, 0.8748] · 평균 0.8774
섞음    [0.8751, 0.8673, 0.8738] · 평균 0.8721
떨어진 폭 0.53%p


**결과** · 예측 정답은 이 PC에서는 0.53%p

- `그대로  [0.8683, 0.889, 0.8748] · 평균 0.8774` — 8-2 결과 재사용
- `섞음    [0.8751, 0.8673, 0.8738] · 평균 0.8721` — 순서를 버려도 거의 그대로
- `떨어진 폭 0.53%p` — 강의 3.6%p보다 작음 · 무너지지 않음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">순서 섞기 검사</td>
<td style="text-align: left;">순서의 몫</td>
<td style="text-align: left;">새 데이터 첫 검사</td>
</tr>
<tr>
<td style="text-align: left;">값 분포만으로 맞힘</td>
<td style="text-align: left;">섞어도 성능 유지</td>
<td style="text-align: left;">활동마다 값 크기 · 흔들림이 다를 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - HAR은 값의 분포(얼마나 흔들리나)만으로도 활동을 꽤 가름 → 순서의 몫이 작게 나옴

> ✅ **한 줄 정리** · HAR에서 시간 순서를 섞어도 LSTM은 0.53%p만 떨어지므로, 이 데이터에서는 순서보다 값의 분포가 큰 몫을 한다.

**딥러닝 코드에서는**

- 순서가 거의 상관없다면 1D CNN · 통계 특징 + 트리 모델도 후보

### 8-5. `B` 식으로 보면 — LSTM과 GRU · `nn.GRUCell` · `nn.LSTMCell` <sub>원본 셀 89</sub>

> 🎯 **PyTorch 문서의 LSTM · GRU 식을 직접 계산하면 nn.LSTMCell · nn.GRUCell과 같은가?**

💡 **쉽게 말하면**

- LSTM · `c = f ⊙ c + i ⊙ g` · `h = o ⊙ tanh(c)` · 네 구간 [i | f | g | o].
- GRU · `h = (1 - z) ⊙ n + z ⊙ h` · 세 구간 [r | z | n] · 남길 몫 + 채울 몫 = 1.

**실행 흐름**

1. GRU 식으로 손계산 → 비교
2. LSTM 식으로 손계산 → 비교
3. forget bias 자리

**먼저 예측** · 식으로 계산한 값은 셀 결과와 같을까? — 답은 코드 실행 결과 아래에

In [36]:
# 식으로 보면 (원본 셀 89): GRU 식을 손으로 계산해 nn.GRUCell과 맞춰 보기
torch.manual_seed(SEED)                                  # 같은 시작점
gru = nn.GRUCell(10, 4)                                  # 작은 GRU 상자
x, h = torch.randn(2, 10), torch.randn(2, 4)             # 입력 · 이전 요약
Wi, Wh = gru.weight_ih, gru.weight_hh                    # (12, 10) · (12, 4) = [r | z | n]
gi, gh = x @ Wi.T + gru.bias_ih, h @ Wh.T + gru.bias_hh  # 입력 쪽 · 요약 쪽
r = torch.sigmoid(gi[:, 0:4] + gh[:, 0:4])               # 예전 요약을 얼마나 참고할지
z = torch.sigmoid(gi[:, 4:8] + gh[:, 4:8])               # 예전 요약을 얼마나 남길지
n = torch.tanh(gi[:, 8:12] + r * gh[:, 8:12])            # 새 후보
h_new = (1 - z) * n + z * h                              # 남길 몫 + 채울 몫 = 1
print("GRU 식 == nn.GRUCell :", torch.allclose(h_new, gru(x, h), atol=1e-6))  # 문서 식 확인


# LSTM 식: c = f ⊙ c + i ⊙ g · h = o ⊙ tanh(c)
lstm = nn.LSTMCell(10, 4)                                                # 작은 LSTM 상자
c = torch.randn(2, 4)                                                    # 이전 메모
g4 = x @ lstm.weight_ih.T + lstm.bias_ih + h @ lstm.weight_hh.T + lstm.bias_hh  # [i | f | g | o]
i, f = torch.sigmoid(g4[:, 0:4]), torch.sigmoid(g4[:, 4:8])              # 적는 문 · 잊는 문
g, o = torch.tanh(g4[:, 8:12]), torch.sigmoid(g4[:, 12:16])              # 적을 내용 · 꺼내는 문
c_new = f * c + i * g                                                    # 덧셈 길
h2, c2 = lstm(x, (h, c))                                                 # PyTorch 계산
print("LSTM 식 == nn.LSTMCell :", torch.allclose(c_new, c2, atol=1e-6),  # 메모 · 요약 둘 다
      torch.allclose(o * torch.tanh(c_new), h2, atol=1e-6))
print("forget bias 자리 bias_ih[H:2*H] · H = 4 →", list(range(4, 8)))    # 두 번째 구간

GRU 식 == nn.GRUCell : True
LSTM 식 == nn.LSTMCell : True True
forget bias 자리 bias_ih[H:2*H] · H = 4 → [4, 5, 6, 7]


**결과** · 예측 정답은 같음 — True

- `GRU 식 == nn.GRUCell : True` — r · z · n 식 그대로
- `LSTM 식 == nn.LSTMCell : True True` — 메모 c · 요약 h 둘 다
- `forget bias 자리 bias_ih[H:2*H] · H = 4 → [4, 5, 6, 7]` — 두 번째 구간

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">σ (sigmoid)</td>
<td style="text-align: left;">0 ~ 1 손잡이</td>
<td style="text-align: left;">문</td>
</tr>
<tr>
<td style="text-align: left;">⊙</td>
<td style="text-align: left;">같은 자리끼리 곱</td>
<td style="text-align: left;">문 적용</td>
</tr>
<tr>
<td style="text-align: left;">GRU r</td>
<td style="text-align: left;">예전 요약을 얼마나 참고</td>
<td style="text-align: left;">새 후보 n 계산</td>
</tr>
<tr>
<td style="text-align: left;">GRU z</td>
<td style="text-align: left;">예전 요약을 얼마나 남길지</td>
<td style="text-align: left;">1 - z만큼 새것</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 다른 자료는 z와 1 - z 자리가 바뀐 표기도 있음 · 이 노트북은 PyTorch 표기
> - 문서는 n도 gate라 부르지만 0 ~ 1 손잡이는 r · z 둘
> - 원본은 '문서(torch 2.14)'라고 적음 · 이 PC는 torch 2.11 → 버전 대신 셀 계산으로 식을 직접 확인

> ✅ **한 줄 정리** · PyTorch 식대로 손계산한 LSTM · GRU는 셀 결과와 같고, 파라미터는 문 순서대로 한 텐서에 붙어 있다.

**딥러닝 코드에서는**

- 가중치 초기화 · 분석을 할 때 [i | f | g | o] · [r | z | n] 순서를 알아야 함

---
## 9. 종합 실습 · 과제 2 ~ 4 빈칸 정답

> **이 장의 흐름**
>
> - 원본 과제 셀 세 개의 `___`를 채워 실행 (과제 1 · 5는 아래 설명)
> - 8장 함수(run_har · report · shuffle_time)와 결과(accs3 · normal · shuffled)를 그대로 씀

<img src="images/rn_tasks.png" alt="과제 2 · hidden 32 vs 128 정확도와 파라미터" width="800">

**파이프라인 위치** · ⑥ 평가 — 조건을 바꿔 다시 재기

> 🎯 **hidden을 키우면 · seed를 늘리면 · 앞 절반만 섞으면 RNN · LSTM · GRU 비교 결론이 바뀌는가?**

💡 **쉽게 말하면**

- **하나만 바꿔** 다시 재는 연습.
- 결론이 그대로면 믿을 만하고, 바뀌면 처음 결론이 우연이었던 것.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰는 문법</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">과제 1</td>
<td style="text-align: left;">8장 셀을 다시 돌려 강의 화면 숫자와 비교 (GPU라 끝자리 다름)</td>
<td style="text-align: left;">8-1 ~ 8-4</td>
</tr>
<tr>
<td style="text-align: left;">과제 2</td>
<td style="text-align: left;">hidden 32 · 128 × 세 모델 · 정확도 · 파라미터 표</td>
<td style="text-align: left;"><code>run_har</code></td>
</tr>
<tr>
<td style="text-align: left;">과제 3</td>
<td style="text-align: left;">seed 5개 (42 · 2026 추가) · 평균 · 폭</td>
<td style="text-align: left;"><code>accs3</code> 재사용</td>
</tr>
<tr>
<td style="text-align: left;">과제 4</td>
<td style="text-align: left;">앞 64칸만 섞기 → 그대로 · 전부 섞음과 비교</td>
<td style="text-align: left;"><code>shuffle_time</code> 참고</td>
</tr>
<tr>
<td style="text-align: left;">과제 5</td>
<td style="text-align: left;">강의 아홉 단계 중 막힌 단계 번호 적기</td>
<td style="text-align: left;">파일별 1 8장</td>
</tr>
</tbody>
</table>

**먼저 예측** · 앞 절반만 섞으면 정확도는 '그대로'와 '전부 섞음' 사이에 올까? — 답은 코드 실행 결과 아래에

In [37]:
# 과제 2 · hidden 크기 바꾸기: (___, ___) → (32, 128)
res2 = {}                               # (모델, hidden) → (정확도, 파라미터)
for kind in ("RNN", "LSTM", "GRU"):     # 세 모델
    for hid in (32, 128):               # 빈칸 정답
        acc, n, _ = run_har(kind, hid)  # 같은 조건 학습
        res2[kind, hid] = (acc, n)      # 저장
        print(f"{kind:<5} hidden {hid:<4} 정확도 {acc:.4f} · 파라미터 {n:>6,}")  # 표의 한 줄

RNN   hidden 32   정확도 0.7635 · 파라미터  1,574


RNN   hidden 128  정확도 0.8554 · 파라미터 18,566


LSTM  hidden 32   정확도 0.9040 · 파라미터  5,702


LSTM  hidden 128  정확도 0.9077 · 파라미터 71,942


GRU   hidden 32   정확도 0.8951 · 파라미터  4,326


GRU   hidden 128  정확도 0.9063 · 파라미터 54,150


In [38]:
# 과제 3 · 시드 늘리기: (SEED, 7, 1234, ___, ___) → 42 · 2026 추가
SEEDS5 = (SEED, 7, 1234, 42, 2026)                                    # 앞 3개 + 빈칸 2개
accs5 = {}                                                            # 모델별 정확도 5개
for kind in ("RNN", "LSTM", "GRU"):                                   # 세 모델
    accs5[kind] = accs3[kind] + [run_har(kind, 64, seed=s)[0] for s in SEEDS5[3:]]  # 앞 3개 재사용
    a = accs5[kind]                                                   # 짧은 이름
    print(f"{kind:<5} {a} · 평균 {statistics.mean(a):.4f} · 폭 {max(a) - min(a):.4f}")  # 흔들림
gap = statistics.mean(accs5["LSTM"]) - statistics.mean(accs5["RNN"])  # 모델 차이
# 비교
print(f"LSTM - RNN 평균 차이 {gap:.4f} · RNN 폭 {max(accs5['RNN']) - min(accs5['RNN']):.4f}")

RNN   [0.8073, 0.8246, 0.8039, 0.7465, 0.8113] · 평균 0.7987 · 폭 0.0781


LSTM  [0.8683, 0.889, 0.8748, 0.8833, 0.8928] · 평균 0.8816 · 폭 0.0245


GRU   [0.9026, 0.9006, 0.9002, 0.907, 0.9009] · 평균 0.9023 · 폭 0.0068
LSTM - RNN 평균 차이 0.0829 · RNN 폭 0.0781


In [39]:
# 과제 4 · 앞 절반만 섞기: ___ → X[b, :half][torch.randperm(half, generator=g)]
def shuffle_front_half(X, seed):                                        # 앞 64칸만 섞기
    g = torch.Generator().manual_seed(seed)                             # 이 작업 전용 난수
    out = X.clone()                                                     # 원본 보존
    half = X.shape[1] // 2                                              # 64칸
    for b in range(X.shape[0]):                                         # 줄마다
        out[b, :half] = X[b, :half][torch.randperm(half, generator=g)]  # 빈칸 정답
    return out                                                          # 섞은 텐서


Xf, Xf_te = shuffle_front_half(Xn, 1), shuffle_front_half(Xn_te, 2)  # 앞 절반 섞은 데이터
front = [run_har("LSTM", 64, seed=s, Xa=Xf, Xb=Xf_te)[0] for s in (SEED, 7, 1234)]  # 3 seed
print(f"그대로       평균 {statistics.mean(normal):.4f}")            # 8-4
print(f"앞 절반 섞음 {front} · 평균 {statistics.mean(front):.4f}")   # 과제 4
print(f"전부 섞음    평균 {statistics.mean(shuffled):.4f}")          # 8-4

그대로       평균 0.8774
앞 절반 섞음 [0.8833, 0.8884, 0.8918] · 평균 0.8878
전부 섞음    평균 0.8721


**결과** · 예측 정답은 이 PC에서는 사이가 아니라 오히려 '그대로'보다 높음 (0.8878)

- `RNN   hidden 32   정확도 0.7635 · 파라미터  1,574` — 작은 RNN은 크게 떨어짐
- `RNN   hidden 128  정확도 0.8554 · 파라미터 18,566` — 키워도 LSTM · GRU 32보다 낮음
- `LSTM  hidden 32   정확도 0.9040 · 파라미터  5,702` — LSTM은 32로도 충분
- `LSTM  hidden 128  정확도 0.9077 · 파라미터 71,942` — 4배 키워도 거의 그대로
- `GRU   hidden 32   정확도 0.8951 · 파라미터  4,326` — GRU도 32로 충분
- `RNN   [0.8073, 0.8246, 0.8039, 0.7465, 0.8113] · 평균 0.7987 · 폭 0.0781` — seed 42에서 0.7465 → 폭이 크게 늘어남
- `GRU   [0.9026, 0.9006, 0.9002, 0.907, 0.9009] · 평균 0.9023 · 폭 0.0068` — GRU가 가장 안정
- `LSTM - RNN 평균 차이 0.0829 · RNN 폭 0.0781` — 차이가 RNN 폭보다 큼 → 문 달린 쪽이 앞선다는 결론 유지
- `앞 절반 섞음 [0.8833, 0.8884, 0.8918] · 평균 0.8878` — 그대로(0.8774)보다 높음 · 순서의 몫이 작은 데이터
- `전부 섞음    평균 0.8721` — 8-4 결과

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">과제 2 · hidden</td>
<td style="text-align: left;">크기만 바꿈</td>
<td style="text-align: left;">문 달린 쪽은 작아도 충분한지</td>
</tr>
<tr>
<td style="text-align: left;">과제 3 · seed</td>
<td style="text-align: left;">우연만 바꿈</td>
<td style="text-align: left;">차이가 흔들림보다 큰지</td>
</tr>
<tr>
<td style="text-align: left;">과제 4 · 앞 절반 섞기</td>
<td style="text-align: left;">순서 일부만 파괴</td>
<td style="text-align: left;">순서가 어디서 중요한지</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 과제 4의 '그대로보다 높음'은 seed 3개 평균 차이 0.01 정도 → 흔들림 안일 수 있음
> - CPU로 돌리면 과제 셀 하나에 몇 분씩 · 시간(초)은 환경마다 다름

> ✅ **한 줄 정리** · hidden · seed를 바꿔도 '문 달린 LSTM · GRU가 RNN보다 앞선다'는 결론은 유지되고, HAR은 순서를 일부 섞어도 정확도가 거의 그대로다.

**딥러닝 코드에서는**

- 실험 보고는 '조건 하나를 바꿔 다시 재도 같은 결론인가'까지 적어야 완성

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">학습 목표</th>
<th style="text-align: left;">할 수 있게 된 것</th>
<th style="text-align: left;">확인 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1. 기준선</td>
<td style="text-align: left;">합 0.4880 · 마지막 0.6065 (기대 0.606) · 펼치기 길이 100 0.8840 · 상태 1.0000</td>
<td style="text-align: left;">1장</td>
</tr>
<tr>
<td style="text-align: left;">2. RNN · 모양</td>
<td style="text-align: left;">파라미터 1,474 · 손 계산 차이 0.0 · RNN 1.0000 · h_n (1, 10, 32) 함정</td>
<td style="text-align: left;">2장</td>
</tr>
<tr>
<td style="text-align: left;">3. 패딩</td>
<td style="text-align: left;">길이 5~60 · -1 0.500 · lens - 1 1.000 · 평균 0.483 · mask 1.000</td>
<td style="text-align: left;">3 · 4장</td>
</tr>
<tr>
<td style="text-align: left;">4. 소실 · LSTM</td>
<td style="text-align: left;">거리 98 20 epoch 0.517 → 60 epoch 1.000 · 20칸 17만 배 · 열어둔 LSTM 99칸 1.6e-05</td>
<td style="text-align: left;">5 ~ 7장</td>
</tr>
<tr>
<td style="text-align: left;">5. HAR 비교</td>
<td style="text-align: left;">RNN 0.8073 · LSTM 0.8683 · GRU 0.9026 · RNN 128도 0.8554 · seed 폭 0.0781</td>
<td style="text-align: left;">8 · 9장</td>
</tr>
</tbody>
</table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">등급</th>
<th style="text-align: left;">주제</th>
<th style="text-align: left;">반드시 기억할 것</th>
<th style="text-align: left;">딥러닝 코드에서</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">기준선</td>
<td style="text-align: left;">넘긴 것의 상한을 먼저 계산</td>
<td style="text-align: left;">순서 없는 모델부터 돌려 보기</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">RNN 분류기</td>
<td style="text-align: left;">마지막 칸 요약 → Linear</td>
<td style="text-align: left;"><code>self.fc(out[:, -1])</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">입력 모양</td>
<td style="text-align: left;">one_hot · batch_first=True</td>
<td style="text-align: left;"><code>print(x.shape)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">패딩 읽기</td>
<td style="text-align: left;">lens - 1 · mask</td>
<td style="text-align: left;"><code>out[arange(N), lens - 1]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">먼 의존 진단</td>
<td style="text-align: left;">거리 · epoch · 기울기를 나눠 잼</td>
<td style="text-align: left;"><code>retain_grad</code> 측정</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">LSTM forget bias</td>
<td style="text-align: left;">열어 둬야 기울기가 남음</td>
<td style="text-align: left;">bias[H:2H] = 1 ~ 2</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">통제된 비교</td>
<td style="text-align: left;">하나만 바꾸고 seed 여러 번</td>
<td style="text-align: left;">평균 · 폭 · 파라미터</td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;">포화 · output.grad</td>
<td style="text-align: left;">측정 함정</td>
<td style="text-align: left;">tanh 미분 · cell로 재기</td>
</tr>
</tbody>
</table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">🎯 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">합으로 질문 A가 0.4880인 이유는?</td>
</tr>
<tr>
<td style="text-align: left;">1-3</td>
<td style="text-align: left;">마지막 값의 기대 정확도 계산은?</td>
</tr>
<tr>
<td style="text-align: left;">1-4</td>
<td style="text-align: left;">모델을 키우면 60%를 넘나?</td>
</tr>
<tr>
<td style="text-align: left;">1-5</td>
<td style="text-align: left;">펼치기가 길어질수록 떨어지는 이유는?</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">RNNCell + for와 nn.RNN의 차이는?</td>
</tr>
<tr>
<td style="text-align: left;">2-4</td>
<td style="text-align: left;">batch_first=False로 (8, 10, 10)을 넣으면?</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">길이 5~60에서 -1로 읽으면?</td>
</tr>
<tr>
<td style="text-align: left;">4-3</td>
<td style="text-align: left;">길이 4~12에서는 왜 차이가 없나?</td>
</tr>
<tr>
<td style="text-align: left;">5-2</td>
<td style="text-align: left;">거리 98 실패는 기억 탓인가?</td>
</tr>
<tr>
<td style="text-align: left;">6-1</td>
<td style="text-align: left;">20칸 거슬러 가면 기울기는?</td>
</tr>
<tr>
<td style="text-align: left;">6-4</td>
<td style="text-align: left;">output.grad로 재면 안 되는 이유는?</td>
</tr>
<tr>
<td style="text-align: left;">6-5</td>
<td style="text-align: left;">한 칸당 비율 0.54는 어디서 오나?</td>
</tr>
<tr>
<td style="text-align: left;">7-2</td>
<td style="text-align: left;">LSTM 기본이 죽는 이유는?</td>
</tr>
<tr>
<td style="text-align: left;">8-3</td>
<td style="text-align: left;">RNN을 키우면 LSTM을 따라잡나?</td>
</tr>
<tr>
<td style="text-align: left;">8-4</td>
<td style="text-align: left;">HAR에서 순서를 섞으면?</td>
</tr>
</tbody>
</table>

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-5** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-5** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**7-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**8-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**8-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `make_order_task(5, 10, seed=3)`로 만든 줄에서 3이 먼저인 줄 수를 직접 세어 정답 합과 비교
2. `nn.LSTM(9, 64)` + `nn.Linear(64, 6)`의 파라미터 수를 손계산과 코드로 각각 출력

In [40]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. `compare_reads(5, 30)`으로 바꾸면 -1의 정확도는 0.500과 1.000 중 어느 쪽에 가까울까?
2. `grad_profile('LSTM', forget_bias=1.0)`의 99칸 기울기는 bias 0 · 2 중 어느 쪽에 가까울까?
3. `HarNet('GRU', 32)`의 파라미터 수는?

In [41]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">합에 순서 정보가 없음 · 질문 B는 0.9575</td>
</tr>
<tr>
<td style="text-align: left;">1-3</td>
<td style="text-align: left;">0.211 × 1 + 0.789 × 0.5 = 0.606</td>
</tr>
<tr>
<td style="text-align: left;">1-4</td>
<td style="text-align: left;">아니요 · 입력에 정보가 없음</td>
</tr>
<tr>
<td style="text-align: left;">1-5</td>
<td style="text-align: left;">입력 · 파라미터가 길이만큼 커지는데 데이터는 그대로</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">같은 가중치면 0.0 · 같은 계산</td>
</tr>
<tr>
<td style="text-align: left;">2-4</td>
<td style="text-align: left;">에러 없이 h_n (1, 10, 32)</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">0.500 · 빈 칸을 지난 요약</td>
</tr>
<tr>
<td style="text-align: left;">4-3</td>
<td style="text-align: left;">빈 칸이 최대 8칸이라 버그가 숨음</td>
</tr>
<tr>
<td style="text-align: left;">5-2</td>
<td style="text-align: left;">아님 · 60 epoch · lr 1e-2로 회복 → 학습 탓</td>
</tr>
<tr>
<td style="text-align: left;">6-1</td>
<td style="text-align: left;">약 17만 배 작아짐</td>
</tr>
<tr>
<td style="text-align: left;">6-4</td>
<td style="text-align: left;">손실에 쓴 칸만 값 · 99칸이 0</td>
</tr>
<tr>
<td style="text-align: left;">6-5</td>
<td style="text-align: left;">W_hh 고유값 크기 0.57 × tanh 미분 0.96</td>
</tr>
<tr>
<td style="text-align: left;">7-2</td>
<td style="text-align: left;">문 값 0.4997 → 백 칸 곱 10^-30</td>
</tr>
<tr>
<td style="text-align: left;">8-3</td>
<td style="text-align: left;">아님 · RNN 128도 0.8554 → 구조 덕</td>
</tr>
<tr>
<td style="text-align: left;">8-4</td>
<td style="text-align: left;">이 PC에서 0.53%p만 하락 · 값 분포의 몫이 큼</td>
</tr>
</tbody>
</table>

#### 코드 문제 답

```python
xs, ys = make_order_task(5, 10, seed=3)
print(sum(s.index(3) < s.index(7) for s in xs) == sum(ys))  # True
hand = 4 * (64 * (9 + 64) + 2 * 64) + 64 * 6 + 6
code = sum(p.numel() for p in nn.LSTM(9, 64).parameters()) + 64 * 6 + 6
print(hand, code)  # 19590 19590
```

#### 변형 문제 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">문제</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">빈 칸이 최대 25칸 · 대부분 줄이 빈 칸을 지나므로 0.500 쪽 (직접 실행해 확인)</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">문 시작값 sigmoid(1) = 0.73 → 0과 2 사이 · 99칸에서 살아남는지 직접 확인</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">GRU 3벌 · 3 × (32 × (9 + 32) + 2 × 32) + Linear 32 × 6 + 6 → 4,326개 (과제 2 출력과 같음)</td>
</tr>
</tbody>
</table>

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">틀린 내용</th>
<th style="text-align: left;">틀린 이유</th>
<th style="text-align: left;">올바른 원칙</th>
<th style="text-align: left;">재발 방지 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">(예) <code>output.grad</code>로 기울기 소실을 판단</td>
<td style="text-align: left;">output이 h와 같은 텐서라고 착각</td>
<td style="text-align: left;">손으로 편 cell + <code>h.retain_grad()</code></td>
<td style="text-align: left;">이 값은 다음 칸으로 가는 길 위의 h인가?</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
</tbody>
</table>

---
## 면접 30초 답변

**Q1. RNN의 기준선 실험이 왜 중요한가요?**
> - 합이나 마지막 값처럼 사람이 정한 요약으로 먼저 풀어 보면 그 입력으로 낼 수 있는 상한이 보입니다.
> - 합은 0.488로 찍기 수준, 마지막 값은 끝 칸이 표식인 비율로 계산한 0.606과 같아서, 모델을 키워도 안 되는 이유가 입력에 있음을 확인할 수 있었습니다.

**Q2. 패딩한 배치에서 무엇을 조심해야 하나요?**
> - output의 마지막 인덱스는 틀의 끝이라 짧은 줄은 빈 칸을 지난 요약입니다.
> - 길이 5~60에서 -1로 읽으면 0.500, lens - 1이나 mask 평균으로 읽으면 1.000이었고, 길이 편차가 작으면 이 버그가 숨으므로 편차가 큰 데이터로 확인해야 합니다.

**Q3. 기울기 소실을 어떻게 측정했나요?**
> - nn.RNN은 안의 h가 감춰져 있어 RNNCell을 손으로 펼치고 h.retain_grad로 칸별 기울기를 쟀습니다.
> - 20칸에 약 17만 배 줄었고, float64로 0이 아님을 확인했으며, output.grad는 손실에 쓴 칸만 값이 있어 측정에 쓰면 안 됩니다.

**Q4. RNN, LSTM, GRU 비교에서 무엇을 배웠나요?**
> - hidden을 같게 한 비교와 파라미터 수를 맞춘 비교, seed 여러 번을 했습니다.
> - RNN은 키워도 LSTM과 GRU보다 낮아 구조 덕이었고, LSTM과 GRU 차이는 흔들림 폭과 비슷해서 비용으로 고르는 것이 맞았습니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
<th style="text-align: left;">코드에서 사용한 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>make_order_task()</code></td>
<td style="text-align: left;">직접 정의</td>
<td style="text-align: left;">3과 7 순서 과제</td>
<td style="text-align: left;">1-1 <code>make_order_task(6000, 10, seed=1)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>train()</code></td>
<td style="text-align: left;">직접 정의</td>
<td style="text-align: left;">Adam · 배치 128 학습 → 정확도</td>
<td style="text-align: left;">1-2 <code>train(MLP(1), summarize(xtr, "sum"), Ytr, …)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>nn.Linear</code> · <code>nn.ReLU</code></td>
<td style="text-align: left;">torch.nn</td>
<td style="text-align: left;">기준선 MLP</td>
<td style="text-align: left;">1-2 <code>layers = [nn.Linear(i, width), nn.ReLU()]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>nn.RNN</code></td>
<td style="text-align: left;">torch.nn</td>
<td style="text-align: left;">칸마다 같은 상자</td>
<td style="text-align: left;">2-1 <code>self.r = layer(10, hid, batch_first=True)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>nn.RNNCell</code></td>
<td style="text-align: left;">torch.nn</td>
<td style="text-align: left;">상자 하나 · for 문</td>
<td style="text-align: left;">2-2 <code>h = cell(X[:, t, :], h)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>F.one_hot()</code></td>
<td style="text-align: left;">torch.nn.functional</td>
<td style="text-align: left;">번호 → 열 칸</td>
<td style="text-align: left;">1-1 <code>F.one_hot(torch.tensor(xs), num_classes=10)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>torch.equal()</code> · <code>torch.allclose()</code></td>
<td style="text-align: left;">torch</td>
<td style="text-align: left;">값 비교</td>
<td style="text-align: left;">2-5 <code>torch.equal(output[:, -1, :], h_n[0])</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>urllib.request.urlretrieve()</code></td>
<td style="text-align: left;">urllib</td>
<td style="text-align: left;">HAR 내려받기</td>
<td style="text-align: left;">3-1 <code>urllib.request.urlretrieve(URL, …)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>permute()</code></td>
<td style="text-align: left;">Tensor 메서드</td>
<td style="text-align: left;">(9, N, 128) → (N, 128, 9)</td>
<td style="text-align: left;">3-1 <code>torch.tensor(chans).permute(1, 2, 0)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>clip_grad_norm_()</code></td>
<td style="text-align: left;">torch.nn.utils</td>
<td style="text-align: left;">기울기 폭발 방지</td>
<td style="text-align: left;">3-2 <code>torch.nn.utils.clip_grad_norm_(m.parameters(), 5.0)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>out[arange(N), lens - 1]</code></td>
<td style="text-align: left;">인덱싱</td>
<td style="text-align: left;">줄마다 진짜 끝</td>
<td style="text-align: left;">4-1 <code>out[torch.arange(len(x), device=x.device), lens - 1]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>retain_grad()</code></td>
<td style="text-align: left;">Tensor 메서드</td>
<td style="text-align: left;">중간 기울기 보존</td>
<td style="text-align: left;">6-1 <code>h.retain_grad()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>torch.linalg.eigvals()</code> · <code>svdvals()</code></td>
<td style="text-align: left;">torch.linalg</td>
<td style="text-align: left;">반복 곱 배율</td>
<td style="text-align: left;">6-5 <code>torch.linalg.eigvals(W).abs().max()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>nn.LSTM</code> · <code>nn.LSTMCell</code></td>
<td style="text-align: left;">torch.nn</td>
<td style="text-align: left;">문 · 메모 c</td>
<td style="text-align: left;">7-1 <code>output, (h_n, c_n) = lstm(x)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>nn.GRU</code> · <code>nn.GRUCell</code></td>
<td style="text-align: left;">torch.nn</td>
<td style="text-align: left;">손잡이 2개</td>
<td style="text-align: left;">8-5 <code>gru = nn.GRUCell(10, 4)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>statistics.mean()</code></td>
<td style="text-align: left;">statistics</td>
<td style="text-align: left;">seed 평균</td>
<td style="text-align: left;">8-2 <code>statistics.mean(a)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>torch.Generator().manual_seed()</code></td>
<td style="text-align: left;">torch</td>
<td style="text-align: left;">섞기 전용 난수</td>
<td style="text-align: left;">8-4 <code>g = torch.Generator().manual_seed(seed)</code></td>
</tr>
</tbody>
</table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">✅ 할 것</th>
<th style="text-align: left;">⚠️ 하지 말 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">기준선부터 돌려 상한 확인</td>
<td style="text-align: left;">안 되면 모델부터 키우기</td>
</tr>
<tr>
<td style="text-align: left;">패딩이면 lens - 1 · mask</td>
<td style="text-align: left;"><code>output[:, -1]</code></td>
</tr>
<tr>
<td style="text-align: left;">길이 편차가 큰 데이터로 검사</td>
<td style="text-align: left;">짧은 길이로만 테스트</td>
</tr>
<tr>
<td style="text-align: left;">칸별 기울기는 cell + retain_grad</td>
<td style="text-align: left;"><code>output.grad</code>로 판단</td>
</tr>
<tr>
<td style="text-align: left;">LSTM forget bias를 열어 시작</td>
<td style="text-align: left;">기본값 그대로 먼 의존 학습</td>
</tr>
<tr>
<td style="text-align: left;">seed 여러 번 · 파라미터 맞춰 비교</td>
<td style="text-align: left;">한 번 돌린 숫자로 결론</td>
</tr>
</tbody>
</table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">짝</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">결론</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>-1</code> vs <code>lens - 1</code></td>
<td style="text-align: left;">틀의 끝 vs 진짜 끝</td>
<td style="text-align: left;">패딩이면 lens - 1</td>
</tr>
<tr>
<td style="text-align: left;">20 epoch vs 60 epoch</td>
<td style="text-align: left;">실패 vs 성공 (거리 98)</td>
<td style="text-align: left;">덜 배운 것</td>
</tr>
<tr>
<td style="text-align: left;">float32 0 vs float64</td>
<td style="text-align: left;">못 적음 vs 3.38e-28</td>
<td style="text-align: left;">학습에는 같음</td>
</tr>
<tr>
<td style="text-align: left;"><code>output.grad</code> vs <code>h.grad</code></td>
<td style="text-align: left;">손실에 쓴 칸만 vs 길 위의 h</td>
<td style="text-align: left;">측정은 h</td>
</tr>
<tr>
<td style="text-align: left;">LSTM bias 0 vs 2</td>
<td style="text-align: left;">99칸 0 vs 1.6e-05</td>
<td style="text-align: left;">열어 둠</td>
</tr>
<tr>
<td style="text-align: left;">hidden 같게 vs 파라미터 같게</td>
<td style="text-align: left;">구조만 vs 크기까지</td>
<td style="text-align: left;">둘 다 해 봄</td>
</tr>
</tbody>
</table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">언제</th>
<th style="text-align: left;">할 일</th>
<th style="text-align: left;">걸리는 시간</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">내일</td>
<td style="text-align: left;">기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td>
<td style="text-align: left;">15분</td>
</tr>
<tr>
<td style="text-align: left;">3일 뒤</td>
<td style="text-align: left;">변형 문제 3개를 직접 실행해 답 확인</td>
<td style="text-align: left;">30분</td>
</tr>
<tr>
<td style="text-align: left;">7일 뒤</td>
<td style="text-align: left;">4-2 compare_reads와 6장 grad_profile을 빈 셀에 다시 짜기</td>
<td style="text-align: left;">40분</td>
</tr>
<tr>
<td style="text-align: left;">14일 뒤</td>
<td style="text-align: left;">오답 노트에 남은 문제만 · 파일별 1 · 2의 ✅ 줄만 이어 읽기</td>
<td style="text-align: left;">15분</td>
</tr>
</tbody>
</table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 · `rnn.ipynb` 97셀을 순서 그대로 8장 35섹션 + 9장 과제로
- 원본 '미리 돌린 결과' 셀도 모두 실행 (GPU · 결정적 모드) · 실행 시간이 긴 셀은 표시
- 강의 화면 숫자와 다른 결과(60 epoch 거리 198 · 학습한 모델 기울기 · HAR 섞기)는 숨기지 않고 섹션에 적음
- 과제 2 · 3 · 4의 `___`는 정답으로 채워 실행 · 과제 1 · 5는 설명으로
- 모든 코드는 실행한 출력과 함께 저장 · 형식 v2.1 (FORMAT_V2.md 1 ~ 7절)
- 원본의 띄어 쓴 조사(`3 과 7 이`)는 붙여 씀 · 셀 89의 존댓말은 음슴체로 · 셀 90 '돔' → '돌아감'

### 읽는 법

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">표시</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code> · <code>B</code></td>
<td style="text-align: left;"><strong>A</strong> 딥러닝 코드에 바로 나오는 필수 · <strong>B</strong> 알아두면 되는 것</td>
</tr>
<tr>
<td style="text-align: left;">🎯</td>
<td style="text-align: left;">이번 섹션에서 답해야 할 질문</td>
</tr>
<tr>
<td style="text-align: left;">💡</td>
<td style="text-align: left;">쉬운 설명 · 비유</td>
</tr>
<tr>
<td style="text-align: left;">⚠️</td>
<td style="text-align: left;">자주 하는 실수</td>
</tr>
<tr>
<td style="text-align: left;">✅</td>
<td style="text-align: left;">기억할 결론 = 🎯 질문의 답</td>
</tr>
</tbody>
</table>

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">원본</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">이 노트북</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">셀 0 ~ 2 · 준비</td>
<td style="text-align: left;">seed · 데이터 함수 · train</td>
<td style="text-align: left;">0 · 1-1</td>
</tr>
<tr>
<td style="text-align: left;">셀 3 ~ 10 · L0</td>
<td style="text-align: left;">합 · 질문 바꾸기 · 평균 · 마지막 · 첫 값 · 마지막 칸 세기</td>
<td style="text-align: left;">1-2 · 1-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 11 ~ 20 · L0</td>
<td style="text-align: left;">모델 키우기 · 펼치기 · 데이터 양 · 봤는가 · 먼저 봤는가</td>
<td style="text-align: left;">1-4 ~ 1-6</td>
</tr>
<tr>
<td style="text-align: left;">셀 21 ~ 28 · L1</td>
<td style="text-align: left;">Net · 손으로 펼친 판 · nn.RNN · 학습</td>
<td style="text-align: left;">2-1 ~ 2-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 29 ~ 32 · L1</td>
<td style="text-align: left;">one_hot · batch_first · 정말 같은 값 (+ 2층 · 양방향)</td>
<td style="text-align: left;">2-4 · 2-5</td>
</tr>
<tr>
<td style="text-align: left;">셀 33 · 34 · L1</td>
<td style="text-align: left;">UCI HAR · 정규화 · HarNet · run_har · report</td>
<td style="text-align: left;">3-1 · 3-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 35 ~ 40 · L1</td>
<td style="text-align: left;">그대로 텐서 · 세 줄 채우기 · 무엇으로 채웠는지</td>
<td style="text-align: left;">3-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 41 ~ 46 · L1b</td>
<td style="text-align: left;">ReadNet · 길이 5~60 · -1 vs lens - 1 · 길이 4~12</td>
<td style="text-align: left;">4-1 ~ 4-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 47 ~ 58 · L2</td>
<td style="text-align: left;">거리 · 60 epoch · lr · 앞 → 뒤 세 모델 · 손으로 심은 가중치</td>
<td style="text-align: left;">5-1 ~ 5-5</td>
</tr>
<tr>
<td style="text-align: left;">셀 59 ~ 68 · L3</td>
<td style="text-align: left;">grad_profile · float64 · 학습한 모델(+ 포화 확인) · output.grad · 가중치 행렬</td>
<td style="text-align: left;">6-1 ~ 6-5</td>
</tr>
<tr>
<td style="text-align: left;">셀 69 ~ 76 · L3 · L3b</td>
<td style="text-align: left;">LSTM 반환 · LSTM 기본 · 문 값 · 문을 열어두면</td>
<td style="text-align: left;">7-1 ~ 7-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 77 ~ 88 · L4</td>
<td style="text-align: left;">RNN · LSTM · GRU · seed · 파라미터 맞춤 · 시간 순서 섞기</td>
<td style="text-align: left;">8-1 ~ 8-4</td>
</tr>
<tr>
<td style="text-align: left;">셀 89 · 식</td>
<td style="text-align: left;">LSTM · GRU 식 · 기호 · 강의 이름과 맞춰 보기 (손계산으로 확인)</td>
<td style="text-align: left;">8-5</td>
</tr>
<tr>
<td style="text-align: left;">셀 90 ~ 96 · 과제</td>
<td style="text-align: left;">과제 1 ~ 5 · 빈칸 3곳 정답</td>
<td style="text-align: left;">9장</td>
</tr>
</tbody>
</table>

### 전체 코드

1 ~ 9장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

In [42]:
# ---------- 1-1 ----------
# 3과 7이 하나씩 든 숫자 줄: 3이 먼저면 1 · 7이 먼저면 0
OTHERS = [d for d in range(10) if d not in (3, 7)]  # 3 · 7을 뺀 채울 숫자


def make_order_task(n, length=10, gap=None, seed=None):           # 순서 과제 줄 n개
    rng = random.Random(seed) if seed is not None else random     # seed가 있으면 전용 난수
    xs, ys = [], []                                               # 입력 줄 · 정답 목록
    for _ in range(n):                                            # 줄 n개
        filler = [rng.choice(OTHERS) for _ in range(length - 2)]  # 채울 숫자 (길이 - 2)
        if gap is None:                                           # 자리를 아무 데나
            i, j = sorted(rng.sample(range(length), 2))           # 3 · 7 자리 두 개
        else:                                                     # 간격을 정해 둘 때 (7-2)
            i = rng.randrange(0, length - gap)                    # 앞 자리
            j = i + gap                                           # 뒤 자리 = 앞 + 간격
        y = rng.randint(0, 1)                                     # 정답을 먼저 정함
        a, b = (3, 7) if y == 1 else (7, 3)                       # 정답 1이면 3이 앞
        s = filler[:]                                             # 채울 숫자 복사
        s.insert(i, a)                                            # 앞 자리에 첫 숫자
        s.insert(j, b)                                            # 뒤 자리에 둘째 숫자
        xs.append(s[:length])                                     # 길이를 맞춰 저장
        ys.append(y)                                              # 정답 저장
    return xs, ys                                                 # 줄 목록 · 정답 목록


def make_front_task(n, length, seed=None):                     # 3 · 7을 맨 앞 두 칸에 (5장)
    rng = random.Random(seed) if seed is not None else random  # 전용 난수
    xs, ys = [], []                                            # 입력 줄 · 정답 목록
    for _ in range(n):                                         # 줄 n개
        y = rng.randint(0, 1)                                  # 정답을 먼저 정함
        a, b = (3, 7) if y == 1 else (7, 3)                    # 정답 1이면 3이 앞
        xs.append([a, b] + [rng.choice(OTHERS) for _ in range(length - 2)])  # 뒤는 상관없는 숫자
        ys.append(y)                                           # 정답 저장
    return xs, ys                                              # 줄 목록 · 정답 목록


def to_tensor(xs):                                              # 숫자 줄 → (줄, 길이, 10) one-hot
    return F.one_hot(torch.tensor(xs), num_classes=10).float()  # 원본의 이중 for 문과 같은 결과


def train(model, X, Y, Xe, Ye, epochs=30, lr=3e-3):               # 학습 → 평가 정확도
    model = model.to(DEV)                                         # 모델을 GPU로 (원본은 CPU)
    X, Y, Xe, Ye = X.to(DEV), Y.to(DEV), Xe.to(DEV), Ye.to(DEV)   # 데이터도 같은 장치로
    opt = torch.optim.Adam(model.parameters(), lr=lr)             # Adam
    lf = nn.CrossEntropyLoss()                                    # 분류 손실
    for _ in range(epochs):                                       # epoch 반복
        p = torch.randperm(len(X))  # 순서 섞기 (CPU 난수 = 원본과 같은 순서)
        for i in range(0, len(X), 128):                           # 128줄씩
            j = p[i:i + 128]                                      # 이번 배치 번호
            opt.zero_grad()                                       # 지난 기울기 지우기
            lf(model(X[j]), Y[j]).backward()                      # 예측 → 손실 → 기울기
            opt.step()                                            # 가중치 한 걸음 수정
    with torch.no_grad():                                         # 평가는 기울기 기록 안 함
        return (model(Xe).argmax(1) == Ye).float().mean().item()  # 맞힌 비율

In [43]:
xtr, ytr = make_order_task(6000, 10, seed=1)                # 학습 6000줄
xte, yte = make_order_task(2000, 10, seed=2)                # 평가 2000줄 (다른 seed)
Ytr, Yte = torch.tensor(ytr), torch.tensor(yte)             # 정답 → 텐서
print(f"학습 {len(xtr)}줄 · 평가 {len(xte)}줄")             # 개수
print("예시 :", xtr[0], "→", "네" if ytr[0] else "아니요")  # 첫 줄과 정답


# ---------- 1-2 ----------
# 줄 하나 → 숫자 하나 (사람이 정한 요약 4가지)
def summarize(xs, how):                                       # 줄 → (N, 1)
    if how == "sum":                                          # 합
        return torch.tensor([[float(sum(s))] for s in xs])    # 줄마다 합
    if how == "mean":                                         # 평균
        return torch.tensor([[sum(s) / len(s)] for s in xs])  # 줄마다 평균
    if how == "last":                                         # 마지막 값
        return torch.tensor([[float(s[-1])] for s in xs])     # 줄마다 끝 칸
    return torch.tensor([[float(s[0])] for s in xs])          # first: 줄마다 첫 칸


class MLP(nn.Module):                                         # 요약값을 받는 분류기
    def __init__(self, i, width=32, depth=2):                 # 입력 크기 · 너비 · 은닉층 수
        super().__init__()                                    # nn.Module 준비 (필수)
        layers = [nn.Linear(i, width), nn.ReLU()]             # 첫 층
        for _ in range(depth - 1):                            # 은닉층 추가
            layers += [nn.Linear(width, width), nn.ReLU()]    # 같은 너비 층
        self.n = nn.Sequential(*layers, nn.Linear(width, 2))  # 출력 2 = 네 · 아니요

    def forward(self, x):  # 입력이 지나가는 길
        return self.n(x)   # 층을 차례로 통과


torch.manual_seed(SEED)                                                      # 모델 시작점 고정
acc = train(MLP(1), summarize(xtr, "sum"), Ytr, summarize(xte, "sum"), Yte)  # 합 하나로 학습
print(f"sum    {acc:.4f}")                                                   # 질문 A 정확도


# 질문만 바꾸기: 합이 45 이상인가 (원본 셀 6)
sums = [sum(s) for s in xtr]                                      # 줄마다 합
print("학습 데이터 합의 중앙값 :", sorted(sums)[len(sums) // 2])  # 기준 45의 근거
ytr45 = torch.tensor([1 if sum(s) >= 45 else 0 for s in xtr])     # 새 질문의 정답 (학습)
yte45 = torch.tensor([1 if sum(s) >= 45 else 0 for s in xte])     # 새 질문의 정답 (평가)
print(f"평가 데이터에서 합이 45 이상인 줄의 비율 : {yte45.float().mean().item():.3f}")  # 반반인지
torch.manual_seed(SEED)                                           # 같은 시작점
acc = train(MLP(1), summarize(xtr, "sum"), ytr45, summarize(xte, "sum"), yte45)  # 같은 합 모델
print(f"합 모델 · '합이 45 이상인가' → {acc:.4f}")                # 질문 B 정확도


# ---------- 1-3 ----------
# 나머지 세 가지 요약 (원본 셀 8)
for how in ("mean", "last", "first"):  # 평균 · 마지막 값 · 첫 값
    torch.manual_seed(SEED)            # 요약마다 같은 시작점
    acc = train(MLP(1), summarize(xtr, how), Ytr, summarize(xte, how), Yte)  # 요약 하나로 학습
    print(f"{how:<6} {acc:.4f}")       # 요약별 정확도

학습 6000줄 · 평가 2000줄
예시 : [2, 3, 1, 7, 5, 1, 9, 9, 9, 8] → 네


sum    0.4880
학습 데이터 합의 중앙값 : 45
평가 데이터에서 합이 45 이상인 줄의 비율 : 0.522


합 모델 · '합이 45 이상인가' → 0.9575


mean   0.4950


last   0.6065


first  0.6085


In [44]:
# 마지막 칸 세어보기 (원본 셀 10)
n_hit = sum(1 for s in xte if s[-1] in (3, 7))  # 끝 칸이 3 · 7인 줄 수
print(f"마지막 칸이 3 이나 7 인 줄 : {n_hit} / {len(xte)} ({n_hit / len(xte):.3f})")  # 비율
print(f"기대 정확도 = {n_hit / len(xte):.3f} × 1 + {1 - n_hit / len(xte):.3f} × 0.5"  # 확정 + 찍기
      f" = {n_hit / len(xte) + (1 - n_hit / len(xte)) / 2:.3f}")
print("마지막 칸이 3 도 7 도 아닌 줄 — 정답별로 하나씩")                              # 단서 없는 예
for want in (1, 0):  # 정답 1 · 0 하나씩
    k = next(i for i, s in enumerate(xte) if s[-1] not in (3, 7) and yte[i] == want)  # 첫 예
    print(" ", " ".join(map(str, xte[k])), "→", "네" if yte[k] else "아니요")  # 끝 칸만으로는 모름


# ---------- 1-4 ----------
# 마지막 값 하나만 넘긴 채 모델만 키우기 (원본 셀 12 · 미리 돌린 결과)
X_last, Xe_last = summarize(xtr, "last"), summarize(xte, "last")             # 입력은 끝 칸 하나
print(" 층  너비  epoch   정확도")                                           # 표 머리
# 5가지
for depth, width, ep in ((2, 32, 30), (4, 64, 30), (8, 128, 30), (4, 64, 150), (8, 256, 300)):
    torch.manual_seed(SEED)                                                  # 같은 시작점
    acc = train(MLP(1, width, depth), X_last, Ytr, Xe_last, Yte, epochs=ep)  # 크기 · 횟수만 바꿈
    print(f" {depth:>2}  {width:>4}  {ep:>5}   {acc:.4f}")                   # 한 줄 결과


# ---------- 1-5 ----------
# 길이 L 줄을 L × 10칸 한 줄로 펼쳐 MLP에 (원본 셀 14)
def flat(xs, L):                                   # (N, L) → (N, L*10)
    return to_tensor(xs).reshape(len(xs), L * 10)  # 칸 t의 숫자 d → 자리 t*10+d


def run_flat(L, n_train=6000):                              # 길이 · 학습량 → 정확도 · 파라미터
    xa, ya = make_order_task(n_train, L, seed=1)            # 학습 줄
    xb, yb = make_order_task(2000, L, seed=2)               # 평가 줄
    torch.manual_seed(SEED)                                 # 같은 시작점
    model = MLP(L * 10)                                     # 입력 크기가 길이를 따라 커짐
    acc = train(model, flat(xa, L), torch.tensor(ya), flat(xb, L), torch.tensor(yb))  # 학습
    return acc, sum(p.numel() for p in model.parameters())  # 정확도 · 파라미터 수


for L in (10, 20, 50, 100):                               # 길이 4가지
    acc, n = run_flat(L)                                  # 펼쳐서 학습
    print(f"길이 {L:>3} → {acc:.4f} · 파라미터 {n:,}개")  # 결과


# 길이 50 고정 · 학습 데이터만 줄이기 (원본 셀 16)
for n_train in (6000, 2000, 500):                                # 학습량 3가지
    acc, _ = run_flat(50, n_train)                               # 같은 모델 크기
    print(f"길이 50 · 학습 데이터 {n_train:>5,}건 → {acc:.4f}")  # 결과


# ---------- 1-6 ----------
# 끝까지 읽고 '3을 봤는가 · 7을 봤는가' 두 칸만 넘기기 (원본 셀 18)
def seen_two(xs):                                                      # 봤는가 2칸
    return torch.tensor([[float(3 in s), float(7 in s)] for s in xs])  # 3 있음 · 7 있음


torch.manual_seed(SEED)                                               # 같은 시작점
acc = train(MLP(2, depth=1), seen_two(xtr), Ytr, seen_two(xte), Yte, epochs=40)  # 2칸으로 학습
print(f"3 을 봤는가 · 7 을 봤는가 → {acc:.4f}")                       # 정확도
print("서로 다른 입력 종류 :", seen_two(xtr).unique(dim=0).tolist())  # 모든 줄이 같은 입력

마지막 칸이 3 이나 7 인 줄 : 422 / 2000 (0.211)
기대 정확도 = 0.211 × 1 + 0.789 × 0.5 = 0.606
마지막 칸이 3 도 7 도 아닌 줄 — 정답별로 하나씩
  8 8 6 9 5 3 0 7 0 6 → 네
  2 9 8 6 6 6 7 3 9 2 → 아니요
 층  너비  epoch   정확도


  2    32     30   0.6065


  4    64     30   0.5865


  8   128     30   0.6070


  4    64    150   0.6235


...

길이 100 → 0.8840 · 파라미터 33,154개


길이 50 · 학습 데이터 6,000건 → 0.9585


길이 50 · 학습 데이터 2,000건 → 0.8935


길이 50 · 학습 데이터   500건 → 0.6695


3 을 봤는가 · 7 을 봤는가 → 0.5120
서로 다른 입력 종류 : [[1.0, 1.0]]


In [45]:
# 상태 셋 (아직 안 봄 · 3이 먼저 · 7이 먼저) 중 끝 상태 넘기기 (원본 셀 20)
def first_seen(xs):                                                  # 끝 상태 3칸 one-hot
    return torch.tensor([[0.0, 1.0, 0.0] if s.index(3) < s.index(7)  # 3이 먼저
                         else [0.0, 0.0, 1.0] for s in xs])          # 7이 먼저


torch.manual_seed(SEED)                                                              # 같은 시작점
acc = train(MLP(3, depth=1), first_seen(xtr), Ytr, first_seen(xte), Yte, epochs=40)  # 상태로 학습
print(f"어느 쪽을 먼저 봤는가 → {acc:.4f}")                                          # 정확도

어느 쪽을 먼저 봤는가 → 1.0000


In [46]:
# ---------- 2-1 ----------
# 한 칸씩 읽고 요약을 넘기는 모델 (원본 셀 22)
class Net(nn.Module):                              # RNN 분류 모델
    def __init__(self, kind="RNN", hid=32):        # 셀 종류 · 요약 크기
        super().__init__()                         # nn.Module 준비 (필수)
        layer = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind]  # 이름으로 층 고르기
        self.r = layer(10, hid, batch_first=True)  # 입력 10 · (배치, 길이, 입력)
        self.o = nn.Linear(hid, 2)                 # 요약 → 점수 2개

    def forward(self, x):           # 입력이 지나가는 길
        o, _ = self.r(x)            # o: (N, L, hid)
        return self.o(o[:, -1, :])  # 마지막 칸 요약 → (N, 2)


net = Net("RNN")                                                        # 학습 전 모델 하나
for L in (10, 50, 200):                                                 # 길이만 바꿈
    xs, _ = make_order_task(4, L, seed=3)                               # 길이 L 4줄
    out = net(to_tensor(xs))                                            # 같은 모델에 넣음
    n = sum(p.numel() for p in net.parameters())                        # 파라미터 수
    print(f"길이 {L:>3} → 출력 {tuple(out.shape)} · 파라미터 {n:,}개")  # 길이와 무관
print("손계산 :", 32 * 10 + 32 * 32 + 32 + 32, "+", 32 * 2 + 2)         # RNN + Linear


# ---------- 2-2 ----------
# 손으로 펼친 판: RNNCell 하나 + for 문 (원본 셀 24)
torch.manual_seed(SEED)                     # 같은 시작점
rnn = nn.RNN(10, 32, batch_first=True)      # PyTorch가 감싸 둔 판
cell = nn.RNNCell(10, 32)                   # 상자 하나: 입력 10 · 요약 32
with torch.no_grad():                       # 맞춰 보려고 같은 가중치를 복사
    cell.weight_ih.copy_(rnn.weight_ih_l0)  # 입력 가중치
    cell.weight_hh.copy_(rnn.weight_hh_l0)  # 이전 요약 가중치
    cell.bias_ih.copy_(rnn.bias_ih_l0)      # bias 두 개
    cell.bias_hh.copy_(rnn.bias_hh_l0)      # (더해서 씀)

X = to_tensor(xtr[:4])                                         # 학습 데이터 네 줄 (4, 10, 10)
h = torch.zeros(4, 32)                                         # 처음 요약 = 0
hs = []                                                        # 칸마다 요약을 모음
for t in range(10):                                            # 10칸 차례로
    h = cell(X[:, t, :], h)                                    # 이번 칸 숫자 + 직전 요약 → 새 요약
    hs.append(h)                                               # 이번 칸 요약 저장
print("h.shape :", tuple(h.shape))                             # 길이 축이 사라짐
print("h[0, :4] :", [round(v, 4) for v in h[0, :4].tolist()])  # 첫 줄 요약 앞 4칸


# nn.RNN 과 맞춰 보기 (원본 셀 26)
manual = torch.stack(hs, 1)                          # 요약 10개 → (4, 10, 32)
output, h_n = rnn(X)                                 # 같은 입력 · 같은 가중치
print("(output - manual).abs().max() :", (output - manual).abs().max().item())  # 0이면 같은 계산
# 세 모양
print("x :", tuple(X.shape), "· output :", tuple(output.shape), "· h_n :", tuple(h_n.shape))
print("torch.allclose(output[:, -1, :], h_n[0]) :",  # 마지막 칸 = 마지막 요약
      torch.allclose(output[:, -1, :], h_n[0]))


# ---------- 2-3 ----------
# 학습시켜 보기 (원본 셀 28): 마지막 요약 → 네 · 아니요
torch.manual_seed(SEED)                                            # 같은 시작점
acc = train(Net("RNN"), to_tensor(xtr), Ytr, to_tensor(xte), Yte)  # 1장과 같은 데이터 · 학습 함수
print(f"RNN {acc:.4f}")                                            # 요약을 학습으로 정한 결과

길이  10 → 출력 (4, 2) · 파라미터 1,474개
길이  50 → 출력 (4, 2) · 파라미터 1,474개
길이 200 → 출력 (4, 2) · 파라미터 1,474개
손계산 : 1408 + 66
h.shape : (4, 32)
h[0, :4] : [0.101, -0.3322, 0.0603, 0.02]
(output - manual).abs().max() : 0.0
x : (4, 10, 10) · output : (4, 10, 32) · h_n : (1, 4, 32)
torch.allclose(output[:, -1, :], h_n[0]) : True


RNN 1.0000


In [47]:
# ---------- 2-4 ----------
# 모양 맞추기: 축만 붙이면 에러 (원본 셀 30)
digits = torch.tensor([xtr[0]])                          # (1, 10) · 숫자 그대로
rnn = nn.RNN(10, 32, batch_first=True)                   # 칸마다 숫자 10개를 기대
x_bad = digits.unsqueeze(-1).float()                     # (1, 10) → (1, 10, 1)
print("unsqueeze 로 축만 붙이면 :", tuple(x_bad.shape))  # 마지막 축이 1
try:
    rnn(x_bad)                                           # 칸마다 1개짜리 입력
except RuntimeError as e:                                # 에러를 잡아 이어서 실행
    print("  에러 :", str(e).splitlines()[0])            # 에러 첫 줄


# one_hot: 숫자 하나를 열 칸에 적기
x = F.one_hot(digits, num_classes=10).float()           # (1, 10) → (1, 10, 10)
output, h_n = rnn(x)                                    # 이번엔 통과
print("one_hot 으로 열 칸에 적으면 :", tuple(x.shape),  # 입력 모양
      "→ output", tuple(output.shape), "· h_n", tuple(h_n.shape))


# batch_first: 같은 (8, 10, 10)을 켜고 끄고
x8 = torch.randn(8, 10, 10)                           # 8줄 · 10칸 · 10개 (앞 두 축 길이가 다름)
for bf in (True, False):                              # 두 설정
    output, h_n = nn.RNN(10, 32, batch_first=bf)(x8)  # 에러 없이 둘 다 돎
    # 모양 비교
    print(f"batch_first={str(bf):<5}  x {tuple(x8.shape)} · output {tuple(output.shape)}"
          f" · h_n {tuple(h_n.shape)}")


# ---------- 2-5 ----------
# 정말 같은 값일까 (원본 셀 32): equal = 비트까지 · allclose = 작은 오차 허용
torch.manual_seed(SEED)                 # 같은 시작점
rnn = nn.RNN(10, 32, batch_first=True)  # 1층 · 한 방향
output, h_n = rnn(to_tensor(xtr[:8]))   # 8줄 넣기
# 비트 비교
print("torch.equal(output[:, -1, :], h_n[0])    :", torch.equal(output[:, -1, :], h_n[0]))
# 오차 허용
print("torch.allclose(output[:, -1, :], h_n[0]) :", torch.allclose(output[:, -1, :], h_n[0]))


# 같다는 조건이 깨지면: 2층 · 양방향 (강의 슬라이드 '그대로 쓸 수 있는 조건')
two = nn.RNN(10, 32, num_layers=2, batch_first=True)       # 2층
output, h_n = two(to_tensor(xtr[:8]))                      # h_n이 층마다 하나
last = torch.allclose(output[:, -1], h_n[-1])              # 마지막 층과 비교
first = torch.allclose(output[:, -1], h_n[0])              # 첫 층과 비교
print("2층   h_n :", tuple(h_n.shape),                     # 층마다 하나
      "· output[:, -1] == h_n[-1] :", last, "· == h_n[0] :", first)
bi = nn.RNN(10, 32, bidirectional=True, batch_first=True)  # 양방향
output, h_n = bi(to_tensor(xtr[:8]))                       # output 마지막 축 = 32 × 2
print("양방향 output :", tuple(output.shape), "· h_n :", tuple(h_n.shape))  # 방향 수만큼

unsqueeze 로 축만 붙이면 : (1, 10, 1)
  에러 : input.size(-1) must be equal to input_size. Expected 10, got 1
one_hot 으로 열 칸에 적으면 : (1, 10, 10) → output (1, 10, 32) · h_n (1, 1, 32)
batch_first=True   x (8, 10, 10) · output (8, 10, 32) · h_n (1, 8, 32)
batch_first=False  x (8, 10, 10) · output (8, 10, 32) · h_n (1, 10, 32)
torch.equal(output[:, -1, :], h_n[0])    : True
torch.allclose(output[:, -1, :], h_n[0]) : True
2층   h_n : (2, 8, 32) · output[:, -1] == h_n[-1] : True · == h_n[0] : False
양방향 output : (8, 10, 64) · h_n : (2, 8, 32)


In [48]:
# ---------- 3-1 ----------
# UCI HAR 내려받기 (처음 한 번만 · 약 60MB · 노트북 옆 UCI_HAR/ 폴더)
URL = ("https://archive.ics.uci.edu/static/public/240/"                  # 원본과 같은 주소
       "human+activity+recognition+using+smartphones.zip")
ROOT = os.path.join("UCI_HAR", "UCI HAR Dataset")  # 압축 푼 폴더 (원본은 노트북 폴더 바로 아래)
if not os.path.exists(ROOT):                                             # 처음 한 번만
    os.makedirs("UCI_HAR", exist_ok=True)                                # 받을 폴더
    urllib.request.urlretrieve(URL, os.path.join("UCI_HAR", "har.zip"))  # 내려받기
    # 바깥
    zipfile.ZipFile(os.path.join("UCI_HAR", "har.zip")).extract("UCI HAR Dataset.zip", "UCI_HAR")
    # 안쪽 압축
    zipfile.ZipFile(os.path.join("UCI_HAR", "UCI HAR Dataset.zip")).extractall("UCI_HAR")

CH = ["body_acc_x", "body_acc_y", "body_acc_z",  # 9채널 = 신호 3종 × 축 3개
      "body_gyro_x", "body_gyro_y", "body_gyro_z",
      "total_acc_x", "total_acc_y", "total_acc_z"]


def load_har(split):                                       # train · test → (N, 128, 9) · 라벨
    base = os.path.join(ROOT, split)                       # train 또는 test 폴더
    chans = []                                             # 채널마다 표
    for c in CH:                                           # 9채널
        with open(os.path.join(base, "Inertial Signals", f"{c}_{split}.txt")) as f:  # 채널 파일
            chans.append([[float(v) for v in line.split()] for line in f])  # 줄마다 128개
    X = torch.tensor(chans).permute(1, 2, 0).contiguous()  # (9, N, 128) → (N, 128, 9)
    with open(os.path.join(base, f"y_{split}.txt")) as f:  # 정답 파일
        y = torch.tensor([int(v) - 1 for v in f.read().split()])  # 라벨 1–6 → 0–5
    return X, y                                            # 입력 · 정답


Xh, Yh = load_har("train")                     # 학습 7352줄
Xh_te, Yh_te = load_har("test")                # 평가 2947줄
print("X.shape   :", tuple(Xh.shape))          # (개수, 길이, 채널)
print("Xte.shape :", tuple(Xh_te.shape))       # 평가도 128칸
mu = Xh.mean((0, 1), keepdim=True)             # 학습 데이터 채널별 평균 (1, 1, 9)
sd = Xh.std((0, 1), keepdim=True)              # 채널별 표준편차
Xn, Xn_te = (Xh - mu) / sd, (Xh_te - mu) / sd  # 평가도 학습 통계로 정규화
print(f"정규화 후 train 평균 {Xn.mean():.3f} · 표준편차 {Xn.std():.3f}")  # 0 · 1 확인


# ---------- 3-2 ----------
# 뒤 8장 · 과제에서 쓰는 정의 (원본 셀 34 뒷부분)
class HarNet(nn.Module):                                               # HAR 분류 모델
    def __init__(self, kind, hid):                                     # 셀 종류 · hidden
        super().__init__()                                             # nn.Module 준비 (필수)
        layer = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind]  # 이름으로 층 고르기
        self.r = layer(9, hid, batch_first=True)                       # 입력 9채널
        self.o = nn.Linear(hid, 6)                                     # 활동 6가지

    def forward(self, x):           # 입력이 지나가는 길
        o, _ = self.r(x)            # 모든 칸 요약
        return self.o(o[:, -1, :])  # 마지막 칸 → 점수 6개

X.shape   : (7352, 128, 9)
Xte.shape : (2947, 128, 9)
정규화 후 train 평균 0.000 · 표준편차 1.000


In [49]:
def run_har(kind, hid, seed=SEED, Xa=None, Xb=None, epochs=12):  # 학습 → 정확도 · 파라미터 · 초
    Xa = (Xn if Xa is None else Xa).to(DEV)                      # 기본은 정규화한 train
    Xb = (Xn_te if Xb is None else Xb).to(DEV)                   # 기본은 정규화한 test
    Ya, Yb = Yh.to(DEV), Yh_te.to(DEV)                           # 정답도 같은 장치로
    random.seed(seed)                                            # seed마다 다른 시작점
    torch.manual_seed(seed)                                      # 모델 초기값 · 섞는 순서
    m = HarNet(kind, hid).to(DEV)                                # 새 모델
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)              # Adam 1e-3
    lf = nn.CrossEntropyLoss()                                   # 분류 손실
    t = time.time()                                              # 시간 재기 시작
    for _ in range(epochs):                                      # epoch 반복
        p = torch.randperm(len(Xa))                              # 순서 섞기 (CPU 난수)
        for i in range(0, len(Xa), 64):                          # 64줄씩
            j = p[i:i + 64]                                      # 이번 배치
            opt.zero_grad()                                      # 지난 기울기 지우기
            lf(m(Xa[j]), Ya[j]).backward()                       # 손실 → 기울기
            torch.nn.utils.clip_grad_norm_(m.parameters(), 5.0)  # 기울기 폭발 방지
            opt.step()                                           # 가중치 한 걸음 수정
    with torch.no_grad():                                        # 평가는 기울기 기록 안 함
        acc = (m(Xb).argmax(1) == Yb).float().mean().item()      # test 정확도
    return round(acc, 4), sum(q.numel() for q in m.parameters()), time.time() - t  # 결과 셋


def report(kind, hid):                # 한 줄 보고
    acc, n, sec = run_har(kind, hid)  # 학습 · 평가
    # 초는 환경마다 다름
    print(f"{kind:<5} hidden {hid:<4} 정확도 {acc:.4f} · 파라미터 {n:>6,} · {sec:.1f}s")


n_lstm = sum(q.numel() for q in HarNet("LSTM", 64).parameters())  # 정의 확인
print("HarNet('LSTM', 64) 파라미터 :", n_lstm)                    # 8장에서 다시 셈


# ---------- 3-3 ----------
# 길이가 다른 세 줄을 그대로 텐서로 (원본 셀 36)
seqs = [[3, 1, 7],                      # 길이 3
        [9, 7, 0, 5, 3, 2],             # 길이 6
        [1, 3, 4, 7]]                   # 길이 4
try:
    torch.tensor(seqs)                  # 직사각형이 아니라 실패
except ValueError as e:                 # 에러를 잡아 이어서 실행
    print("그대로 텐서로 만들면 :", e)  # 에러 내용


# 세 줄 채워 보기 (원본 셀 38): 가장 긴 6칸 틀 · 빈 칸은 열 칸 모두 0
ML = max(len(s) for s in seqs)                                 # 가장 긴 길이 6
X = torch.zeros(len(seqs), ML, 10)                             # (3, 6, 10) 빈 틀
for b, s in enumerate(seqs):                                   # 줄마다
    for t, d in enumerate(s):                                  # 진짜 칸만
        X[b, t, d] = 1.0                                       # 숫자 d 자리에 1
lens = torch.tensor([len(s) for s in seqs])                    # 진짜 길이표
print("X.shape :", tuple(X.shape), "· lens :", lens.tolist())  # 모양 · 길이

HarNet('LSTM', 64) 파라미터 : 19590
그대로 텐서로 만들면 : expected sequence of length 3 at dim 1 (got 6)
X.shape : (3, 6, 10) · lens : [3, 6, 4]


In [50]:
# 무엇으로 채웠는지 (원본 셀 40): 빈 벡터 vs 숫자 0
X0 = X.clone()                                                            # 복사본
X0[0, 3:, 0] = 1.0  # 첫 줄 빈 칸을 '숫자 0'(0번 자리 1)으로 채우면
print("빈 벡터로 채움   · 첫 줄 칸별 합 :", X[0].sum(1).int().tolist())   # 빈 칸 = 0
print("숫자 0 으로 채움 · 첫 줄 칸별 합 :", X0[0].sum(1).int().tolist())  # 빈 칸도 1
# 구분 불가
print("숫자 0 으로 채운 빈 칸 == 둘째 줄의 진짜 0 칸 :", torch.equal(X0[0, 3], X[1, 2]))

빈 벡터로 채움   · 첫 줄 칸별 합 : [1, 1, 1, 0, 0, 0]
숫자 0 으로 채움 · 첫 줄 칸별 합 : [1, 1, 1, 1, 1, 1]
숫자 0 으로 채운 빈 칸 == 둘째 줄의 진짜 0 칸 : True


In [51]:
# ---------- 4-1 ----------
# 길이 5~60 줄을 빈 벡터로 채워 한 텐서에 · 요약 꺼내는 법 4가지 (원본 셀 42)
def make_ragged(n, lo, hi, seed):                                # 길이가 제각각인 줄
    rng = random.Random(seed)                                    # 전용 난수
    xs, ys = [], []                                              # 입력 줄 · 정답 목록
    for _ in range(n):                                           # 줄 n개
        L = rng.randint(lo, hi)                                  # 줄마다 길이
        x, y = make_order_task(1, L, seed=rng.randrange(10**6))  # 줄 하나
        xs.append(x[0])                                          # 줄 저장
        ys.append(y[0])                                          # 정답 저장
    return xs, ys                                                # 줄 목록 · 정답 목록


def pad(xs, T):                            # 길이 T 틀에 채우기 (뒤는 빈 벡터)
    X = torch.zeros(len(xs), T, 10)        # 빈 틀
    for b, s in enumerate(xs):             # 줄마다
        X[b, :len(s)] = to_tensor([s])[0]  # 앞쪽에 진짜 칸
    return X                               # 채운 텐서


class ReadNet(nn.Module):                          # 요약 꺼내는 방법만 다른 모델
    def __init__(self, how):                       # 꺼내는 방법
        super().__init__()                         # nn.Module 준비 (필수)
        self.r = nn.RNN(10, 32, batch_first=True)  # 같은 RNN
        self.o = nn.Linear(32, 2)                  # 같은 분류기
        self.how = how                             # 방법 기억

    def forward(self, x, lens):                                       # 입력 + 길이표
        out, _ = self.r(x)                                            # (N, T, 32)
        if self.how == "마지막 칸(-1)":                               # 틀의 끝
            h = out[:, -1, :]                                         # 빈 칸을 지난 값일 수 있음
        elif self.how == "실제 마지막 칸":                            # 줄마다 진짜 끝
            h = out[torch.arange(len(x), device=x.device), lens - 1]  # 줄마다 자기 끝
        elif self.how == "전부 평균":                                 # 빈 칸까지 평균
            h = out.mean(1)                                           # 길이 축 평균
        else:                                                         # mask 평균: 진짜 칸만 1
            m = (torch.arange(x.shape[1], device=x.device)[None, :] < lens[:, None])  # (N, T)
            m = m.float().unsqueeze(-1)                               # (N, T, 1)
            h = (out * m).sum(1) / m.sum(1)                           # 진짜 칸만 평균
        return self.o(h)                                              # 요약 → 점수


for how in ("마지막 칸(-1)", "실제 마지막 칸", "전부 평균", "mask 평균"):  # 4가지
    print(f"{how:<10} → 출력 {tuple(ReadNet(how)(X, lens).shape)}")        # 3-3의 세 줄로 모양 확인

마지막 칸(-1)  → 출력 (3, 2)
실제 마지막 칸   → 출력 (3, 2)
전부 평균      → 출력 (3, 2)
mask 평균    → 출력 (3, 2)


In [52]:
# ---------- 4-2 ----------
def compare_reads(lo, hi):  # 같은 데이터 · 같은 모델 · 꺼내는 법만 4가지
    xa, ya = make_ragged(3000, lo, hi, seed=11)                          # 학습 3000줄
    xb, yb = make_ragged(1000, lo, hi, seed=12)                          # 평가 1000줄
    T = max(len(s) for s in xa + xb)                                     # 가장 긴 길이 = 틀
    Xa, Xb = pad(xa, T).to(DEV), pad(xb, T).to(DEV)                      # 채워서 장치로
    La = torch.tensor([len(s) for s in xa], device=DEV)                  # 학습 길이표
    Lb = torch.tensor([len(s) for s in xb], device=DEV)                  # 평가 길이표
    Ya, Yb = torch.tensor(ya, device=DEV), torch.tensor(yb, device=DEV)  # 정답
    for how in ("마지막 칸(-1)", "실제 마지막 칸", "전부 평균", "mask 평균"):  # 4가지
        random.seed(SEED)                                                # 방법마다 같은 시작점
        torch.manual_seed(SEED)                                          # 모델 초기값 고정
        model = ReadNet(how).to(DEV)                                     # 새 모델
        opt = torch.optim.Adam(model.parameters(), lr=3e-3)              # Adam
        lf = nn.CrossEntropyLoss()                                       # 분류 손실
        for _ in range(25):                                              # 25 epoch
            p = torch.randperm(len(Xa))                                  # 순서 섞기
            for i in range(0, len(Xa), 64):                              # 64줄씩
                j = p[i:i + 64].to(DEV)                                  # 이번 배치
                opt.zero_grad()                                          # 지난 기울기 지우기
                lf(model(Xa[j], La[j]), Ya[j]).backward()                # 손실 → 기울기
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)  # 폭발 방지
                opt.step()                                               # 가중치 수정
        with torch.no_grad():                                            # 평가
            acc = (model(Xb, Lb).argmax(1) == Yb).float().mean().item()  # 정확도
        print(f"{how:<10} {acc:.3f}")                                    # 방법별 결과


print("길이 5~60")    # 편차가 큰 경우
compare_reads(5, 60)  # 4가지 비교


# ---------- 4-3 ----------
# -1 과 lens - 1 (원본 셀 44): 세 줄 X (3, 6, 10)에서 같은 자리인가
torch.manual_seed(SEED)                                             # 같은 시작점
output, _ = nn.RNN(10, 32, batch_first=True)(X)                     # 3-2에서 채운 세 줄
idx = torch.arange(len(X))                                          # 줄 번호 0 · 1 · 2
print("줄마다 output[:, -1] 과 output[idx, lens - 1] 이 같은가 :",  # 길이 6 줄만 True
      [torch.equal(a, b) for a, b in zip(output[:, -1, :], output[idx, lens - 1])])
m = (torch.arange(X.shape[1])[None, :] < lens[:, None]).int()       # 진짜 칸 = 1
print("mask 첫 줄 :", m[0].tolist())                                # 1 1 1 0 0 0


# 길이 4~12 로 좁히면 (원본 셀 46)
print("길이 4~12")    # 편차가 작은 경우
compare_reads(4, 12)  # 4가지 비교

길이 5~60


마지막 칸(-1)  0.500


실제 마지막 칸   1.000


전부 평균      0.483


mask 평균    1.000
줄마다 output[:, -1] 과 output[idx, lens - 1] 이 같은가 : [False, True, False]
mask 첫 줄 : [1, 1, 1, 0, 0, 0]
길이 4~12


마지막 칸(-1)  1.000


실제 마지막 칸   1.000


전부 평균      0.999


mask 평균    1.000


In [53]:
# ---------- 5-1 ----------
# 3 · 7 을 맨 앞에 두고 끝까지의 거리를 늘리기 · 같은 20 epoch (원본 셀 48)
for dist in (8, 48, 78, 98, 198, 398):                        # 거리 = 길이 - 2
    L = dist + 2                                              # 줄 길이
    xa, ya = make_front_task(4000, L, seed=1)                 # 학습 4000줄
    xb, yb = make_front_task(1000, L, seed=2)                 # 평가 1000줄
    torch.manual_seed(SEED)                                   # 같은 시작점
    acc = train(Net("RNN"), to_tensor(xa), torch.tensor(ya),  # 같은 모델 · 같은 예산
                to_tensor(xb), torch.tensor(yb), epochs=20)
    print(f"거리 {dist:>3} → {acc:.3f}")                      # 거리별 정확도


# ---------- 5-2 ----------
# 학습을 세 배 더 시키면 · 60 epoch (원본 셀 50)
for dist in (98, 198, 398):                                   # 20 epoch에서 무너진 거리
    xa, ya = make_front_task(4000, dist + 2, seed=1)          # 학습
    xb, yb = make_front_task(1000, dist + 2, seed=2)          # 평가
    torch.manual_seed(SEED)                                   # 같은 시작점
    acc = train(Net("RNN"), to_tensor(xa), torch.tensor(ya),  # epoch만 60
                to_tensor(xb), torch.tensor(yb), epochs=60)
    print(f"60 epoch · 거리 {dist:>3} → {acc:.3f}")           # 결과


# 횟수 대신 학습률을 키우면 (20 epoch · lr 1e-2)
xa, ya = make_front_task(4000, 100, seed=1)               # 거리 98
xb, yb = make_front_task(1000, 100, seed=2)               # 평가
torch.manual_seed(SEED)                                   # 같은 시작점
acc = train(Net("RNN"), to_tensor(xa), torch.tensor(ya),  # lr만 바꿈
            to_tensor(xb), torch.tensor(yb), epochs=20, lr=1e-2)
print(f"20 epoch · 학습률 1e-2 · 거리  98 → {acc:.3f}")   # 결과


# ---------- 5-3 ----------
# 앞 → 뒤: 맨 앞 두 칸만 맞바꾼 두 줄의 요약 차이가 칸마다 얼마나 남나 (원본 셀 52)
def plain(v):                                # 0.0000077 처럼 소수로 적기
    return format(Decimal(f"{v:.3g}"), "f")  # 유효숫자 3자리


def whisper(model, L=100, n=512, cols=(1, 2, 5, 10, 20, 40, 70, 99)):  # 칸별 상대 차이
    xs, _ = make_front_task(n, L, seed=31)                             # 512줄
    xs_b = [[s[1], s[0]] + s[2:] for s in xs]                          # 맨 앞 두 칸만 맞바꿈
    r = model.r.cpu()                                                  # 비교는 CPU에서
    with torch.no_grad():                                              # 측정만
        oa, _ = r(to_tensor(xs))                                       # 원래 줄 요약
        ob, _ = r(to_tensor(xs_b))                                     # 바꾼 줄 요약
    # 차이 ÷ 크기
    diff = {t: ((oa[:, t] - ob[:, t]).norm(dim=1) / oa[:, t].norm(dim=1)).mean().item()
            for t in cols}
    model.to(DEV)                                                      # 장치 원래대로
    return diff                                                        # {칸: 평균 상대 차이}


torch.manual_seed(SEED)                  # 같은 시작점
untrained = Net("RNN")                   # 학습 전 모델
for t, v in whisper(untrained).items():  # 칸마다
    print(f"{t:>2}칸  {plain(v)}")       # 상대 차이
# 측정의 바닥
print("float32 가 구분하는 가장 작은 상대 차이 :", plain(torch.finfo(torch.float32).eps))

거리   8 → 1.000


거리  48 → 1.000


거리  78 → 1.000


거리  98 → 0.517


거리 198 → 0.501


...

60 epoch · 거리  98 → 1.000


60 epoch · 거리 198 → 0.499


60 epoch · 거리 398 → 0.473


20 epoch · 학습률 1e-2 · 거리  98 → 1.000


 1칸  0.736
 2칸  0.378
 5칸  0.061
10칸  0.00255
20칸  0.00000767
40칸  0.0000000782
70칸  0.000000079
99칸  0.0000000794
float32 가 구분하는 가장 작은 상대 차이 : 0.000000119


In [54]:
# ---------- 5-4 ----------
# 같은 측정을 학습시킨 모델로 · 거리 98 (원본 셀 54 · 56)
def train_front(dist, epochs):                           # 거리 · epoch → 학습한 모델 · 정확도
    xa, ya = make_front_task(4000, dist + 2, seed=1)     # 학습
    xb, yb = make_front_task(1000, dist + 2, seed=2)     # 평가
    torch.manual_seed(SEED)                              # 같은 시작점
    model = Net("RNN")                                   # 새 모델
    acc = train(model, to_tensor(xa), torch.tensor(ya),  # 같은 학습 함수
                to_tensor(xb), torch.tensor(yb), epochs=epochs)
    return model, acc                                    # 모델 · 정확도


trained = {}                                              # epoch별 모델 (6-3에서 다시 씀)
for ep in (20, 60):                                       # 20 · 60 epoch
    trained[ep], acc = train_front(98, ep)                # 거리 98 학습
    print(f"{ep} epoch 모델 · 거리 98 정확도 {acc:.3f}")  # 정확도
    # 칸별 차이
    print("   " + " · ".join(f"{t}칸 {plain(v)}" for t, v in whisper(trained[ep]).items()))


# ---------- 5-5 ----------
# 학습 없이 손으로 심은 가중치 (원본 셀 58): 요약을 그대로 넘기는 RNNCell
def handmade(L, scale=0.5):                                       # 길이 L → 정확도 · 끝에 남은 크기
    torch.manual_seed(SEED)                                       # 같은 시작점
    cell = nn.RNNCell(10, 8)                                      # 요약 8칸
    with torch.no_grad():                                         # 가중치를 직접 적음
        cell.weight_ih.zero_()                                    # 입력 가중치 0
        cell.bias_ih.zero_()                                      # bias 0
        cell.bias_hh.zero_()                                      # bias 0
        cell.weight_hh.copy_(torch.eye(8))                        # 이전 요약을 그대로 넘김
        cell.weight_ih[0, 3] = scale                              # 3이면 첫 칸에 +0.5
        cell.weight_ih[0, 7] = -scale                             # 7이면 첫 칸에 -0.5
    xs, ys = make_front_task(1000, L, seed=2)                     # 평가 1000줄
    X, Y = to_tensor(xs), torch.tensor(ys)                        # 텐서
    h = torch.zeros(1000, 8)                                      # 처음 요약 0
    with torch.no_grad():                                         # 측정만
        for t in range(L):                                        # L칸
            h = cell(X[:, t, :], h)                               # 같은 상자 반복
    acc = max(((h[:, 0] > 0).long() == Y).float().mean().item(),  # 부호로 판정 (양쪽 중 큰 쪽)
              ((h[:, 0] < 0).long() == Y).float().mean().item())
    return acc, h[:, 0].abs().mean().item()                       # 정확도 · 첫 칸 크기


for L in (10, 100, 400):                                               # 거리 8 · 98 · 398
    acc, size = handmade(L)                                            # 손 가중치로
    print(f"거리 {L - 2:>3} → {acc:.3f} · 끝에 남은 크기 {size:.4f}")  # 결과

20 epoch 모델 · 거리 98 정확도 0.517


   1칸 1.72 · 2칸 1.42 · 5칸 0.971 · 10칸 0.228 · 20칸 0.0139 · 40칸 0.000151 · 70칸 0.000000572 · 99칸 0.00000013


60 epoch 모델 · 거리 98 정확도 1.000


   1칸 1.91 · 2칸 1.99 · 5칸 2 · 10칸 2 · 20칸 2 · 40칸 2 · 70칸 2 · 99칸 2
거리   8 → 1.000 · 끝에 남은 크기 0.0377


거리  98 → 1.000 · 끝에 남은 크기 0.0362


거리 398 → 1.000 · 끝에 남은 크기 0.0322


In [55]:
# ---------- 6-1 ----------
# 뒤 → 앞: 마지막 손실의 기울기가 k칸 거슬러 간 요약에 얼마나 오나 (원본 셀 60 · CPU)
STEPS = (0, 10, 20, 40, 70, 99)  # 거슬러 간 칸 수


def grad_profile(kind, T=100, hid=32, batch=64,           # 칸별 기울기 크기
                 forget_bias=None, dtype=torch.float32, steps=STEPS, model=None):
    torch.manual_seed(SEED)                               # 같은 시작점
    cell = {"RNN": nn.RNNCell, "LSTM": nn.LSTMCell}[kind](10, hid).to(dtype)  # 손으로 펼칠 상자
    head = nn.Linear(hid, 2).to(dtype)                    # 끝 분류기
    if model is not None:                                 # 학습한 Net의 가중치를 옮김 (6-3)
        with torch.no_grad():                             # 복사만
            for a, b in [("weight_ih", "weight_ih_l0"), ("weight_hh", "weight_hh_l0"),  # 이름 짝
                         ("bias_ih", "bias_ih_l0"), ("bias_hh", "bias_hh_l0")]:
                getattr(cell, a).copy_(getattr(model.r, b).cpu())  # RNN 가중치
            head.weight.copy_(model.o.weight.cpu())       # 분류기 가중치
            head.bias.copy_(model.o.bias.cpu())           # 분류기 bias
    if forget_bias is not None:  # bias 순서 [input | forget | cell | output] (7장)
        with torch.no_grad():                             # 값만 넣음
            cell.bias_ih[hid:2 * hid].fill_(forget_bias)  # forget 구간
            cell.bias_hh[hid:2 * hid].fill_(0.0)          # 두 bias가 더해지므로 한쪽은 0
    xs, ys = make_front_task(batch, T, seed=7)            # 64줄 · 길이 T
    X, Y = to_tensor(xs).to(dtype), torch.tensor(ys)      # 자료형 맞춤
    h = torch.zeros(batch, hid, dtype=dtype)              # 처음 요약
    c = torch.zeros(batch, hid, dtype=dtype)              # LSTM 메모
    hs = []                                               # 칸마다 요약
    for t in range(T):                                    # T칸 차례로
        if kind == "RNN":                                 # RNN은 h만
            h = cell(X[:, t, :], h)                       # 새 요약
        else:                                             # LSTM은 h · c 둘
            h, c = cell(X[:, t, :], (h, c))               # 새 요약 · 새 메모
        h.retain_grad()                                   # 중간 요약의 기울기를 남겨 달라고 표시
        hs.append(h)                                      # 저장
    nn.CrossEntropyLoss()(head(hs[-1]), Y).backward()     # 끝에서 손실 → 역전파
    return [hs[T - 1 - b].grad.norm().item() for b in steps]  # b칸 거슬러 간 기울기 크기


def show(name, values, fmt=".2e"):                            # 한글 2칸 폭으로 줄 맞춤
    width = sum(2 if ord(ch) > 0x1100 else 1 for ch in name)  # 화면 폭
    print(name + " " * (14 - width) + "".join(f"{v:>10}" if isinstance(v, int)  # 정수는 그대로
                                              else f"{v:>10{fmt}}" for v in values))


show("거슬러 간 칸", STEPS)                                          # 머리 줄
rnn_prof = grad_profile("RNN")                                       # 학습 전 RNN
show("RNN", rnn_prof)                                                # 칸별 크기
ratios = [f"{rnn_prof[0] / rnn_prof[k]:,.0f}배" for k in (1, 2, 3)]  # 0칸 ÷ k칸
print("배율 10 · 20 · 40스텝 :", " · ".join(ratios))                 # 줄어든 배율


# ---------- 6-2 ----------
# 0 이 진짜 0 일까 (원본 셀 62): 같은 계산을 float64로
show("거슬러 간 칸", STEPS)                                    # 머리 줄
show("RNN float32", grad_profile("RNN"))                       # 99칸이 0
show("RNN float64", grad_profile("RNN", dtype=torch.float64))  # 0이 아니라 아주 작은 수
every = grad_profile("RNN", steps=range(100))                  # 0 ~ 99칸 전부
print("float32 에서 처음 0 으로 찍히는 자리 :",                # 적을 수 있는 한계
      next(b for b in range(100) if every[b] == 0.0), "스텝 거슬러 간 곳")

거슬러 간 칸           0        10        20        40        70        99
RNN             5.16e-02  1.06e-04  2.94e-07  1.33e-12  1.50e-20  0.00e+00
배율 10 · 20 · 40스텝 : 486배 · 175,503배 · 38,772,643,119배
거슬러 간 칸           0        10        20        40        70        99
RNN float32     5.16e-02  1.06e-04  2.94e-07  1.33e-12  1.50e-20  0.00e+00
RNN float64     5.16e-02  1.06e-04  2.94e-07  1.33e-12  1.50e-20  3.38e-28


float32 에서 처음 0 으로 찍히는 자리 : 77 스텝 거슬러 간 곳


In [56]:
# ---------- 6-3 ----------
# 학습시킨 모델의 기울기 (원본 셀 64): 5-4의 20 · 60 epoch 모델
show("거슬러 간 칸", STEPS)                                      # 머리 줄
show("학습 전", grad_profile("RNN"))                             # 출발점
for ep in (20, 60):                                              # 학습한 두 모델
    show(f"{ep} epoch", grad_profile("RNN", model=trained[ep]))  # 가중치를 옮겨 측정


# 기울기가 0이 된 이유 확인: 요약 값이 ±1에 붙으면(포화) tanh 미분 1 - h² 이 0 근처
xs7, _ = make_front_task(64, 100, seed=7)                                # 같은 데이터
torch.manual_seed(SEED)                                                  # 학습 전 모델 시작점
# 세 모델
for name, m in [("학습 전", Net("RNN")), ("20 epoch", trained[20]), ("60 epoch", trained[60])]:
    with torch.no_grad():                                                # 측정만
        o, _ = m.r.cpu()(to_tensor(xs7))                                 # 칸별 요약 (64, 100, 32)
    m.to(DEV)                                                            # 장치 원래대로
    print(f"{name:<8} tanh 미분 평균 {(1 - o ** 2).mean().item():.3f}")  # 작을수록 포화


# ---------- 6-4 ----------
# output.grad 로 재면 (원본 셀 66): 쓰면 안 되는 지름길
torch.manual_seed(SEED)                                              # 같은 시작점
rnn = nn.RNN(10, 32, batch_first=True)                               # 감싼 판
head = nn.Linear(32, 2)                                              # 분류기
xs, ys = make_front_task(64, 100, seed=7)                            # 6장과 같은 데이터
output, h_n = rnn(to_tensor(xs))                                     # 칸마다 요약
output.retain_grad()                                                 # output의 기울기 남기기
nn.CrossEntropyLoss()(head(output[:, -1, :]), torch.tensor(ys)).backward()  # 끝 칸만 손실에 씀
g = output.grad                                                      # (64, 100, 32)
show("거슬러 간 칸", STEPS)                                          # 머리 줄
show("output.grad", [g[:, 99 - b, :].norm().item() for b in STEPS])  # 끝 칸만 값이 있음
print("기울기가 정확히 0 인 칸 :",                                   # 손실에 직접 쓰지 않은 칸
      sum(1 for t in range(100) if g[:, t, :].abs().max().item() == 0.0), "/ 100")

거슬러 간 칸           0        10        20        40        70        99
학습 전         5.16e-02  1.06e-04  2.94e-07  1.33e-12  1.50e-20  0.00e+00


20 epoch        3.86e-02  1.17e-02  1.19e-03  3.22e-05  1.26e-07  7.10e-10


60 epoch        1.55e-05  4.38e-12  3.87e-18  0.00e+00  0.00e+00  0.00e+00
학습 전     tanh 미분 평균 0.963
20 epoch tanh 미분 평균 0.945
60 epoch tanh 미분 평균 0.145


거슬러 간 칸           0        10        20        40        70        99
output.grad     5.16e-02  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00
기울기가 정확히 0 인 칸 : 99 / 100


In [57]:
# ---------- 6-5 ----------
# 가중치 행렬을 재면 (원본 셀 68): 학습 전 · float64 · 6-1과 같은 모델 · 데이터
torch.manual_seed(SEED)                                   # 같은 시작점
cell = nn.RNNCell(10, 32).to(torch.float64)               # 상자
head = nn.Linear(32, 2).to(torch.float64)                 # 분류기
xs, ys = make_front_task(64, 100, seed=7)                 # 같은 데이터
X, Y = to_tensor(xs).to(torch.float64), torch.tensor(ys)  # float64
h = torch.zeros(64, 32, dtype=torch.float64)              # 처음 요약
hs = []                                                   # 칸마다 요약
for t in range(100):                                      # 100칸
    h = cell(X[:, t, :], h)                               # 새 요약
    h.retain_grad()                                       # 기울기 남기기
    hs.append(h)                                          # 저장
nn.CrossEntropyLoss()(head(hs[-1]), Y).backward()         # 역전파
norms = [hs[t].grad.norm().item() for t in range(100)]    # 칸별 기울기 크기

with torch.no_grad():                                                      # 측정만
    W = cell.weight_hh                                                     # 이전 요약에 곱하는 행렬
    radius = torch.linalg.eigvals(W).abs().max().item()                    # 고유값 크기의 최댓값
    tanh_d = (1 - torch.stack(hs) ** 2).mean().item()                      # tanh 미분 평균
    smax = torch.linalg.svdvals(W).max().item()                            # 최대 특이값
    print(f"한 번 곱할 때 최대로 늘이는 배율 (최대 특이값) : {smax:.2f}")  # 한 번 곱의 최대
    print(f"거듭 곱할 때 남는 배율 (고유값 크기의 최댓값)  : {radius:.2f}")  # 반복 곱의 배율
    print(f"tanh 미분 평균                               : {tanh_d:.2f}")  # 활성화가 깎는 몫
    print(f"두 값을 곱하면                                : {radius * tanh_d:.2f}")  # 예상 배율
per = (norms[99 - 70] / norms[99]) ** (1 / 70)                             # 70칸 비율의 70제곱근
print(f"실제 기울기에서 잰 한 칸당 비율 (0 → 70스텝)   : {per:.2f}")       # 실측

한 번 곱할 때 최대로 늘이는 배율 (최대 특이값) : 1.10
거듭 곱할 때 남는 배율 (고유값 크기의 최댓값)  : 0.57
tanh 미분 평균                               : 0.96
두 값을 곱하면                                : 0.55
실제 기울기에서 잰 한 칸당 비율 (0 → 70스텝)   : 0.54


In [58]:
# ---------- 7-1 ----------
# LSTM 이 돌려주는 것 (원본 셀 70)
torch.manual_seed(SEED)                                      # 같은 시작점
lstm = nn.LSTM(10, 32, batch_first=True)                     # 입력 10 · hidden 32
x = to_tensor(xtr[:8])                                       # (8, 10, 10)
output, (h_n, c_n) = lstm(x)                                 # 괄호 안 두 개 = 튜플 풀기
print("x      :", tuple(x.shape))                            # 입력
print("output :", tuple(output.shape))                       # 칸마다 h
print("h_n    :", tuple(h_n.shape))                          # 마지막 h
print("c_n    :", tuple(c_n.shape))                          # 마지막 메모 c
print("h_n 과 c_n 이 같은 값인가 :", torch.equal(h_n, c_n))  # h = o × tanh(c) 라 다름
print("파라미터 RNN · LSTM :", sum(p.numel() for p in nn.RNN(10, 32).parameters()),  # 4벌 계산
      "·", sum(p.numel() for p in lstm.parameters()))


# ---------- 7-2 ----------
# 같은 측정을 LSTMCell로 · forget 문 bias 0 = 'LSTM 기본' (원본 셀 72)
show("거슬러 간 칸", STEPS)                                      # 머리 줄
show("RNN", grad_profile("RNN"), ".1e")                          # 비교 기준
show("LSTM 기본", grad_profile("LSTM", forget_bias=0.0), ".1e")  # 문을 달았지만


# 문이 얼마나 열려 있었나 (원본 셀 74): 3과 7 사이가 99칸인 데이터
def forget_stats(forget_bias, T=100, hid=32, batch=64):                # 칸별 forget 문 평균값
    torch.manual_seed(SEED)                                            # 같은 시작점
    cell = nn.LSTMCell(10, hid)                                        # LSTM 상자
    with torch.no_grad():                                              # 값만 넣음
        cell.bias_ih[hid:2 * hid].fill_(forget_bias)                   # forget 구간
        cell.bias_hh[hid:2 * hid].fill_(0.0)                           # 한쪽은 0
    xs, _ = make_order_task(batch, T, gap=T - 1, seed=7)               # 3 · 7이 양 끝
    X = to_tensor(xs)                                                  # 텐서
    h, c = torch.zeros(batch, hid), torch.zeros(batch, hid)            # 처음 요약 · 메모
    fs = []                                                            # 칸마다 문 값 평균
    with torch.no_grad():                                              # 측정만
        for t in range(T):                                             # 100칸
            # 네 문 계산
            g = X[:, t, :] @ cell.weight_ih.T + cell.bias_ih + h @ cell.weight_hh.T + cell.bias_hh
            fs.append(torch.sigmoid(g[:, hid:2 * hid]).mean().item())  # forget 문 값 평균
            h, c = cell(X[:, t, :], (h, c))                            # 한 칸 진행
    return sum(fs) / len(fs), sum(math.log10(v) for v in fs)           # 평균 · 곱의 log10


for b in (0.0, 2.0):                                            # 더하는 값 0 · 2
    mean_f, log_prod = forget_stats(b)                          # 측정
    print(f"문에 더하는 값 {b:.0f} · 백 칸 평균 {mean_f:.4f}",  # 문 값 평균
          f"· 백 칸의 값을 다 곱하면 10^{log_prod:.1f}")


# ---------- 7-3 ----------
# 문을 열어두면 (원본 셀 76): forget 문 bias 2 · 4
show("거슬러 간 칸", STEPS)                                            # 머리 줄
show("RNN", grad_profile("RNN"), ".1e")                                # 기준
show("LSTM 기본", grad_profile("LSTM", forget_bias=0.0), ".1e")        # 0 → 문 값 0.5
show("LSTM 열어둠", grad_profile("LSTM", forget_bias=2.0), ".1e")      # 2 → 0.88
show("LSTM 활짝", grad_profile("LSTM", forget_bias=4.0), ".1e")        # 4 → 0.98
opened = grad_profile("LSTM", forget_bias=2.0)                         # 열어둔 경우
print(f"열어둔 경우 0스텝 ÷ 99스텝 : {opened[0] / opened[5]:,.0f}배")  # 해결이 아니라 늦춤
start = torch.sigmoid(torch.tensor([0., 2., 4.]))                      # bias → 문 시작값
print("sigmoid(0 · 2 · 4) :", [round(v, 2) for v in start.tolist()])   # 0.5 · 0.88 · 0.98

x      : (8, 10, 10)
output : (8, 10, 32)
h_n    : (1, 8, 32)
c_n    : (1, 8, 32)
h_n 과 c_n 이 같은 값인가 : False
파라미터 RNN · LSTM : 1408 · 5632
거슬러 간 칸           0        10        20        40        70        99


RNN              5.2e-02   1.1e-04   2.9e-07   1.3e-12   1.5e-20   0.0e+00
LSTM 기본        5.3e-02   4.6e-05   4.0e-07   2.8e-11   2.1e-17   0.0e+00
문에 더하는 값 0 · 백 칸 평균 0.4997 · 백 칸의 값을 다 곱하면 10^-30.1


문에 더하는 값 2 · 백 칸 평균 0.8779 · 백 칸의 값을 다 곱하면 10^-5.7
거슬러 간 칸           0        10        20        40        70        99
RNN              5.2e-02   1.1e-04   2.9e-07   1.3e-12   1.5e-20   0.0e+00
LSTM 기본        5.3e-02   4.6e-05   4.0e-07   2.8e-11   2.1e-17   0.0e+00


LSTM 열어둠      5.3e-02   1.0e-03   5.2e-04   1.9e-04   3.9e-05   1.6e-05
LSTM 활짝        5.3e-02   7.2e-04   5.9e-04   6.7e-04   9.8e-04   4.5e-03
열어둔 경우 0스텝 ÷ 99스텝 : 3,303배
sigmoid(0 · 2 · 4) : [0.5, 0.88, 0.98]


In [59]:
# ---------- 8-1 ----------
# 같은 조건 비교 (원본 셀 78 · 80): hidden 64 · Adam 1e-3 · batch 64 · 12 epoch · clip 5.0
report("RNN", 64)   # 문 없음
report("LSTM", 64)  # 문 3개 + 메모
report("GRU", 64)   # 문 2개


# ---------- 8-2 ----------
# 시드를 세 번 바꾸면 (원본 셀 82)
accs3 = {}                           # 모델별 정확도 3개 (8-4 · 과제에서 재사용)
for kind in ("RNN", "LSTM", "GRU"):  # 세 모델
    accs3[kind] = [run_har(kind, 64, seed=s)[0] for s in (SEED, 7, 1234)]  # seed만 바꿔 3번
    a = accs3[kind]                  # 짧은 이름
    print(f"{kind:<5} {a} · 평균 {statistics.mean(a):.4f} · 폭 {max(a) - min(a):.4f}")  # 흔들림


# ---------- 8-3 ----------
# 파라미터 수를 비슷하게 맞추면 (원본 셀 84 · 학습 없이 셈)
for kind, hid in (("RNN", 64), ("RNN", 128), ("GRU", 74), ("LSTM", 64)):  # 크기 조합
    n = sum(p.numel() for p in HarNet(kind, hid).parameters())            # 파라미터 수
    print(f"{kind:<5} hidden {hid:<4} 파라미터 {n:>6,}")                  # LSTM 64 근처로
print("손계산 RNN 64 :", 64 * (9 + 64) + 2 * 64 + 64 * 6 + 6)             # 1벌 + Linear


# 맞춘 크기로 다시 돌리면 (원본 셀 86)
report("RNN", 128)  # 3.5배 넘게 키운 RNN
report("GRU", 74)   # LSTM 근처 GRU


# ---------- 8-4 ----------
# 한 줄 안의 시간 순서를 섞으면 (원본 셀 88): 128칸 순서만 섞음 · LSTM hidden 64
def shuffle_time(X, seed):                                      # 줄마다 칸 순서 섞기
    g = torch.Generator().manual_seed(seed)                     # 이 작업 전용 난수
    out = X.clone()                                             # 원본 보존
    for b in range(X.shape[0]):                                 # 줄마다
        out[b] = X[b][torch.randperm(X.shape[1], generator=g)]  # 값 모음은 그대로 · 순서만
    return out                                                  # 섞은 텐서


normal = accs3["LSTM"]  # 그대로 = 8-2 결과 재사용 (같은 조건)
Xs, Xs_te = shuffle_time(Xn, 1), shuffle_time(Xn_te, 2)  # 학습 · 평가 모두 섞음
shuffled = [run_har("LSTM", 64, seed=s, Xa=Xs, Xb=Xs_te)[0] for s in (SEED, 7, 1234)]    # 3 seed
print(f"그대로  {normal} · 평균 {statistics.mean(normal):.4f}")                          # 순서 유지
print(f"섞음    {shuffled} · 평균 {statistics.mean(shuffled):.4f}")                      # 순서 파괴
print(f"떨어진 폭 {(statistics.mean(normal) - statistics.mean(shuffled)) * 100:.2f}%p")  # 순서의 몫


# ---------- 8-5 ----------
# 식으로 보면 (원본 셀 89): GRU 식을 손으로 계산해 nn.GRUCell과 맞춰 보기
torch.manual_seed(SEED)                                  # 같은 시작점
gru = nn.GRUCell(10, 4)                                  # 작은 GRU 상자
x, h = torch.randn(2, 10), torch.randn(2, 4)             # 입력 · 이전 요약
Wi, Wh = gru.weight_ih, gru.weight_hh                    # (12, 10) · (12, 4) = [r | z | n]
gi, gh = x @ Wi.T + gru.bias_ih, h @ Wh.T + gru.bias_hh  # 입력 쪽 · 요약 쪽
r = torch.sigmoid(gi[:, 0:4] + gh[:, 0:4])               # 예전 요약을 얼마나 참고할지
z = torch.sigmoid(gi[:, 4:8] + gh[:, 4:8])               # 예전 요약을 얼마나 남길지
n = torch.tanh(gi[:, 8:12] + r * gh[:, 8:12])            # 새 후보
h_new = (1 - z) * n + z * h                              # 남길 몫 + 채울 몫 = 1
print("GRU 식 == nn.GRUCell :", torch.allclose(h_new, gru(x, h), atol=1e-6))  # 문서 식 확인

RNN   hidden 64   정확도 0.8073 · 파라미터  5,190 · 31.0s


LSTM  hidden 64   정확도 0.8683 · 파라미터 19,590 · 18.3s


GRU   hidden 64   정확도 0.9026 · 파라미터 14,790 · 15.4s


RNN   [0.8073, 0.8246, 0.8039] · 평균 0.8119 · 폭 0.0207


LSTM  [0.8683, 0.889, 0.8748] · 평균 0.8774 · 폭 0.0207


GRU   [0.9026, 0.9006, 0.9002] · 평균 0.9011 · 폭 0.0024
RNN   hidden 64   파라미터  5,190
RNN   hidden 128  파라미터 18,566
GRU   hidden 74   파라미터 19,320
LSTM  hidden 64   파라미터 19,590
손계산 RNN 64 : 5190


RNN   hidden 128  정확도 0.8554 · 파라미터 18,566 · 23.4s


GRU   hidden 74   정확도 0.9097 · 파라미터 19,320 · 21.1s


그대로  [0.8683, 0.889, 0.8748] · 평균 0.8774
섞음    [0.8751, 0.8673, 0.8738] · 평균 0.8721
떨어진 폭 0.53%p
GRU 식 == nn.GRUCell : True


In [60]:
# LSTM 식: c = f ⊙ c + i ⊙ g · h = o ⊙ tanh(c)
lstm = nn.LSTMCell(10, 4)                                                # 작은 LSTM 상자
c = torch.randn(2, 4)                                                    # 이전 메모
g4 = x @ lstm.weight_ih.T + lstm.bias_ih + h @ lstm.weight_hh.T + lstm.bias_hh  # [i | f | g | o]
i, f = torch.sigmoid(g4[:, 0:4]), torch.sigmoid(g4[:, 4:8])              # 적는 문 · 잊는 문
g, o = torch.tanh(g4[:, 8:12]), torch.sigmoid(g4[:, 12:16])              # 적을 내용 · 꺼내는 문
c_new = f * c + i * g                                                    # 덧셈 길
h2, c2 = lstm(x, (h, c))                                                 # PyTorch 계산
print("LSTM 식 == nn.LSTMCell :", torch.allclose(c_new, c2, atol=1e-6),  # 메모 · 요약 둘 다
      torch.allclose(o * torch.tanh(c_new), h2, atol=1e-6))
print("forget bias 자리 bias_ih[H:2*H] · H = 4 →", list(range(4, 8)))    # 두 번째 구간

LSTM 식 == nn.LSTMCell : True True
forget bias 자리 bias_ih[H:2*H] · H = 4 → [4, 5, 6, 7]


In [61]:
# ---------- 9. 종합 실습 ----------
# 과제 2 · hidden 크기 바꾸기: (___, ___) → (32, 128)
res2 = {}                               # (모델, hidden) → (정확도, 파라미터)
for kind in ("RNN", "LSTM", "GRU"):     # 세 모델
    for hid in (32, 128):               # 빈칸 정답
        acc, n, _ = run_har(kind, hid)  # 같은 조건 학습
        res2[kind, hid] = (acc, n)      # 저장
        print(f"{kind:<5} hidden {hid:<4} 정확도 {acc:.4f} · 파라미터 {n:>6,}")  # 표의 한 줄


# 과제 3 · 시드 늘리기: (SEED, 7, 1234, ___, ___) → 42 · 2026 추가
SEEDS5 = (SEED, 7, 1234, 42, 2026)                                    # 앞 3개 + 빈칸 2개
accs5 = {}                                                            # 모델별 정확도 5개
for kind in ("RNN", "LSTM", "GRU"):                                   # 세 모델
    accs5[kind] = accs3[kind] + [run_har(kind, 64, seed=s)[0] for s in SEEDS5[3:]]  # 앞 3개 재사용
    a = accs5[kind]                                                   # 짧은 이름
    print(f"{kind:<5} {a} · 평균 {statistics.mean(a):.4f} · 폭 {max(a) - min(a):.4f}")  # 흔들림
gap = statistics.mean(accs5["LSTM"]) - statistics.mean(accs5["RNN"])  # 모델 차이
# 비교
print(f"LSTM - RNN 평균 차이 {gap:.4f} · RNN 폭 {max(accs5['RNN']) - min(accs5['RNN']):.4f}")


# 과제 4 · 앞 절반만 섞기: ___ → X[b, :half][torch.randperm(half, generator=g)]
def shuffle_front_half(X, seed):                                        # 앞 64칸만 섞기
    g = torch.Generator().manual_seed(seed)                             # 이 작업 전용 난수
    out = X.clone()                                                     # 원본 보존
    half = X.shape[1] // 2                                              # 64칸
    for b in range(X.shape[0]):                                         # 줄마다
        out[b, :half] = X[b, :half][torch.randperm(half, generator=g)]  # 빈칸 정답
    return out                                                          # 섞은 텐서


Xf, Xf_te = shuffle_front_half(Xn, 1), shuffle_front_half(Xn_te, 2)  # 앞 절반 섞은 데이터
front = [run_har("LSTM", 64, seed=s, Xa=Xf, Xb=Xf_te)[0] for s in (SEED, 7, 1234)]  # 3 seed
print(f"그대로       평균 {statistics.mean(normal):.4f}")            # 8-4
print(f"앞 절반 섞음 {front} · 평균 {statistics.mean(front):.4f}")   # 과제 4
print(f"전부 섞음    평균 {statistics.mean(shuffled):.4f}")          # 8-4

RNN   hidden 32   정확도 0.7635 · 파라미터  1,574


RNN   hidden 128  정확도 0.8554 · 파라미터 18,566


LSTM  hidden 32   정확도 0.9040 · 파라미터  5,702


LSTM  hidden 128  정확도 0.9077 · 파라미터 71,942


GRU   hidden 32   정확도 0.8951 · 파라미터  4,326


GRU   hidden 128  정확도 0.9063 · 파라미터 54,150


RNN   [0.8073, 0.8246, 0.8039, 0.7465, 0.8113] · 평균 0.7987 · 폭 0.0781


LSTM  [0.8683, 0.889, 0.8748, 0.8833, 0.8928] · 평균 0.8816 · 폭 0.0245


GRU   [0.9026, 0.9006, 0.9002, 0.907, 0.9009] · 평균 0.9023 · 폭 0.0068
LSTM - RNN 평균 차이 0.0829 · RNN 폭 0.0781


그대로       평균 0.8774
앞 절반 섞음 [0.8833, 0.8884, 0.8918] · 평균 0.8878
전부 섞음    평균 0.8721
